# LLM 제품 사전

규칙 버전 제품 사전(`product_dictionary.ipynb`)의 노드를 LLM으로 다듬는다(`graphDB.md` 6.3). 이 노트북은 **LLM 실행과 원본 보관**, 그리고 결과를 "원래 언급 → 제품" 표로 내보내는 것까지 한다(23절). 그래프에 적용하는 것은 `product_apply.ipynb`가 한다.

| 단계 | 하는 일 | 단위 |
|---|---|---|
| 0 | **제품군 목록**: WI26 업종마다 제품 이름을 보여 주고 제품군을 제안받는다 | 업종 하나 |
| 0-2 | **제품군 통합**: 업종별 제안을 업종을 넘는 하나의 목록으로 합친다 | 종류 하나(3건) |
| 검토 | 사람이 검토한 것을 표로 적어 확정본을 만든다(10절) | - |
| 1 | 노드 점검·분류: 종류(제품·서비스·시설·지표), 섞인 노드 나누기, 여러 품목 이름 나누기, 제품군, 대표 이름 | 노드 100개 |
| 2 | 제품군 안 정리: 동의어 합치기, 계층, 세대·규격 속성, 후보 하위 제품 | 제품군 하나 |
| 3 | 브랜드 → 제품 | 브랜드 묶음 |

지금은 **0-2단계, 검토 반영, 1단계(시범·API 확인·전체 배치), 2단계 시범·API 확인·전체 배치, 내보내기, 3단계 시범·전체 실행(Codex 구독)**까지 있다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/graph/products.csv`, `product_mentions.csv` | 규칙 버전 제품 노드와 언급(`product_dictionary`) |
| `source/news/final/news_final.json` | 뉴스 언급이 관계에 쓰였는지(그래프에 쓰이는 노드만 다룬다) |
| `prompts/product_family_v1.md`, `product_family_merge_v1.md` (+ `.schema.json`), `product_node_v4.md`, `product_tree_v4.md`, `product_brand_v3.md` | 0단계, 0-2단계, 1·2·3단계 지시문(최종본만 남김. 버전마다 고친 것은 12·20·25절 표와 `graphDB.md` 7장)과 결과 스키마 |

**출력** (`source/products/llm/`)

| 파일 | 내용 |
|---|---|
| `raw/지시문/codex-모델/항목.json` | LLM 응답 전체. 고치지 않는다 |
| `runs.csv` | 실행 기록: 단계, 항목, 입력 해시, 지시문 해시, 실행 수단, 모델, 시각, 상태, 토큰, 비용(API·배치) |
| `stage1_groups_지시문.csv` | 1단계 결과: 노드 → 그룹(종류, 제품군, 대표 이름, 담은 줄번호·언급 수) |
| `batches/`, `batches.csv` | 배치로 보낸 요청 파일과 배치 기록(배치 ID, 상태, 묶음 수, 결과를 받았는지) |
| `families_by_sector.csv` | 0단계: 업종별로 제안받은 제품군 |
| `families_draft.csv` | 0단계: key가 같은 것만 합친 초안 |
| `families_merged.csv`, `family_map.csv` | 0-2단계: 통합한 제품군 목록, 업종별 제안 → 새 제품군 대응 |
| `families_final.csv`, `family_map_final.csv` | 검토 반영: **확정 제품군 목록**(1단계가 쓴다), 업종별 제안 → 확정 제품군 |
| `schemas/product_node_v1.schema.json` | 1단계 결과 스키마(제품군 key만 고르게 만든 것) |
| `stage3_brands_지시문.csv` | 3단계 결과: 브랜드 언급(출처, 문서, 기사 개체 번호, 이름) → 종류, 제품군, 일반 제품 이름, 코드가 정한 제품과 방법 |
| `sample_node_v1.csv` | 1단계 시범 묶음(어려운 사례 33개 + 무작위 37개, 고정) |
| `llm_mention_map.csv`, `llm_products.csv`, `llm_candidates.csv` | 내보내기(23절): 언급 → 제품 ID, 제품(제품군·상위 제품·규격), 후보 하위 제품. `product_apply`가 읽는다 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `product_dictionary`(규칙 버전) → **`product_llm`** → `product_apply`(LLM 사전 적용) → `graph_db` |
| 외부 요청·비용 | 있음. 0단계는 Codex 구독(비용 없음, 업종 26건), 0-2단계도 Codex(3건). 1·2단계 전체 실행은 OpenAI 배치 API, 3단계 전체(81건)는 규모가 작아 Codex 구독(한도에 걸리면 텔레그램 알림). 실행 전에 건수·시간·비용을 알리고 승인받는다 |
| 다시 실행 | 된다. 같은 지시문·입력으로 성공한 항목은 건너뛴다 |
| 멈추는 경우 | `RUN=False`(기본)면 대상만 보여 주고 보내지 않는다. Codex 사용량 한도에 걸리면 새로 시작하지 않는다 |

## 1. 설정

In [1]:
# ── 설정 ──
import datetime as dt
import hashlib
import json
import os
import re
import subprocess
import tempfile
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import jsonschema
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
GRAPH = ROOT / 'source' / 'graph'
LLM = ROOT / 'source' / 'products' / 'llm'        # 이 노트북이 만드는 것
LLM.mkdir(parents=True, exist_ok=True)
RUNS_PATH = LLM / 'runs.csv'

MODEL = 'gpt-6.1-terra'
EFFORT = 'medium'
PARALLEL = 3              # 동시에 돌릴 수
TIMEOUT_SEC = 900         # 항목 하나에 기다리는 최대 시간
FAMILY_PROMPT = 'product_family_v1'
RUN = False               # True로 바꿔야 실행한다

## 2. 입력: 그래프에 쓰이는 제품 노드

규칙 버전 제품 사전의 노드 중 **그래프에 쓰이는 것**만 다룬다. DART 언급(제품·원재료)은 모두 관계가 되고, 뉴스 언급은 기사 관계(주체·대상·제품)에 쓰였을 때만 쓰인다. 관계가 없는 노드는 그래프에서 빠지므로 처리하지 않는다.

업종은 노드의 첫 번째 관련 업종(가장 많이 나온 업종)이다. 업종마다 `영어 이름 | 한글 이름 | 언급 수 | 범주`를 언급이 많은 순서로 한 줄씩 적은 글이 0단계 입력이다.

In [2]:
# ── 입력: 그래프에 쓰이는 제품 노드와 업종별 이름 목록 ──
P = pd.read_csv(GRAPH / 'products.csv', dtype=str).fillna('')
PM = pd.read_csv(GRAPH / 'product_mentions.csv', dtype=str).fillna('')
news = json.loads((ROOT / 'source' / 'news' / 'final' / 'news_final.json').read_text(encoding='utf-8'))['articles']
# 기사 관계에 쓰인 개체: (기사 ID, 개체 ID)
in_relation = {(aid, x) for aid, a in news.items() for r in a['relations'] for x in (r['subject'], r['object'], r['product']) if x}
pm = PM[PM['book'] == '제품']
used = pm[(pm['source'] == 'dart') | pd.Series([(d, ref) in in_relation for d, ref in zip(pm['doc'], pm['ref'])], index=pm.index)]
nodes = P[P['id'].isin(set(used['node_id']))].copy()
nodes['n_mentions'] = pd.to_numeric(nodes['n_mentions'])
nodes['sector'] = nodes['sectors'].str.split(' | ', regex=False).str[0]
print(f"제품 노드 {len(P):,}개 중 그래프에 쓰이는 것 {len(nodes):,}개 / 업종 없는 것 {(nodes['sector'] == '').sum()}개(0단계에서 빼고 1단계에서 제품군을 정한다)")


def name_line(r):
    """0단계 입력 한 줄: 영어 이름 | 한글 이름 | 언급 수 | 범주."""
    return f"{r.name_en or r.id[2:]} | {r.name_ko} | {r.n_mentions} | {r.category}"


FAMILY_INPUTS = {}                                   # 업종 → 입력 글
for sec, g in nodes[nodes['sector'] != ''].groupby('sector'):
    g = g.sort_values(['n_mentions', 'name_en'], ascending=[False, True])
    FAMILY_INPUTS[sec] = f"업종: {sec}\n이름 {len(g)}개\n\n" + '\n'.join(name_line(r) for r in g.itertuples())
sizes = pd.Series({k: len(v) for k, v in FAMILY_INPUTS.items()}).sort_values(ascending=False)
print(f'업종 {len(FAMILY_INPUTS)}개 / 입력 글자 합 {sizes.sum():,}, 가장 긴 업종 {sizes.index[0]} {sizes.iloc[0]:,}자')

제품 노드 13,767개 중 그래프에 쓰이는 것 12,643개 / 업종 없는 것 3개(0단계에서 빼고 1단계에서 제품군을 정한다)
업종 26개 / 입력 글자 합 557,581, 가장 긴 업종 건강관리 95,951자


## 3. 실행 도구 (Codex)

뉴스·DART 추출과 같은 방식이다. 항목 하나마다 `codex exec`를 빈 임시 폴더에서 새로 띄운다(읽기 전용, 사용자 설정·규칙 무시, 출력 스키마 지정). API 키는 넘기지 않아 구독으로만 돈다. 응답 전체를 `raw/`에 저장하고 `runs.csv`에 한 줄을 남긴다.

In [3]:
# ── 실행 도구: Codex로 항목 하나를 돌리고 원본을 저장한다 ──
RUN_CWD = Path(tempfile.gettempdir()) / 'graphdb_llm_run'
RUN_CWD.mkdir(exist_ok=True)
# Codex에 API 키가 넘어가지 않게 뺀다(구독으로만 돈다)
ENV = {k: v for k, v in os.environ.items() if k not in ('ANTHROPIC_API_KEY', 'ANTHROPIC_AUTH_TOKEN', 'OPENAI_API_KEY')}
LIMIT_RE = re.compile(r'usage limit|rate limit|limit reached|quota|too many requests|429', re.I)
CODEX_SUFFIX = '\n\n## 결과 형식\n\n결과는 주어진 출력 스키마에 맞는 JSON 하나다. 파일을 읽거나 명령을 실행하지 말고, 아래 입력만 보고 답한다.'
RUN_COLUMNS = ['stage', 'item', 'input_hash', 'prompt_hash', 'runner', 'model', 'effort', 'run_utc', 'raw_file',
               'status', 'error', 'duration_s', 'input_tokens', 'cache_read_tokens', 'output_tokens', 'reasoning_tokens']


def text_hash(text):
    """글의 짧은 해시. 입력이나 지시문이 바뀌면 다시 돌린다."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


def load_prompt(name):
    """지시문(Codex용 결과 형식 문단을 붙임), 스키마 경로, 지시문 해시."""
    prompt = (ROOT / 'prompts' / f'{name}.md').read_text(encoding='utf-8') + CODEX_SUFFIX
    return prompt, ROOT / 'prompts' / f'{name}.schema.json', text_hash(prompt + MODEL + EFFORT)


def read_runs():
    """실행 기록을 읽는다."""
    return pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS) if RUNS_PATH.exists() else pd.DataFrame(columns=RUN_COLUMNS)


def parse_result(resp):
    """원본 응답에서 결과 JSON을 꺼낸다(Codex 최종 메시지)."""
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', (resp.get('last_message') or '').strip())
    return json.loads(text[text.find('{'):])


def run_codex(stage, item, prompt, schema_path, prompt_hash, text):
    """Codex로 항목 하나를 돌리고 원본 응답을 저장한 뒤 실행 기록 한 줄을 돌려준다."""
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    raw_dir = LLM / 'raw' / stage / f'codex-{MODEL}'
    raw_dir.mkdir(parents=True, exist_ok=True)
    safe = re.sub(r'[^0-9A-Za-z가-힣_]+', '_', item)
    raw_file = f"{safe}_{now.strftime('%Y%m%dT%H%M%S')}.json"
    out_file = RUN_CWD / f'{stage}_{safe}_{now.strftime("%H%M%S")}.out'
    row = dict(stage=stage, item=item, input_hash=text_hash(text), prompt_hash=prompt_hash, runner='codex', model=MODEL, effort=EFFORT,
               run_utc=now.isoformat(), raw_file=raw_file, status='', error='')
    cmd = ['codex', 'exec', '--json', '--ephemeral', '--skip-git-repo-check', '-s', 'read-only', '--ignore-user-config', '--ignore-rules',
           '-C', str(RUN_CWD), '--output-schema', str(schema_path), '-o', str(out_file),
           '-c', f'model_reasoning_effort="{EFFORT}"', '-m', MODEL, '-']
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, input=prompt + '\n\n---\n\n# 입력\n\n' + text, capture_output=True, text=True,
                              timeout=TIMEOUT_SEC, cwd=RUN_CWD, env=ENV)
    except subprocess.TimeoutExpired:
        return row | dict(status='timeout', duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    events = []
    for line in proc.stdout.splitlines():
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            events.append({'type': 'text', 'text': line})
    last = out_file.read_text(encoding='utf-8') if out_file.exists() else ''
    out_file.unlink(missing_ok=True)
    usage = next((e['usage'] for e in reversed(events) if e.get('type') == 'turn.completed' and isinstance(e.get('usage'), dict)), {})
    errors = [e for e in events if e.get('type') in ('error', 'turn.failed')]
    resp = {'runner': 'codex', 'model': MODEL, 'stage': stage, 'item': item, 'returncode': proc.returncode, 'usage': usage,
            'last_message': last, 'events': events, 'stderr': proc.stderr[-3000:]}
    (raw_dir / raw_file).write_text(json.dumps(resp, ensure_ascii=False, indent=1), encoding='utf-8')
    row.update(input_tokens=usage.get('input_tokens'), cache_read_tokens=usage.get('cached_input_tokens'),
               output_tokens=usage.get('output_tokens'), reasoning_tokens=usage.get('reasoning_output_tokens'))
    if errors or proc.returncode != 0 or not last:
        msg = ' '.join([proc.stderr, json.dumps(errors, ensure_ascii=False)])
        return row | dict(status='limit' if LIMIT_RE.search(msg) else 'error', error=msg.strip()[:300])
    try:
        errs = [e.message for e in jsonschema.Draft202012Validator(json.loads(schema_path.read_text(encoding='utf-8'))).iter_errors(parse_result(resp))]
        row.update(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


def run_items(stage, items, prompt, schema_path, prompt_hash):
    """항목들을 동시 PARALLEL개씩 돌린다. 하나 끝날 때마다 실행 기록에 붙이고, 사용량 한도에 걸리면 새로 시작하지 않는다."""
    stop, queue = False, list(items)
    with ThreadPoolExecutor(PARALLEL) as pool:
        pending = {pool.submit(run_codex, stage, k, prompt, schema_path, prompt_hash, items[k]): k for k in queue[:PARALLEL]}
        queue = queue[PARALLEL:]
        while pending:
            fut = next(as_completed(list(pending)))
            k = pending.pop(fut)
            row = fut.result()
            pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
            print(f"  {k:12} {row['status']:8} {row.get('duration_s', '')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')}")
            stop = stop or row['status'] == 'limit'
            if queue and not stop:
                nxt = queue.pop(0)
                pending[pool.submit(run_codex, stage, nxt, prompt, schema_path, prompt_hash, items[nxt])] = nxt
    if stop:
        print('사용량 한도에 걸려 멈췄다. 한도가 풀린 뒤 다시 실행하면 남은 항목부터 한다.')

## 4. 0단계: 제품군 목록 (Codex)

업종마다 한 건씩 보낸다. 같은 지시문·입력으로 성공한 업종은 건너뛴다. 지시문: `prompts/product_family_v1.md`.

In [4]:
# ── 0단계 실행: 업종마다 제품군을 제안받는다 ──
FAMILY_TEXT, FAMILY_SCHEMA, FAMILY_HASH = load_prompt(FAMILY_PROMPT)
runs = read_runs()
done = set(runs.loc[(runs['stage'] == FAMILY_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == FAMILY_HASH),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: v for k, v in FAMILY_INPUTS.items() if (k, text_hash(v)) not in done}
print(f'지시문 {len(FAMILY_TEXT):,}자 / 업종 {len(FAMILY_INPUTS)}개 중 이미 끝남 {len(FAMILY_INPUTS) - len(todo)}개 / 이번에 돌릴 것 {len(todo)}개, '
      f'입력 {sum(map(len, todo.values())):,}자')
if RUN and todo:
    run_items(FAMILY_PROMPT, todo, FAMILY_TEXT, FAMILY_SCHEMA, FAMILY_HASH)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

지시문 1,708자 / 업종 26개 중 이미 끝남 26개 / 이번에 돌릴 것 0개, 입력 0자
RUN = False: 실행하지 않았다.


## 5. 0단계 결과 모으기

- 업종마다 성공한 마지막 결과를 읽는다. 예시(`examples`)가 입력 목록에 실제로 있는지 센다(없는 이름을 지어냈는지 확인).
- 업종 사이에 `key`가 같은 제품군을 합쳐 초안을 만든다. 영어·한글 이름과 설명은 가장 먼저 나온 것을 쓰고, 어느 업종에서 나왔는지와 예시를 모은다.
- 초안(`families_draft.csv`)은 사람이 검토해 확정한다. 확정본은 1단계의 제품군 목록이 된다.

In [5]:
# ── 0단계 결과 모으기: 업종별 제품군 → 업종 사이에 합친 초안 ──
runs = read_runs()
ok = runs[(runs['stage'] == FAMILY_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == FAMILY_HASH)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
input_names = {sec: {l.split(' | ')[0] for l in txt.split('\n')[3:]} | {l.split(' | ')[1] for l in txt.split('\n')[3:] if ' | ' in l}
               for sec, txt in FAMILY_INPUTS.items()}
rows, notes = [], {}
for r in ok.itertuples():
    out = parse_result(json.loads((LLM / 'raw' / FAMILY_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))
    notes[r.item] = out['notes']
    for f in out['families']:
        rows.append(dict(sector=r.item, key=f['key'].strip().lower(), name_en=f['name_en'], name_ko=f['name_ko'], kind=f['kind'], scope=f['scope'],
                         examples=' | '.join(f['examples']), n_examples=len(f['examples']),
                         n_examples_in_input=sum(e in input_names.get(r.item, set()) for e in f['examples'])))
by_sector = pd.DataFrame(rows)
if len(by_sector):
    by_sector.to_csv(LLM / 'families_by_sector.csv', index=False, encoding='utf-8-sig')
    draft = (by_sector.groupby('key')
             .agg(name_en=('name_en', 'first'), name_ko=('name_ko', 'first'), kind=('kind', lambda s: Counter(s).most_common(1)[0][0]),
                  sectors=('sector', lambda s: ' | '.join(dict.fromkeys(s))), n_sectors=('sector', 'nunique'),
                  scope=('scope', 'first'), examples=('examples', lambda s: ' | '.join(list(dict.fromkeys(' | '.join(s).split(' | ')))[:8])))
             .reset_index().sort_values(['n_sectors', 'key'], ascending=[False, True]))
    draft.to_csv(LLM / 'families_draft.csv', index=False, encoding='utf-8-sig')
    print(f'업종 {by_sector.sector.nunique()}개에서 제품군 {len(by_sector)}개 제안 → 합친 초안 {len(draft)}개 / 종류', draft['kind'].value_counts().to_dict())
    print(f"예시 {by_sector.n_examples.sum()}개 중 입력 목록에 있는 것 {by_sector.n_examples_in_input.sum()}개")
    display(by_sector.groupby('sector').size().rename('제품군 수').to_frame().T)
else:
    print('아직 결과가 없다.')

업종 26개에서 제품군 1034개 제안 → 합친 초안 983개 / 종류 {'제품·원자재': 605, '서비스': 317, '기타': 31, '시설·공장': 19, '가격·지표': 11}
예시 4739개 중 입력 목록에 있는 것 4739개


sector,IT가전,IT하드웨어,건강관리,건설,기계,디스플레이,"미디어,교육",반도체,보험,비철금속,...,은행,자동차,조선,증권,철강,통신서비스,필수소비재,"호텔,레저","화장품,의류",화학
제품군 수,42,40,43,41,40,33,40,41,35,40,...,38,41,39,39,39,39,44,38,40,40


## 6. 0단계 결과 보기

In [6]:
# ── 확인: 여러 업종에서 나온 제품군, 업종 하나의 제품군, 애매했던 점 ──
if len(by_sector):
    print('1) 여러 업종에서 나온 제품군')
    display(draft[draft.n_sectors > 1][['key', 'name_ko', 'kind', 'sectors', 'examples']].head(30))
    print('2) 반도체 업종의 제품군')
    display(by_sector[by_sector.sector == '반도체'][['key', 'name_ko', 'kind', 'scope', 'examples']])
    print('3) 업종별로 애매했던 점')
    for k, v in notes.items():
        if v:
            print(f'  [{k}] {v[:300]}')

1) 여러 업종에서 나온 제품군


,key,name_ko,kind,sectors,examples
315,enterprise_software_and_it_services,기업용 소프트웨어·IT 서비스,서비스,"미디어,교육 | 상사,자본재 | 운송 | 증권",IT consulting and system integration | Information and communications servic...
658,packaging_materials,포장재,제품·원자재,"건설 | 필수소비재 | 화장품,의류 | 화학",Packaging containers | Pharmaceutical packaging | Alu-alu packaging | 18-lit...
66,basic_petrochemicals,기초 석유화학제품,제품·원자재,"상사,자본재 | 에너지 | 화학",monoethylene glycol | Terephthalic acid | ethylene dichloride | mixed xylene...
476,industrial_gases,산업용 가스,제품·원자재,반도체 | 비철금속 | 화학,Industrial gas | Specialty gas | Carbon monoxide | Hexafluorobutadiene | dis...
661,paper_and_packaging_materials,종이·포장재,제품·원자재,"미디어,교육 | 상사,자본재 | 철강",Paper | Uncoated printing paper | Cases and boxes | Printing paper | Fancy p...
4,advertising_and_marketing_services,광고 및 마케팅 서비스,서비스,"소프트웨어 | 호텔,레저",Advertising agency services | Search advertising | Display advertising | Mar...
19,ai_assistants_and_agents,인공지능 비서 및 에이전트,서비스,소프트웨어 | 은행,ChatGPT | Muse | AI agents | Microsoft Copilot | AI shopping assistant | Mot...
30,alcoholic_beverages,주류,제품·원자재,"필수소비재 | 호텔,레저",Wine | Soju | Beer | Makgeolli | Whisky | Hops | Korean traditional alcoholi...
51,automotive_electronics,자동차 전장 부품,제품·원자재,IT하드웨어 | 자동차,Advanced driver assistance system | electronic shifter | Powertrain controll...
55,automotive_parts_and_materials,자동차 부품·소재,제품·원자재,"화장품,의류 | 화학",Automotive drivetrain components | Automotive alternators | Engine control u...


2) 반도체 업종의 제품군


,key,name_ko,kind,scope,examples
279,memory_chips,메모리 반도체,제품·원자재,"DRAM, HBM, NAND·NOR 플래시와 메모리 모듈을 포함하고 SSD 완제품과 메모리 검사장비는 제외한다.",memory chips | High Bandwidth Memory | DRAM | NAND flash memory | DDR5 | Mem...
280,ai_accelerator_chips,AI 가속 반도체,제품·원자재,"AI 학습·추론용 GPU, TPU와 전용 가속기를 포함하고 범용 CPU와 완성형 서버 시스템은 제외한다.",AI chips | graphics processing units | Nvidia H200 | TPU 8i | FuriosaAI RNGD...
281,processors_and_logic_chips,프로세서 및 로직 반도체,제품·원자재,"CPU, 마이크로컨트롤러, SoC, FPGA, ASIC과 용도가 특정되지 않은 칩을 포함하고 AI 전용 가속기와 메모리는 제외한다.",central processing units | microcontrollers | Application-specific integrate...
282,power_semiconductors,전력 반도체,제품·원자재,"전력관리 IC, 게이트 드라이버, 전력 트랜지스터와 다이오드를 포함하고 완성형 전원공급장치는 제외한다.",power management integrated circuit | silicon carbide power devices | API215...
283,interface_and_connectivity_chips,인터페이스 및 통신 반도체,제품·원자재,"네트워크·무선 연결, 데이터 인터커넥트, SSD 제어와 신호 처리용 칩을 포함하고 통신 장비 완제품은 제외한다.",CXL switch chip | SSD controller | smartphone modems | switch silicon | wire...
284,display_and_sensor_chips,디스플레이 및 센서 반도체,제품·원자재,"디스플레이 구동 IC, 이미지·지문 센서 관련 반도체와 센서 모듈을 포함하고 디스플레이 패널과 검사장비는 제외한다.",Display driver IC | AMOLED driver ICs | Fingerprint sensor module | Time-of-...
285,semiconductor_wafers_and_ingots,반도체 웨이퍼 및 잉곳,제품·원자재,실리콘과 화합물 반도체의 웨이퍼·잉곳을 포함하고 공정장비용 실리콘·SiC 부품은 제외한다.,Semiconductor wafer | Wafer | 300mm silicon carbide wafers | Gallium oxide w...
286,semiconductor_process_chemicals,반도체 공정용 화학재료,제품·원자재,"포토레지스트, 식각액, 현상액, 세정액, 슬러리와 그 원료를 포함하고 가스와 패키징 재료는 제외한다.",Etchant | Photoresist | Wet chemicals | Buffered oxide etch | Developer | Se...
287,industrial_gases,산업용 가스,제품·원자재,반도체 공정용 특수가스와 일반 산업용 가스를 포함하고 가스 공급·정화 장비는 제외한다.,Industrial gas | Specialty gas | Carbon monoxide | Hexafluorobutadiene | dis...
288,semiconductor_deposition_materials,반도체 증착재료,제품·원자재,"증착용 전구체, 스퍼터링 타깃과 금속 증착재를 포함하고 증착장비와 일반 웨이퍼는 제외한다.",Semiconductor precursor | Sputtering target | Gold deposition material | Bis...


3) 업종별로 애매했던 점
  [IT가전] Chip, Powder, pad, Aluminum case, Die casting, Wheels and others, Lead plate와 원재료 C·H·P·S 등은 용도나 조성이 불명확해 임시 분류했다. Metal composite precipitate와 NI-TOP도 세부 확인이 필요하다. Battery header와 배터리 조립 자재는 배터리 부품에, High-temperature battery는 특수전지에, 일반 Metallized film은 커패시터 소재에 임시 배치했다. 시설 항목은 관련 공사 명칭을 포함한다.
  [IT하드웨어] NA, PGC, B/S, CD and others, NMEA, PDL-WM-Wafer, TS3000-PCB-EDFA, Z-stopper 등은 뜻이나 용도를 확인하기 어려워 확정 분류하기 어렵다. Modules, base material, Equipment components, development materials, Raw material kit 등은 대상이 불명확하며, 복수 품목이 함께 적힌 항목은 구성품별 제품군에 연결해야 한다. C-band spectrum, LTE Cat 1bis, open radio access networ
  [건강관리] 입력에 적응증이나 실체가 없는 코드·약어와 복수 품목이 섞인 항목이 있어 제품군을 확정하기 어렵다. 예: HCP2203, CT702 500, Chemical P, Head, 버틀, Terazosin hydrochloride, fertilized eggs, ilaprazole sulfide and others. Pharmaceuticals·Medical devices·Pharmaceutical tablets처럼 치료 분야가 없는 상위명이나 제형은 적응증 확인 후 연결해야 한다. 동일 성분이 원료와 완제로 모두 쓰이거나 세포·유전자치료
  [건설] Dubai crude oil은 실물 원유 또는 가격 기준일 수 있어 문맥 확인이 필요하다. 복

## 7. 0-2단계: 제품군 통합 (Codex)

업종마다 따로 제안받아 같은 제품군을 업종마다 다른 이름으로 불렀다("전기차 충전 장비"와 "EV 충전 장비"). 업종별 제안 전체를 보여 주고 **업종을 넘는 하나의 목록**으로 합친다. 지시문: `prompts/product_family_merge_v1.md`.

- 종류별로 나눠 보낸다(제품·원자재 / 서비스 / 시설·지표·기타). 한 번에 보내면 결과가 길어 시간 초과가 날 수 있다.
- 제안마다 번호를 붙이고, 결과의 제품군마다 담은 번호(`members`)를 받는다. 모든 번호가 정확히 한 번씩 들어갔는지 9절에서 확인한다.
- 목표 수: 제품·원자재 약 180개, 서비스 약 70개, 시설·지표·기타 약 20개.

In [7]:
# ── 0-2단계 실행: 종류별로 업종별 제안을 하나의 목록으로 합친다 ──
MERGE_PROMPT = 'product_family_merge_v1'
TIMEOUT_SEC = 1800                                   # 통합은 결과가 길다
by_sector = pd.read_csv(LLM / 'families_by_sector.csv', dtype=str).fillna('')
by_sector['pid'] = [f'f{i}' for i in range(1, len(by_sector) + 1)]      # 제안 번호
MERGE_GROUPS = {'제품·원자재': (['제품·원자재'], 180), '서비스': (['서비스'], 70), '시설·지표·기타': (['시설·공장', '가격·지표', '기타'], 20)}
MERGE_INPUTS = {}
for name, (kinds, target) in MERGE_GROUPS.items():
    g = by_sector[by_sector['kind'].isin(kinds)]
    lines = [f"{r.pid} | {r.sector} | {r.name_ko} | {r.key} | {r.scope} | {' | '.join(r.examples.split(' | ')[:3])}" for r in g.itertuples()]
    MERGE_INPUTS[name] = f'종류: {name}\n제안 {len(g)}개\n목표 제품군 수: 약 {target}개\n\n' + '\n'.join(lines)
MERGE_TEXT, MERGE_SCHEMA, MERGE_HASH = load_prompt(MERGE_PROMPT)
runs = read_runs()
done = set(runs.loc[(runs['stage'] == MERGE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == MERGE_HASH),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: v for k, v in MERGE_INPUTS.items() if (k, text_hash(v)) not in done}
print(f'지시문 {len(MERGE_TEXT):,}자 / 묶음', {k: f'{len(v):,}자' for k, v in MERGE_INPUTS.items()}, f'/ 이번에 돌릴 것 {len(todo)}개')
if RUN and todo:
    run_items(MERGE_PROMPT, todo, MERGE_TEXT, MERGE_SCHEMA, MERGE_HASH)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

지시문 1,525자 / 묶음 {'제품·원자재': '112,546자', '서비스': '61,966자', '시설·지표·기타': '10,554자'} / 이번에 돌릴 것 0개
RUN = False: 실행하지 않았다.


## 8. 0-2단계 결과 모으기

- 모든 제안 번호가 정확히 한 제품군에 들어갔는지 센다(빠진 번호, 두 번 들어간 번호, 없는 번호).
- `families_merged.csv`: 통합한 제품군 하나에 한 줄(이름, 종류, 설명, 담은 제안 수, 나온 업종, 예시 제품).
- `family_map.csv`: 업종별 제안 하나에 한 줄(업종, 옛 key → 새 key). 1단계에서 노드의 업종별 제품군을 새 목록으로 바꿀 때 쓴다.

In [8]:
# ── 0-2단계 결과 모으기: 통합 목록과 업종별 제안 → 새 제품군 대응 ──
runs = read_runs()
ok = runs[(runs['stage'] == MERGE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == MERGE_HASH)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
pid_info = by_sector.set_index('pid')
fam_rows, map_rows, merge_notes = [], [], {}
for r in ok.itertuples():
    out = parse_result(json.loads((LLM / 'raw' / MERGE_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))
    merge_notes[r.item] = out['notes']
    for f in out['families']:
        members = [m.strip() for m in f['members']]
        known = [m for m in members if m in pid_info.index]
        fam_rows.append(dict(group=r.item, key=f['key'].strip().lower(), name_en=f['name_en'], name_ko=f['name_ko'], kind=f['kind'], scope=f['scope'],
                             n_members=len(known), sectors=' | '.join(dict.fromkeys(pid_info.loc[known, 'sector'])),
                             examples=' | '.join(list(dict.fromkeys(' | '.join(pid_info.loc[known, 'examples']).split(' | ')))[:8])))
        for m in members:
            map_rows.append(dict(group=r.item, pid=m, new_key=f['key'].strip().lower()))
merged = pd.DataFrame(fam_rows)
fmap = pd.DataFrame(map_rows)
if len(merged):
    expected = set(by_sector[by_sector['kind'].isin([k for n in ok['item'] for k in MERGE_GROUPS[n][0]])]['pid'])
    got = Counter(fmap['pid'])
    print(f'통합 제품군 {len(merged)}개', merged.groupby('group').size().to_dict(), '/ key가 겹친 것', merged['key'].duplicated().sum())
    print(f'제안 {len(expected)}개 중 빠진 것 {len(expected - set(got))}개, 두 번 이상 들어간 것 {sum(1 for v in got.values() if v > 1)}개, '
          f'없는 번호 {len(set(got) - set(pid_info.index))}개')
    fmap = fmap[fmap['pid'].isin(pid_info.index)].drop_duplicates('pid')
    fmap = fmap.join(pid_info[['sector', 'key', 'name_ko']], on='pid').rename(columns={'key': 'old_key', 'name_ko': 'old_name_ko'})
    merged.sort_values(['group', 'n_members'], ascending=[True, False]).to_csv(LLM / 'families_merged.csv', index=False, encoding='utf-8-sig')
    fmap.to_csv(LLM / 'family_map.csv', index=False, encoding='utf-8-sig')
else:
    print('아직 결과가 없다.')

통합 제품군 294개 {'서비스': 76, '시설·지표·기타': 33, '제품·원자재': 185} / key가 겹친 것 3
제안 1034개 중 빠진 것 1개, 두 번 이상 들어간 것 1개, 없는 번호 0개


## 9. 0-2단계 결과 보기

In [9]:
# ── 확인: 많이 합쳐진 제품군, 반도체·자동차 관련 제품군, 기타로 모인 것, 애매했던 점 ──
if len(merged):
    print('1) 제안을 많이 합친 제품군')
    display(merged.sort_values('n_members', ascending=False).head(15)[['key', 'name_ko', 'kind', 'n_members', 'sectors']])
    print('2) 업종별 제안 → 새 제품군 예(전기차 충전, 포장재)')
    display(fmap[fmap['old_key'].str.contains('charging|packaging', regex=True)][['sector', 'old_name_ko', 'old_key', 'new_key']].head(20))
    print('3) other로 모인 제안')
    display(fmap[fmap['new_key'] == 'other'][['sector', 'old_name_ko']].head(20))
    print('4) 애매했던 점')
    for k, v in merge_notes.items():
        if v:
            print(f'  [{k}] {v[:600]}')

1) 제안을 많이 합친 제품군


,key,name_ko,kind,n_members,sectors
290,software,소프트웨어,제품·원자재,16,건강관리 | 소프트웨어 | 은행 | 통신서비스
55,enterprise_software_and_it_services,기업용 소프트웨어 및 IT 서비스,서비스,15,"IT하드웨어 | 기계 | 미디어,교육 | 반도체 | 상사,자본재 | 소프트웨어 | 운송 | 유틸리티 | 은행 | 증권 | 통신서비스 | ..."
181,steel_products,철강 제품,제품·원자재,15,건설 | 기계 | 비철금속 | 에너지 | 자동차 | 조선 | 철강
221,automotive_parts,자동차 부품,제품·원자재,14,"건설 | 상사,자본재 | 자동차 | 철강 | 화장품,의류 | 화학"
32,other,기타,기타,13,"IT가전 | 건설 | 디스플레이 | 반도체 | 상사,자본재 | 소프트웨어 | 에너지 | 운송 | 조선 | 철강 | 필수소비재 | 화장품,..."
49,real_estate_leasing_and_management,부동산 임대 및 관리,서비스,12,"IT하드웨어 | 건설 | 미디어,교육 | 보험 | 소프트웨어 | 운송 | 은행 | 증권 | 철강 | 호텔,레저 | 화장품,의류 | 소매(유통)"
192,industrial_equipment_and_tools,산업 설비·공구,제품·원자재,11,"건강관리 | 건설 | 디스플레이 | 반도체 | 비철금속 | 상사,자본재 | 운송 | 증권 | 필수소비재 | 화장품,의류"
169,packaging_materials,포장재,제품·원자재,11,"건강관리 | 건설 | 미디어,교육 | 비철금속 | 상사,자본재 | 철강 | 필수소비재 | 화장품,의류"
167,plastic_resins,플라스틱 수지,제품·원자재,11,"IT가전 | 건설 | 비철금속 | 상사,자본재 | 에너지 | 자동차 | 화학"
80,freight_transport_and_logistics,화물운송 및 물류,서비스,11,"기계 | 미디어,교육 | 반도체 | 비철금속 | 소프트웨어 | 운송 | 유틸리티 | 자동차 | 철강 | 필수소비재 | 화학"


2) 업종별 제안 → 새 제품군 예(전기차 충전, 포장재)


,sector,old_name_ko,old_key,new_key
19,통신서비스,전기차 충전 인프라,electric_vehicle_charging_infrastructure,vehicle_charging_and_refueling_infrastructure
68,운송,전기차 충전,electric_vehicle_charging,vehicle_charging_and_refueling_services
69,유틸리티,에너지 소매 및 충전 서비스,energy_retail_and_charging_services,vehicle_charging_and_refueling_services
300,철강,산업 포장 서비스,industrial_packaging_services,industrial_packaging_services
420,소프트웨어,반도체 및 패키징,semiconductors_and_packaging,processors_and_logic_chips
436,디스플레이,반도체 패키징 소재,semiconductor_packaging_materials,semiconductor_packaging_materials
437,반도체,반도체 패키징 재료,semiconductor_packaging_materials,semiconductor_packaging_materials
440,반도체,반도체 패키징,semiconductor_packaging,semiconductor_packaging
446,반도체,반도체 패키징 장비,semiconductor_packaging_equipment,semiconductor_manufacturing_equipment
504,IT가전,전기차 충전 장비,electric_vehicle_charging_equipment,ev_charging_equipment


3) other로 모인 제안


,sector,old_name_ko
49,IT가전,미확인 원자재 및 제품
50,건설,기타 제품·서비스
51,디스플레이,미확인 제품 및 코드
52,반도체,미분류 항목
53,"상사,자본재",소비재 및 소매 서비스
54,소프트웨어,기타 미분류 항목
55,에너지,미분류 제품 및 명칭
56,운송,기타 상품·서비스
57,조선,분류 미상 상품 및 기술
58,철강,기타 철강 및 소재


4) 애매했던 점
  [시설·지표·기타] 목표 수에 맞추기보다 뉴스에서 구분해 다룰 수 있는 시설·기술·지표의 경계를 유지했다. 한 번호는 분할할 수 없어 f507의 물가 지표는 연료 가격과, f675의 생산라인은 충전시설과 함께 배정했다. f782의 개별 장비는 산업 시설에, f793의 철강·합금과 다른 소재가 섞인 제안은 기타에 배정했다.
  [서비스] 혼합 제안은 번호를 나눌 수 없어 예시와 설명의 중심에 가장 가까운 제품군에 배정했다. 따라서 일부 번호에는 해당 제품군의 경계를 벗어나는 서비스나 실물 제품이 함께 포함된다. 원자력·유전 전문 서비스처럼 근접한 전문 서비스와 보험·운송의 일부 세부 유형은 목표 개수를 고려해 묶었다.
  [제품·원자재] 혼합 제안은 번호를 나누지 않고 대표 예시가 가장 가까운 제품군에 한 번만 배정했으므로 일부 예시는 해당 제품군의 경계를 벗어난다. f380은 산업기계·부품과 유체 장비에 중복 기재되어 있어 제거가 필요하다.


## 10. 검토 반영: 제품군 확정본

0-2단계 통합본(294개)을 사람이 검토해 정한 것(2026-11-13)을 **검토표 `review/family_review.csv`**에 적고, 그 표대로 확정본을 만든다. LLM 요청은 없다.

> **다시 실행할 때 주의**: 검토표는 2026-11-13에 받은 0단계·0-2단계 LLM 결과를 보고 만든 것이다. LLM은 같은 지시문에도 실행마다 다른 제품군 이름(key)을 낼 수 있어서, 처음부터 다시 돌리면 **지금의 정리가 이번 결과에 맞지 않을 수 있다.** 이 셀은 맞지 않는 곳에서 멈추지 않고 알려 준다(이번 결과에 없어 적용하지 않은 합치기, 검토표에 없는 새 제품군 등). 알림이 나오면 **LLM 결과(6·9절)를 직접 검토해 검토표를 고친 뒤 이 셀을 다시 실행하고 다음 단계(11절)로 넘어가기를 권한다.** 검토하지 않고 넘어가도 오류는 나지 않지만, 비슷한 제품군이 따로 남은 채로 1단계가 진행된다.

**검토표 열**: `LLM_이름`(0-2단계 key, 빠진 제안이면 업종별 제안 번호), `처리`(그대로 / 합치기 / 이름·범위 고치기 / 새로 만들기 / 빠진 제안), `확정_이름`, `영어_이름`·`한글_이름`·`종류`·`범위`(이름·범위 고치기와 새로 만들기만), `모음`(넓은 모음 묶음이면 예), `메모`.

| 검토 | 정한 것 |
|---|---|
| 같은 주제가 제품·서비스·시설로 갈라짐 | AI 모델과 AI 플랫폼, 게임 셋, 데이터센터 둘, 전력 둘, 충전 서비스와 충전 시설, 통신 장비와 통신망 인프라를 합친다. 충전 **장비**는 만드는 회사가 달라 따로 둔다 |
| 소프트웨어 | 제품·서비스 구분 없이 용도별로: 기업용(업무·금융·물류, SI·IT 운영) / 보안 / 산업·설계 / 의료(의료 및 디지털 헬스 서비스에 넣음). 고객센터 위탁 운영은 기업 지원 서비스로 |
| 섞인 묶음 | 원자력·유전 전문 서비스를 원자력 엔지니어링과 유전 서비스로 나눈다 |
| 기타 | 셋을 하나로. 넓은 모음 묶음(산업 설비·공구, 산업용 금속·범용 부품, 산업용 화학 원료·첨가제, 기타)은 두되 **구체적인 제품군에 넣을 수 없을 때만** 쓴다(`catch_all`) |
| 가격·지표 | **특정 제품의 가격은 그 제품군에 붙인다**(메모리 가격 → 메모리 반도체, 유가 → 원유). 어느 제품에도 붙지 않는 물가·금리·주가지수·운임 지수는 "시장·경제 지표" 하나로 |
| 기술 | 관련 제품군 아래로(OLED 기술 → 디스플레이 패널, 5G·6G 기술 → 통신 장비). 제품이 아닌 것(다크패턴, 식품 오염물질, 하천 수위)은 기타로 |
| 작은 묶음 | 수자원(Colorado River water 하나)은 수처리·폐기물 관리로 |
| 1단계 v3 검토 후 추가 | 맞는 제품군이 없어 기타로 간 것을 위해 8개: 생활·주방용품, 의료기기 일반, 희귀·대사질환 치료제, 적응증 미상 의약품, 인터넷 검색·포털, 보안 서비스, 생활 서비스, 반려동물 용품 |

`family_map_final.csv`는 업종별 제안이 어느 확정 제품군으로 갔는지 기록이다. 1단계는 노드를 하나씩 다시 배정하므로, 이 대응은 참고용이다.

In [10]:
# ── 검토 반영: 검토표(review/family_review.csv)대로 합치기·새로 만들기·이름·범위 고치기·모음 표시. 검토표와 이번 결과가 안 맞으면 멈추지 않고 알린다 ──
REVIEW_PATH = ROOT / 'review' / 'family_review.csv'
REVIEW = pd.read_csv(REVIEW_PATH, dtype=str).fillna('')
of = lambda action: REVIEW[REVIEW['처리'] == action]
merged_keys = set(merged['key'])
NOTICE = {}                                         # 알림 종류 → 내용 목록
note = lambda kind, text: NOTICE.setdefault(kind, []).append(text)
DEFINE = {r.확정_이름: (r.영어_이름, r.한글_이름, r.종류, r.범위) for r in of('새로 만들기').itertuples()}
for r in of('이름·범위 고치기').itertuples():
    if r.확정_이름 in merged_keys:
        DEFINE[r.확정_이름] = (r.영어_이름, r.한글_이름, r.종류, r.범위)
    else:
        note('이번 결과에 없어 적용하지 않은 이름·범위 고치기', r.확정_이름)
REMAP = {}
for r in of('합치기').itertuples():
    if r.LLM_이름 not in merged_keys:
        note('이번 결과에 없어 적용하지 않은 합치기', f'{r.LLM_이름} → {r.확정_이름}')
    elif r.확정_이름 not in merged_keys | set(DEFINE):
        note('합칠 대상 제품군이 없어 그대로 둔 것', f'{r.LLM_이름} → {r.확정_이름}')
    else:
        REMAP[r.LLM_이름] = r.확정_이름
CATCH_ALL = set(REVIEW.loc[REVIEW['모음'] == '예', '확정_이름'])
MISSING = {}
for r in of('빠진 제안').itertuples():
    if r.LLM_이름 in set(by_sector['pid']):
        MISSING[r.LLM_이름] = r.확정_이름
    else:
        note('이번 0단계 결과에 없는 제안 번호(빠진 제안 줄)', f'{r.LLM_이름} → {r.확정_이름}')
for k in sorted(merged_keys - set(REVIEW['LLM_이름'])):
    note('검토표에 없는 새 제품군(LLM이 낸 그대로 둠)', k)

# 0-2단계 제품군(같은 key가 여러 묶음에 있으면 제품 → 서비스 → 기타 순서로 하나만)
order = {'제품·원자재': 0, '서비스': 1, '시설·지표·기타': 2}
base = merged.assign(o=merged['group'].map(order)).sort_values(['o', 'n_members'], ascending=[True, False]).drop_duplicates('key')
final_key = lambda k: REMAP.get(k, k)
rows = {}
for r in base.itertuples():
    k = final_key(r.key)
    if k in rows:
        continue
    src = base[base['key'] == k]
    name_en, name_ko, kind, scope = DEFINE.get(k) or ((src.iloc[0].name_en, src.iloc[0].name_ko, src.iloc[0].kind, src.iloc[0].scope) if len(src) else
                                                      (r.name_en, r.name_ko, r.kind, r.scope))
    rows[k] = dict(key=k, name_en=name_en, name_ko=name_ko, kind=kind, scope=scope, catch_all=k in CATCH_ALL)
for k, (name_en, name_ko, kind, scope) in DEFINE.items():
    rows.setdefault(k, dict(key=k, name_en=name_en, name_ko=name_ko, kind=kind, scope=scope, catch_all=k in CATCH_ALL))
final = pd.DataFrame(rows.values())

# 업종별 제안 → 확정 제품군
fmap_final = pd.concat([fmap[['pid', 'sector', 'old_key', 'old_name_ko', 'new_key']],
                        pd.DataFrame([dict(pid=p, sector=by_sector.set_index('pid').at[p, 'sector'], old_key=by_sector.set_index('pid').at[p, 'key'],
                                           old_name_ko=by_sector.set_index('pid').at[p, 'name_ko'], new_key=k) for p, k in MISSING.items()])])
fmap_final['final_key'] = fmap_final['new_key'].map(final_key)
for k in sorted(set(fmap_final['final_key']) - set(final['key'])):
    note('확정 목록에 없는 제품군으로 간 업종별 제안', k)
for p in sorted(set(by_sector['pid']) - set(fmap_final['pid'])):
    note('어느 제품군에도 들지 않은 업종별 제안(빠진 제안 줄로 넣을 수 있다)', f"{p} {by_sector.set_index('pid').at[p, 'name_ko']}")
counts = fmap_final.groupby('final_key').agg(n_proposals=('pid', 'size'), sectors=('sector', lambda s: ' | '.join(dict.fromkeys(s))))
final = final.join(counts, on='key').fillna({'n_proposals': 0, 'sectors': ''}).sort_values(['kind', 'key'])
for df, name in ((final, 'families_final.csv'), (fmap_final, 'family_map_final.csv')):   # 내용이 바뀔 때만 쓴다
    text = df.to_csv(index=False)
    if not (LLM / name).exists() or (LLM / name).read_text(encoding='utf-8-sig') != text:
        (LLM / name).write_text(text, encoding='utf-8-sig')
print(f"확정 제품군 {len(final)}개(0-2단계 {len(merged)}개) / 종류", final['kind'].value_counts().to_dict(),
      f"/ 넓은 모음 묶음 {final['catch_all'].sum()}개 / 업종별 제안 {len(fmap_final)}개 / 검토표 {len(REVIEW)}줄")
if NOTICE:
    print('\n※ 검토표가 이번 LLM 결과와 맞지 않는 곳이 있다. 검토표는 2026-11-13 결과로 만든 것이라, 다시 실행해 LLM 결과가 달라졌다면 이 정리가 적절하지 않을 수 있다.'
          '\n  아래를 보고 LLM 결과(6·9절)를 직접 검토해 review/family_review.csv를 고친 뒤 이 셀을 다시 실행하고 다음 단계로 넘어가기를 권한다.')
    for kind, items in NOTICE.items():
        print(f'  - {kind} {len(items)}개: {items[:8]}' + (' …' if len(items) > 8 else ''))
else:
    print('검토표가 이번 LLM 결과와 모두 맞는다.')

확정 제품군 273개(0-2단계 294개) / 종류 {'제품·원자재': 185, '서비스': 74, '시설·공장': 9, '기타': 4, '가격·지표': 1} / 넓은 모음 묶음 4개 / 업종별 제안 1034개 / 검토표 306줄
검토표가 이번 LLM 결과와 모두 맞는다.


In [11]:
# ── 확인: 바뀐 제품군과 종류별 수 ──
display(final[final['key'].isin(DEFINE)][['key', 'name_ko', 'kind', 'n_proposals', 'scope']])
display(final.groupby('kind').size().rename('제품군 수').to_frame().T)

,key,name_ko,kind,n_proposals,scope
253,market_indicators,시장·경제 지표,가격·지표,8.0,"물가·금리·주가지수·운임 지수 등 특정 제품군에 붙지 않는 시장 지표를 담는다. 특정 제품의 가격(메모리 가격, 유가)은 그 제품군에 붙인다."
184,other,기타,기타,19.0,"정체가 불명확하거나 어느 제품군에도 맞지 않는 항목과 제품이 아닌 개념(다크패턴, 식품 오염물질, 하천 수위 등)을 담는다."
84,electricity_and_heat_supply,전력·열 공급,서비스,5.0,전력·증기·지역난방의 생산·판매와 재생에너지 발전 사업을 담고 발전 설비와 발전소 건설은 제외한다.
0,enterprise_software_and_it_services,기업용 소프트웨어·IT 서비스,서비스,36.0,"기업 업무·금융·물류용 소프트웨어와 시스템 통합·IT 운영을 담고 보안, 산업·설계, 의료 소프트웨어는 제외한다."
212,healthcare_and_digital_health_services,의료·디지털 헬스 서비스,서비스,4.0,진료·환자 모니터링·디지털 건강관리와 의료 AI·의료 소프트웨어를 담고 의약품과 의료기기 하드웨어는 제외한다.
263,industrial_and_engineering_software,산업·설계 소프트웨어,서비스,0.0,스마트팩토리·설계(CAD·EDA)·시뮬레이션 등 제조·엔지니어링용 소프트웨어를 담고 산업 자동화 장비 자체는 제외한다.
269,internet_search_and_portals,인터넷 검색·포털,서비스,0.0,검색엔진·포털·지도 등 인터넷 플랫폼 서비스를 담고 소셜미디어와 전자상거래는 제외한다.
222,nuclear_services,원자력 엔지니어링 서비스,서비스,3.0,원자력 발전소 설계·핵연료 가공·해체 서비스를 담고 원자로 설비와 발전소 운영·정비는 제외한다.
264,oilfield_services,유전 서비스,서비스,0.0,시추·완결·유정 폐쇄 등 석유·가스 유전 서비스를 담고 생산설비 자체와 원유는 제외한다.
271,personal_and_lifestyle_services,생활 서비스,서비스,0.0,웨딩·세차·사진관·미용 등 개인 대상 생활 서비스를 담고 의료·교육·숙박·외식은 제외한다.


kind,가격·지표,기타,서비스,시설·공장,제품·원자재
제품군 수,1,4,74,9,185


## 11. 1단계 입력: 노드 점검·분류

노드마다 언급을 보여 주고, 섞인 것은 나누고, 그룹마다 종류·제품군·대표 이름을 받는다. 지시문: `prompts/product_node_v4.md`(최종본. 버전별 차이는 12절 표). 지시문 끝에 **확정 제품군 목록**(`families_final.csv`, key | 한글 이름 | 종류 | 범위)을 붙이고, 결과 스키마의 `family`는 그 key만 고를 수 있게 한다.

- 언급 줄: 같은 (자리, 원래 이름, 영어 이름, 출처, 범주)는 한 줄로 묶고 개수를 적는다. 출처는 공시면 회사(업종), 뉴스면 기사(업종)다.
- 노드 하나에 **언급 줄은 많이 나온 순서로 15줄까지** 보인다. 보이지 않는 줄은 첫 번째 그룹으로 간다.
- 결과는 줄번호(m1, m2, …)로 언급을 가리킨다. 적용할 때 줄번호를 실제 언급으로 되돌린다.

In [12]:
# ── 1단계 입력: 노드마다 언급 줄, 확정 제품군 목록을 붙인 지시문, 제품군 key만 고르게 한 스키마 ──
NODE_PROMPT = 'product_node_v4'           # 버전별 차이는 12절 표
MAX_LINES = 15
FAMILIES = pd.read_csv(LLM / 'families_final.csv', dtype=str).fillna('')
co_info = pd.read_csv(ROOT / 'source' / 'dart' / 'parsed' / 'companies.csv', dtype=str).fillna('').set_index('stock_code')
art_sector = {aid: '·'.join(a['sectors']) for aid, a in news.items()}
mentions = used.copy()                                # 그래프에 쓰이는 언급(2절)
mentions['place'] = [('뉴스 ' + f) if s == 'news' else f for s, f in zip(mentions['source'], mentions['field'])]
mentions['src'] = [f"기사({art_sector.get(d) or '업종 없음'})" if s == 'news' else f"{co_info['stock_name'].get(d, d)}({co_info['wi26_sector'].get(d, '')})"
                   for s, d in zip(mentions['source'], mentions['doc'])]
LINE_KEYS = ['place', 'name', 'name_en', 'src', 'category']
lines_all = (mentions.groupby(['node_id'] + LINE_KEYS).size().rename('n').reset_index()
             .sort_values(['node_id', 'n', 'place', 'name'], ascending=[True, False, True, True]))
lines_all['line'] = 'm' + (lines_all.groupby('node_id').cumcount() + 1).astype(str)
NODE_INFO = nodes.set_index('id')


def node_block(nid):
    """노드 하나의 입력 글: 머리줄과 언급 줄(많이 나온 순서로 MAX_LINES줄까지)."""
    g = lines_all[lines_all['node_id'] == nid]
    head = f"## {nid} | {NODE_INFO.at[nid, 'sector'] or '업종 없음'} | {int(g['n'].sum())}"
    body = [f"{r.line} | {r.place} | {r.name} | {r.name_en} | {r.src} | {r.category} | {r.n}" for r in g.head(MAX_LINES).itertuples()]
    more = [f'(보이지 않는 언급 줄 {len(g) - MAX_LINES}개, 언급 {int(g["n"].iloc[MAX_LINES:].sum())}개)'] if len(g) > MAX_LINES else []
    return '\n'.join([head] + body + more)


family_table = '\n'.join(f"{r.key} | {r.name_ko}{' [모음]' if r.catch_all == 'True' else ''} | {r.kind} | {r.scope}" for r in FAMILIES.itertuples())
NODE_TEXT = ((ROOT / 'prompts' / f'{NODE_PROMPT}.md').read_text(encoding='utf-8')
             + '\n\n## 제품군 목록\n\nkey | 한글 이름 | 종류 | 범위\n' + family_table + CODEX_SUFFIX)
NODE_HASH = text_hash(NODE_TEXT + MODEL + EFFORT)
node_schema = {
    'type': 'object', 'additionalProperties': False, 'required': ['nodes', 'notes'],
    'properties': {
        'nodes': {'type': 'array', 'items': {
            'type': 'object', 'additionalProperties': False, 'required': ['node_id', 'groups'],
            'properties': {
                'node_id': {'type': 'string'},
                'groups': {'type': 'array', 'items': {
                    'type': 'object', 'additionalProperties': False, 'required': ['lines', 'kind', 'family', 'name_en', 'name_ko', 'note'],
                    'properties': {'lines': {'type': 'array', 'items': {'type': 'string'}},
                                   'kind': {'type': 'string', 'enum': ['제품·원자재', '서비스', '시설·공장', '가격·지표', '기타']},
                                   'family': {'type': 'string', 'enum': sorted(FAMILIES['key'])},
                                   'name_en': {'type': 'string'}, 'name_ko': {'type': 'string'}, 'note': {'type': 'string'}}}}}}},
        'notes': {'type': 'string'}}}
NODE_SCHEMA = LLM / 'schemas' / f'{NODE_PROMPT}.schema.json'
NODE_SCHEMA.parent.mkdir(exist_ok=True)
NODE_SCHEMA.write_text(json.dumps(node_schema, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'노드 {len(nodes):,}개 / 언급 줄 {len(lines_all):,}개 / 15줄을 넘는 노드 {(lines_all.groupby("node_id").size() > MAX_LINES).sum()}개'
      f' / 지시문 {len(NODE_TEXT):,}자(제품군 목록 {len(FAMILIES)}개 포함)')
print(node_block('P:wafer'))

노드 12,643개 / 언급 줄 20,852개 / 15줄을 넘는 노드 64개 / 지시문 28,636자(제품군 목록 273개 포함)
## P:wafer | 반도체 | 15
m1 | 원재료 | WAFER | Wafer | HD현대에너지솔루션(에너지) |  | 1
m2 | 원재료 | WAFER | Wafer | KEC(반도체) |  | 1
m3 | 원재료 | WAFER | Wafer | SK스퀘어(반도체) |  | 1
m4 | 원재료 | WAFER | Wafer | SK하이닉스(반도체) |  | 1
m5 | 원재료 | WAFER | Wafer | 드림텍(IT하드웨어) |  | 1
m6 | 원재료 | WAFER | Wafer | 한국전자홀딩스(반도체) |  | 1
m7 | 원재료 | WAFER | wafer | 광전자(IT하드웨어) |  | 1
m8 | 원재료 | WAFER 외 | Wafer and others | LX세미콘(디스플레이) |  | 1
m9 | 원재료 | WAFER 외 | Wafer and others | LX홀딩스(상사,자본재) |  | 1
m10 | 원재료 | Wafer | Wafer | 대덕(IT하드웨어) |  | 1
m11 | 원재료 | 웨이퍼 | Wafer | 아이쓰리시스템(상사,자본재) |  | 1
m12 | 원재료 | 웨이퍼 | wafer | 제주반도체(반도체) |  | 1
m13 | 제품 | 웨이퍼 | wafer | 한솔테크닉스(IT하드웨어) | LED 소재 | 1
m14 | 제품 | 웨하스 | Wafers | 크라운제과(필수소비재) | 제과 | 1
m15 | 제품 | 웨하스 | wafer biscuits | 크라운해태홀딩스(필수소비재) | 과자 | 1


## 12. 1단계 시범 묶음과 정답 확인 규칙

지시문을 다듬을 때 쓰는 고정 묶음이다(`sample_node_v1.csv`, 처음 한 번 만든다).

- **어려운 사례 33개**: 섞인 노드(웨이퍼–웨하스, 필름–영화매출, 정제–태블릿PC, 에틸렌–에틸렌 플랜트, 증기–Steam, 스위치–Nintendo Switch, 케이스), 여러 품목 이름, 가격·지표, 서비스, 시설, 이름에 "plant"가 들어간 식품(plant protein) 같은 함정, 그리고 섞이지 않은 대표 제품(HBM, 패키지 기판, 원유, 포토마스크, GLP-1, 양극재 등).
- **무작위 37개**: 일반적인 노드에서 얼마나 맞는지 본다. 정답 규칙 없이 결과를 사람이 본다.
- 두 건으로 나눠 보낸다(노드 35개씩).

정답 확인 규칙(`CHECKS`): `apart`(두 언급이 다른 그룹), `family`(그 언급의 제품군이 정답 중 하나), `kind`(노드의 첫 그룹 종류), `first`(첫 그룹의 제품군), `groups`(그룹 수 이상), `groups1`(그룹 하나이고 제품군이 정답).

| 지시문 | 정답 확인 | 고친 것 |
|---|---|---|
| v1 | 41/43(틀린 2개는 입력에 없는 Steam 언급을 본 시범 설계 실수) | - |
| v2 | 43/44 | 첫 그룹은 언급이 가장 많은 뜻(필름 노드에서 영화를 첫 그룹에 둠), 같은 물질의 원료와 완제품은 나누지 않음(세프포독심), 막연한 이름은 회사 사업으로 짐작(삼화콘덴서 CASE를 기타로 보냄) |
| v3 | 44/44. 정답 규칙이 없는 노드는 애매한 5개만 v2와 달라짐(CKD-843 치료 분야, 리스·로봇 렌탈을 나눌지, V2H 시스템) | 전체 배치 결과 검토: 기타로 간 그룹 246개 중 171개가 막연하지 않은 이름(맞는 제품군이 없음, 기술·코드명을 둘 곳을 모름) |
| v4 | Codex 62/62(v3가 기타로 보낸 18개 모두 새 제품군·규칙대로). API 38/39(틀린 1개는 태양광 회사의 웨이퍼를 태양광 제조로 뗀 것으로, 정답 규칙이 너무 단순했다) | 제품군 8개 추가(10절), 기술·공정·플랫폼은 쓰이는 제품군에, 제약사 개발 코드명은 적응증 미상 의약품에. 시범에 v3가 기타로 보낸 노드 18개(`sample_3`)를 더함 | 짐작은 이름 자체가 막연할 때만. 뜻이 있는 이름은 회사가 다른 뜻의 사업을 하는 게 분명할 때만 다르게 봄(v2가 미스토홀딩스의 원재료 "원유"를 우유 원료로 뗌) |

> **다시 실행할 때**: 시범 사례와 정답 확인은 2026-11 데이터와 LLM 결과에서 고른 것이다. 데이터를 새로 만들면 없는 사례는 건너뛰고 알린다. 새 데이터로 지시문을 다시 다듬는다면 그 데이터에서 어려운 사례를 다시 골라 넣기를 권한다.

In [13]:
# ── 시범 묶음(고정)과 정답 확인 규칙 ──
HARD = ['P:wafer', 'P:film', 'P:pharmaceutical tablet', 'P:ethylene', 'P:steam', 'P:switch', 'P:case',
        'P:electrical steel electrical insulating oil aluminum copper', 'P:lens sensor integrated circuit hard drive and memory',
        'P:manganese ore and silica ore', 'P:beverage concentrate sugar and fruit juice',
        'P:memory price', 'P:core consumer price index', 'P:US domestic steel price', 'P:fuel price', 'P:freight rate',
        'P:leasing', 'P:construction management', 'P:venture capital investment', 'P:industrial plant construction', 'P:virtual power plant',
        'P:high bandwidth memory', 'P:semiconductor package substrate', 'P:crude oil', 'P:memory chip', 'P:photomask',
        'P:GLP 1 receptor agonist', 'P:cathode active material', 'P:hot rolled coil', 'P:plant protein', 'P:television drama',
        'P:submarine cable', 'P:robot']
SAMPLE_PATH = LLM / 'sample_node_v1.csv'
if not SAMPLE_PATH.exists():
    missing = [n for n in HARD if n not in NODE_INFO.index]
    if missing:
        print(f'※ 어려운 사례 {len(missing)}개는 이번 데이터에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다):', missing[:5])
    HARD = [n for n in HARD if n in NODE_INFO.index]
    rest = nodes[~nodes['id'].isin(HARD)].sample(37, random_state=11)['id'].tolist()
    order_ = pd.Series(HARD + rest).sample(frac=1, random_state=5).tolist()
    pd.DataFrame({'node_id': order_, 'hard': [n in HARD for n in order_]}).to_csv(SAMPLE_PATH, index=False, encoding='utf-8-sig')
SAMPLE = pd.read_csv(SAMPLE_PATH, dtype=str)
if not SAMPLE['node_id'].isin(NODE_INFO.index).all():
    print(f'※ 시범 묶음의 노드 {(~SAMPLE["node_id"].isin(NODE_INFO.index)).sum()}개가 이번 데이터에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다)')
    SAMPLE = SAMPLE[SAMPLE['node_id'].isin(NODE_INFO.index)]
NODE_SAMPLE_INPUTS = {f'sample_{i + 1}': '\n\n'.join(node_block(n) for n in SAMPLE['node_id'][i * 35:(i + 1) * 35]) for i in range(2)}
# v4: v3 전체 결과에서 기타로 간 노드 중 새 제품군·규칙으로 갈 곳이 생긴 것
EXTRA_V4 = ['P:tableware', 'P:tumbler', 'P:kitchenware', 'P:hunter syndrome treatment', 'P:KL1333 rare disease treatment', 'P:unicracking',
            'P:C4 oleflex', 'P:antibody drug conjugate platform technology', 'P:DA 5215', 'P:DA 2803', 'P:petroleum and petrochemical',
            'P:google search', 'P:medical device', 'P:US container import', 'P:artificial intelligence', 'P:physical security', 'P:wedding', 'P:pet supply']
EXTRA_V4 = [n for n in EXTRA_V4 if n in NODE_INFO.index]
NODE_SAMPLE_INPUTS['sample_3'] = '\n\n'.join(node_block(n) for n in EXTRA_V4)
drug = lambda f: 'drug' in f or 'pharma' in f or 'medicine' in f
CHECKS = [  # (노드, 확인, 언급을 찾는 말 또는 None, 정답)
    ('P:wafer', 'apart', ('웨하스', 'WAFER'), None), ('P:wafer', 'family', '웨하스', {'bakery_and_confectionery'}),
    ('P:wafer', 'family', 'WAFER', {'semiconductor_wafers'}),
    ('P:film', 'apart', ('영화매출', 'Process film'), None), ('P:film', 'family', '영화매출', {'entertainment_content', 'content_production_and_distribution'}),
    ('P:pharmaceutical tablet', 'apart', ('정제', '태블릿PC'), None), ('P:pharmaceutical tablet', 'family', '정제', drug),
    ('P:pharmaceutical tablet', 'family', '태블릿PC', {'consumer_electronics', 'computers_and_data_storage', 'equipment_rental_and_leasing'}),
    ('P:ethylene', 'apart', ('Ethylene plant', 'C2'), None), ('P:ethylene', 'family', 'C2', {'basic_petrochemicals'}),
    ('P:ethylene', 'family', 'Ethylene plant', {'industrial_and_energy_plant_construction', 'industrial_plants_and_facilities'}),
    ('P:steam', 'groups1', None, {'electricity_and_heat_supply'}),     # 게임 플랫폼 Steam 언급은 그래프에 쓰이지 않아 입력에 없다
    ('P:film', 'first', None, {'plastic_films_and_packaging', 'semiconductor_process_materials', 'display_materials', 'electronic_chemicals'}),
    ('P:cefpodoxime proxetil', 'groups1', None, {'anti_infective_drugs'}), ('P:case', 'family', '삼화콘덴서', {'passive_electronic_components'}),
    ('P:switch', 'apart', ('Nintendo Switch', '보안'), None), ('P:switch', 'family', 'Nintendo Switch', {'consumer_electronics'}),
    ('P:electrical steel electrical insulating oil aluminum copper', 'groups', None, 3),
    ('P:lens sensor integrated circuit hard drive and memory', 'groups', None, 3), ('P:manganese ore and silica ore', 'groups', None, 2),
    ('P:memory price', 'kind', None, {'가격·지표'}), ('P:memory price', 'family', None, {'memory_chips'}),
    ('P:core consumer price index', 'kind', None, {'가격·지표'}), ('P:core consumer price index', 'family', None, {'market_indicators'}),
    ('P:US domestic steel price', 'kind', None, {'가격·지표'}), ('P:US domestic steel price', 'family', None, {'steel_products'}),
    ('P:fuel price', 'kind', None, {'가격·지표'}), ('P:fuel price', 'family', None, {'refined_petroleum_fuels'}),
    ('P:freight rate', 'kind', None, {'가격·지표'}),
    ('P:construction management', 'kind', None, {'서비스'}), ('P:venture capital investment', 'family', None, {'private_equity_and_venture_capital'}),
    ('P:industrial plant construction', 'kind', None, {'서비스'}),
    ('P:industrial plant construction', 'family', None, {'industrial_and_energy_plant_construction'}),
    ('P:virtual power plant', 'kind', None, {'서비스', '시설·공장'}),
    ('P:high bandwidth memory', 'groups1', None, {'memory_chips'}), ('P:semiconductor package substrate', 'groups1', None, {'semiconductor_package_substrates'}),
    ('P:crude oil', 'groups1', None, {'crude_oil'}), ('P:memory chip', 'groups1', None, {'memory_chips'}),
    ('P:photomask', 'groups1', None, {'semiconductor_lithography_materials'}), ('P:GLP 1 receptor agonist', 'groups1', None, {'diabetes_and_obesity_drugs'}),
    ('P:cathode active material', 'groups1', None, {'battery_cathode_materials'}), ('P:hot rolled coil', 'groups1', None, {'steel_products'}),
    ('P:plant protein', 'family', None, {'food_ingredients'}),
    ('P:tableware', 'family', None, {'household_and_kitchen_goods'}), ('P:tumbler', 'family', None, {'household_and_kitchen_goods'}),
    ('P:kitchenware', 'family', None, {'household_and_kitchen_goods'}), ('P:hunter syndrome treatment', 'family', None, {'rare_and_metabolic_disease_drugs'}),
    ('P:KL1333 rare disease treatment', 'family', None, {'rare_and_metabolic_disease_drugs'}),
    ('P:unicracking', 'family', None, {'industrial_and_energy_plant_construction'}), ('P:C4 oleflex', 'family', None, {'industrial_and_energy_plant_construction'}),
    ('P:antibody drug conjugate platform technology', 'family', None, {'research_testing_and_engineering_services', 'oncology_drugs'}),
    ('P:DA 5215', 'family', None, drug), ('P:DA 2803', 'family', None, drug),
    ('P:petroleum and petrochemical', 'family', None, {'refined_petroleum_fuels', 'basic_petrochemicals', 'petroleum_feedstocks_and_lubricants'}),
    ('P:google search', 'family', None, {'internet_search_and_portals'}), ('P:medical device', 'family', None, {'medical_devices_general'}),
    ('P:US container import', 'family', None, {'market_indicators'}), ('P:artificial intelligence', 'family', None, {'ai_models_and_platforms'}),
    ('P:physical security', 'family', None, {'security_services'}), ('P:wedding', 'family', None, {'personal_and_lifestyle_services'}),
    ('P:pet supply', 'family', None, {'pet_products'}), ('P:television drama', 'family', None, {'entertainment_content', 'content_production_and_distribution'}),
]
gone = [c for c in CHECKS if c[0] not in NODE_INFO.index]
if gone:
    print(f'※ 정답 확인 {len(gone)}개는 대상 노드가 이번 데이터에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다)')
CHECKS = [c for c in CHECKS if c[0] in NODE_INFO.index]
print(f'시범 노드 {len(SAMPLE)}개(어려운 사례 {SAMPLE.hard.eq("True").sum()}개) / 정답 확인 {len(CHECKS)}개 / 묶음',
      {k: f'{len(v):,}자' for k, v in NODE_SAMPLE_INPUTS.items()})

시범 노드 70개(어려운 사례 33개) / 정답 확인 62개 / 묶음 {'sample_1': '9,462자', 'sample_2': '8,869자', 'sample_3': '3,489자'}


## 13. 1단계 시범 실행 (Codex)

시범 묶음 두 건을 Codex로 보낸다. 지시문을 고치면 다시 돌린다(같은 지시문·입력이면 건너뛴다).

In [14]:
# ── 1단계 시범 실행 ──
runs = read_runs()
done = set(runs.loc[(runs['stage'] == NODE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: v for k, v in NODE_SAMPLE_INPUTS.items() if (k, text_hash(v)) not in done}
print(f'이번에 돌릴 것 {len(todo)}건')
if RUN and todo:
    run_items(NODE_PROMPT, todo, NODE_TEXT, NODE_SCHEMA, NODE_HASH)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

이번에 돌릴 것 0건
RUN = False: 실행하지 않았다.


## 14. 1단계 시범 채점

정답 확인 규칙을 하나씩 본다. 함께 형식 점검도 한다: 입력 노드가 모두 결과에 있는지, 모든 줄번호가 어느 그룹엔가 들어갔는지, 없는 줄번호를 쓰지 않았는지.

In [15]:
# ── 1단계 시범 채점 ──
runs = read_runs()
ok = runs[(runs['stage'] == NODE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH) & runs['item'].str.startswith('sample_')]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
RESULT = {}
for r in ok.itertuples():
    for n in parse_result(json.loads((LLM / 'raw' / NODE_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))['nodes']:
        RESULT[n['node_id']] = n['groups']


def find_lines(nid, text):
    """노드의 언급 줄 중 이름·영어 이름·자리에 text가 들어간 줄번호. '뉴스|Steam'은 자리에 뉴스, 이름에 Steam."""
    g = lines_all[lines_all['node_id'] == nid]
    if '|' in text:
        a, b = text.split('|')
        g = g[g['place'].str.contains(a) & (g['name'].str.contains(b) | g['name_en'].str.contains(b))]
    else:
        g = g[g['name'].str.contains(text, regex=False) | g['name_en'].str.contains(text, regex=False) | g['src'].str.contains(text, regex=False)
              | g['category'].str.contains(text, regex=False)]
    return set(g['line'])


def groups_of(nid, ls, res=None):
    """줄번호들이 들어간 그룹 번호."""
    return {i for i, gr in enumerate((RESULT if res is None else res).get(nid, [])) for l in gr['lines'] if l in ls}


def check(nid, what, arg, answer, res=None):
    """정답 확인 하나. (맞음, 설명). res를 주면 그 결과로 본다."""
    res = RESULT if res is None else res
    gs = res.get(nid)
    if gs is None:
        return False, '결과 없음'
    ok_ = lambda f: answer(f) if callable(answer) else f in answer
    if what == 'apart':
        a, b = (groups_of(nid, find_lines(nid, t), res) for t in arg)
        return bool(a and b and not (a & b)), f'그룹 {sorted(a)} / {sorted(b)}'
    if what == 'family':
        idx = groups_of(nid, find_lines(nid, arg), res) if arg else {0}
        fams = [gs[i]['family'] for i in idx]
        return bool(fams) and all(ok_(f) for f in fams), str(fams)
    if what == 'kind':
        return gs[0]['kind'] in answer, gs[0]['kind']
    if what == 'groups':
        return len(gs) >= answer, f'{len(gs)}개'
    if what == 'first':
        return ok_(gs[0]['family']), gs[0]['family']
    if what == 'groups1':
        return len(gs) == 1 and ok_(gs[0]['family']), f"{len(gs)}개, {[g['family'] for g in gs]}"


if RESULT:
    graded = pd.DataFrame([dict(node=n, check=w, arg=str(a), ok=c[0], got=c[1]) for n, w, a, ans in CHECKS for c in [check(n, w, a, ans)]])
    lost = [n for n in list(SAMPLE['node_id']) + EXTRA_V4 if n not in RESULT]
    bad_lines = [(n, l) for n, gs in RESULT.items() for g in gs for l in g['lines'] if l not in set(lines_all.loc[lines_all['node_id'] == n, 'line'])]
    unplaced = [(n, l) for n in RESULT for l in lines_all.loc[lines_all['node_id'] == n, 'line'].head(MAX_LINES)
                if not any(l in g['lines'] for g in RESULT[n])]
    print(f"정답 확인 {graded.ok.sum()}/{len(graded)} 맞음 / 결과에 없는 노드 {len(lost)} / 없는 줄번호 {len(bad_lines)} / 그룹에 안 들어간 줄 {len(unplaced)}")
    print(f"나눈 노드 {sum(len(g) > 1 for g in RESULT.values())}개 / 그룹 종류", Counter(g['kind'] for gs in RESULT.values() for g in gs))
    display(graded[~graded.ok])
else:
    print('아직 결과가 없다.')

정답 확인 62/62 맞음 / 결과에 없는 노드 0 / 없는 줄번호 0 / 그룹에 안 들어간 줄 0
나눈 노드 13개 / 그룹 종류 Counter({'제품·원자재': 86, '서비스': 18, '가격·지표': 6})


,node,check,arg,ok,got


In [16]:
# ── 시범 결과 전체 보기(노드 → 그룹: 종류 / 제품군 / 이름 / 담은 언급) ──
if RESULT:
    for nid in list(SAMPLE['node_id']) + EXTRA_V4:
        g = lines_all[lines_all['node_id'] == nid].set_index('line')
        print(f"■ {nid}{' (어려운 사례)' if nid in HARD else ''}")
        for gr in RESULT.get(nid, []):
            names = ', '.join(dict.fromkeys(g.at[l, 'name'] for l in gr['lines'] if l in g.index))[:90]
            print(f"   {gr['kind']:6} {gr['family']:40} {gr['name_en']} / {gr['name_ko']}  ← {names}" + (f"  ({gr['note']})" if gr['note'] else ''))

■ P:AV delivery developer portal
   서비스    mobile_and_service_robots                autonomous delivery developer platform / 자율주행 배송 개발 플랫폼  ← AV Delivery Developer Portal  (로봇 배송 기술을 위한 개발 플랫폼이다.)
■ P:golf gear
   제품·원자재 sports_and_leisure_goods                 golf gear / 골프 용품  ← Golf Gear
■ P:hot rolled coil (어려운 사례)
   제품·원자재 steel_products                           hot-rolled steel coil / 열연코일  ← hot rolled steel, hot-rolled coil, hot-rolled coils, HR COIL, HR COIL 외, HR Coil, HR Coil 
■ P:digital transformation solution
   서비스    enterprise_software_and_it_services      digital transformation solutions / 디지털 전환 솔루션  ← DX솔루션
■ P:beverage concentrate sugar and fruit juice (어려운 사례)
   제품·원자재 food_ingredients                         beverage concentrates / 음료 원액  ← 원액, 당류, 과즙 등  (나열된 원료를 품목별로 분리했다.)
   제품·원자재 food_ingredients                         sugars / 당류  ← 원액, 당류, 과즙 등
   제품·원자재 food_ingredients                         fruit juice concentrates / 과즙  ← 원액, 당류, 과즙 등  (음료 제조용 원

## 15. 1단계 API 확인

전체는 OpenAI 배치 API로 돌리므로, 같은 지시문이 API에서도 Codex와 같은 결과를 내는지 먼저 본다(뉴스 v4는 같은 지시문이 Codex에서 기타 4%, API에서 22%였다). 어려운 사례 20개 노드를 API 일반 요청 1건으로 보내 정답 확인을 하고, Codex v3 결과와 언급 줄마다 제품군·종류가 같은지 비교한다.

- API는 구조화 출력(엄격 모드)으로 결과 형식을 정한다. 지시문은 Codex용 결과 형식 문단을 뺀 것이다.
- 인증키는 `.env`의 `OPENAI_API_KEY`에서 읽고, 화면·저장 파일·오류 메시지에 남기지 않는다(`sk-…`, `org-…`는 가린다).
- 가격(100만 토큰당, 2026-11-12 공식 가격표): `gpt-6.1-terra` 입력 $1.00, 캐시 입력 $0.05, 출력 $5.00. 배치는 모두 50%.

In [17]:
# ── API 실행 도구: 엄격 스키마 일반 요청 한 건을 보내고 원본을 저장한다 ──
PRICES = {'gpt-6.1-terra': (1.00, 0.05, 5.00)}
mask = lambda s: re.sub(r'org-[A-Za-z0-9]{6,}', 'org-***', re.sub(r'sk-[A-Za-z0-9_\-]{6,}', 'sk-***', str(s)))   # 키·조직 ID를 가린다


def api_cost(model, input_tokens, cached_tokens, output_tokens, batch=False):
    """토큰 수로 OpenAI 비용(달러)을 계산한다. 배치는 절반."""
    p_in, p_cache, p_out = PRICES[model]
    usd = ((input_tokens - cached_tokens) * p_in + cached_tokens * p_cache + output_tokens * p_out) / 1e6
    return usd / 2 if batch else usd


def env_value(name):
    """환경변수 → 프로젝트 루트 .env → graph_db/.env 순서로 값을 찾는다. 값은 화면에 쓰지 않는다."""
    if os.environ.get(name):
        return os.environ[name]
    for env_file in (ROOT.parent / '.env', ROOT / '.env'):
        if env_file.exists():
            for line in env_file.read_text(encoding='utf-8-sig').splitlines():
                key, _, value = line.partition('=')
                if key.strip() == name and value.strip():
                    return value.strip().strip('"\'')
    return None


_client = None


def openai_client():
    """OpenAI 클라이언트를 한 번만 만든다."""
    global _client
    if _client is None:
        import openai
        key = env_value('OPENAI_API_KEY')
        assert key, '.env에 OPENAI_API_KEY가 없다.'
        _client = openai.OpenAI(api_key=key, max_retries=3)
    return _client


NODE_TEXT_API = NODE_TEXT[:-len(CODEX_SUFFIX)]             # Codex용 결과 형식 문단을 뺀 지시문
NODE_HASH_API = text_hash(NODE_TEXT_API + MODEL + EFFORT)


def api_body(text):
    """1단계 요청 본문(Responses API, 엄격 스키마). 일반 요청과 배치가 같은 본문을 쓴다."""
    return dict(model=MODEL, instructions=NODE_TEXT_API, input=text, reasoning={'effort': EFFORT}, store=False,
                text={'format': {'type': 'json_schema', 'name': 'product_nodes', 'schema': node_schema, 'strict': True}})


def api_result(body):
    """API 응답 본문에서 결과 JSON을 꺼낸다."""
    text = next((c.get('text') or '' for item in body.get('output') or [] if item.get('type') == 'message'
                 for c in item.get('content') or [] if c.get('type') == 'output_text'), '')
    return json.loads(text)


def run_api(stage, item, text, body_fn=None, schema=None, prompt_hash=None):
    """API 일반 요청 한 건을 보내고 원본을 저장한 뒤 실행 기록 한 줄을 돌려준다. 기본은 1단계(본문·스키마·지시문 해시), 2단계는 따로 넘긴다."""
    body_fn, schema, prompt_hash = body_fn or api_body, schema or node_schema, prompt_hash or NODE_HASH_API
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    raw_dir = LLM / 'raw' / stage / f'api-{MODEL}'
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_file = f"{item}_{now.strftime('%Y%m%dT%H%M%S')}.json"
    row = dict(stage=stage, item=item, input_hash=text_hash(text), prompt_hash=prompt_hash, runner='api', model=MODEL, effort=EFFORT,
               run_utc=now.isoformat(), raw_file=raw_file, status='', error='')
    t0 = time.time()
    try:
        body = openai_client().responses.create(**body_fn(text)).model_dump()
    except Exception as e:
        return row | dict(status='error', error=mask(e)[:300], duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    (raw_dir / raw_file).write_text(json.dumps(body, ensure_ascii=False, indent=1), encoding='utf-8')
    u = body.get('usage') or {}
    cached = (u.get('input_tokens_details') or {}).get('cached_tokens') or 0
    row.update(input_tokens=u.get('input_tokens'), cache_read_tokens=cached, output_tokens=u.get('output_tokens'),
               reasoning_tokens=(u.get('output_tokens_details') or {}).get('reasoning_tokens'),
               cost_usd=round(api_cost(MODEL, u.get('input_tokens') or 0, cached, u.get('output_tokens') or 0), 5))
    if body.get('status') != 'completed':
        return row | dict(status='error', error=mask(json.dumps(body.get('incomplete_details') or body.get('error') or body.get('status')))[:300])
    errs = [e.message for e in jsonschema.Draft202012Validator(schema).iter_errors(api_result(body))]
    return row | dict(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])


if 'cost_usd' not in RUN_COLUMNS:
    RUN_COLUMNS.append('cost_usd')
    if RUNS_PATH.exists():                            # 열이 늘었으면 기존 기록을 새 열 구성으로 옮겨 쓴다
        read_runs().to_csv(RUNS_PATH, index=False, encoding='utf-8-sig')

In [18]:
# ── 1단계 API 확인: 어려운 사례 20개를 일반 요청 1건으로 ──
API_CHECK_NODES = ['P:wafer', 'P:film', 'P:pharmaceutical tablet', 'P:ethylene', 'P:switch', 'P:case', 'P:crude oil',
                   'P:electrical steel electrical insulating oil aluminum copper', 'P:lens sensor integrated circuit hard drive and memory',
                   'P:manganese ore and silica ore', 'P:memory price', 'P:US domestic steel price', 'P:core consumer price index', 'P:fuel price',
                   'P:industrial plant construction', 'P:venture capital investment', 'P:cefpodoxime proxetil', 'P:high bandwidth memory',
                   'P:plant protein', 'P:semiconductor package substrate',
                   'P:tableware', 'P:hunter syndrome treatment', 'P:unicracking', 'P:DA 5215', 'P:google search']    # v4에서 더함
api_text = '\n\n'.join(node_block(n) for n in API_CHECK_NODES)
runs = read_runs()
done = runs[(runs['stage'] == NODE_PROMPT) & (runs['item'] == 'api_check') & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH_API)
            & (runs['input_hash'] == text_hash(api_text))]
print(f'노드 {len(API_CHECK_NODES)}개, 입력 {len(api_text):,}자 / 이미 끝남 {len(done) > 0}')
if RUN and not len(done):
    row = run_api(NODE_PROMPT, 'api_check', api_text)
    pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
    print(f"  {row['status']} {row.get('duration_s')}초 입력 {row.get('input_tokens')}(캐시 {row.get('cache_read_tokens')}) 출력 {row.get('output_tokens')}"
          f"(추론 {row.get('reasoning_tokens')}) 비용 ${row.get('cost_usd')} {row.get('error') or ''}")
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

노드 25개, 입력 8,444자 / 이미 끝남 True
RUN = False: 실행하지 않았다.


In [19]:
# ── API 확인 채점: 정답 확인, Codex v3와 언급 줄마다 같은지 ──
runs = read_runs()
r = runs[(runs['stage'] == NODE_PROMPT) & (runs['item'] == 'api_check') & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH_API)]
if len(r):
    r = r.sort_values('run_utc').iloc[-1]
    RESULT_API = {n['node_id']: n['groups'] for n in api_result(json.loads((LLM / 'raw' / NODE_PROMPT / f'api-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))['nodes']}
    graded_api = pd.DataFrame([dict(node=n, check=w, ok=c[0], got=c[1]) for n, w, a, ans in CHECKS if n in API_CHECK_NODES
                               for c in [check(n, w, a, ans, RESULT_API)]])
    per_line = lambda res, n: {l: (g['family'], g['kind']) for g in res.get(n, []) for l in g['lines']}
    same = [(n, l, per_line(RESULT, n).get(l), v) for n in API_CHECK_NODES for l, v in per_line(RESULT_API, n).items()]
    diff = [x for x in same if x[2] != x[3]]
    print(f'정답 확인 {graded_api.ok.sum()}/{len(graded_api)} 맞음 / 결과에 없는 노드 {len([n for n in API_CHECK_NODES if n not in RESULT_API])} '
          f'/ 언급 줄 {len(same)}개 중 Codex v3와 제품군·종류가 다른 줄 {len(diff)}개 / 비용 ${float(r.cost_usd):.4f}')
    display(graded_api[~graded_api.ok])
    for n, l, a, b in diff:
        print(f'  {n} {l}: Codex {a} / API {b}')
else:
    print('아직 결과가 없다.')

정답 확인 38/39 맞음 / 결과에 없는 노드 0 / 언급 줄 124개 중 Codex v3와 제품군·종류가 다른 줄 2개 / 비용 $0.0393


,node,check,ok,got
2,P:wafer,family,False,"['semiconductor_wafers', 'solar_manufacturing_and_system_equipment']"


  P:wafer m1: Codex ('semiconductor_wafers', '제품·원자재') / API ('solar_manufacturing_and_system_equipment', '제품·원자재')
  P:case m2: Codex ('industrial_equipment_and_tools', '제품·원자재') / API ('semiconductor_equipment_parts', '제품·원자재')


## 16. 1단계 전체 실행 (배치 API)

그래프에 쓰이는 노드 전부를 **업종별로 100개씩** 묶어 배치 하나로 보낸다. 같은 업종끼리 묶으면 지시문(제품군 목록 포함) 부분이 캐시돼 싸지고, LLM이 비슷한 노드를 함께 본다.

노트북을 실행할 때마다 아래 순서로 한다(뉴스 추출과 같다).

1. **결과 받기**: 기다리는 배치가 있으면 상태를 묻고, 끝났으면 결과를 받아 묶음마다 `raw/`에 저장하고 실행 기록에 붙인다.
2. **남은 묶음 계산**: 같은 지시문·입력으로 성공한 묶음과 결과를 기다리는 묶음은 뺀다.
3. **예상 비용**: 입력 토큰은 API 확인에서 잰 값(지시문 부분과 노드당 토큰)으로 어림한다.
4. **제출**: `RUN = True`이고 `BATCH_SUBMIT = True`일 때만 보낸다.

In [20]:
# ── 1단계 전체: 묶음 만들기와 예상 비용 ──
CHUNK = 100
BATCH_SUBMIT = False                                  # True로 바꿔야 배치를 제출한다
BATCH_DIR = LLM / 'batches'
BATCH_LOG = LLM / 'batches.csv'
order_nodes = nodes.assign(sec=nodes['sector'].replace('', '업종없음')).sort_values(['sec', 'id'])
NODE_CHUNKS = {}                                      # 묶음 이름 → 노드 ID 목록
for sec, g in order_nodes.groupby('sec', sort=True):
    ids = g['id'].tolist()
    for i in range(0, len(ids), CHUNK):
        NODE_CHUNKS[f'{sec}_{i // CHUNK + 1:02d}'] = ids[i:i + CHUNK]
NODE_INPUTS = {k: '\n\n'.join(node_block(n) for n in v) for k, v in NODE_CHUNKS.items()}

# 어림값: API 확인 1건(노드 20개)의 실제 토큰으로 지시문 부분과 글자당 토큰을 정한다
chk = read_runs().query("item == 'api_check' and status == 'ok'").iloc[-1]
chk_in, chk_out = float(chk.input_tokens), float(chk.output_tokens)
prompt_tok = chk_in * len(NODE_TEXT_API) / (len(NODE_TEXT_API) + len(api_text))       # 글자 비율로 나눈 지시문 몫
tok_per_char = (chk_in - prompt_tok) / len(api_text)
out_per_node = chk_out / len(API_CHECK_NODES)
est_in = {k: prompt_tok + tok_per_char * len(v) for k, v in NODE_INPUTS.items()}
est_out = {k: out_per_node * len(NODE_CHUNKS[k]) for k in NODE_INPUTS}
tin, tout = sum(est_in.values()), sum(est_out.values())
lo = api_cost(MODEL, tin, prompt_tok * len(NODE_INPUTS), tout, batch=True)
hi = api_cost(MODEL, tin, 0, tout, batch=True)
print(f'노드 {sum(map(len, NODE_CHUNKS.values())):,}개 → 묶음 {len(NODE_CHUNKS)}건(업종별 {CHUNK}개씩) / 입력 약 {tin / 1e6:.2f}백만 토큰'
      f'(지시문 약 {prompt_tok:,.0f} 토큰 × {len(NODE_CHUNKS)}건), 출력 약 {tout / 1e6:.2f}백만 토큰'
      f' → 배치 예상 ${lo:.2f}(지시문 캐시) ~ ${hi:.2f}(캐시 없음)')

노드 12,643개 → 묶음 141건(업종별 100개씩) / 입력 약 3.39백만 토큰(지시문 약 16,463 토큰 × 141건), 출력 약 1.82백만 토큰 → 배치 예상 $5.14(지시문 캐시) ~ $6.24(캐시 없음)


In [21]:
# ── 1단계 전체: 결과 받기 → 남은 묶음 → 제출 ──
def batch_log():
    """배치 기록(batches.csv)을 읽는다."""
    return pd.read_csv(BATCH_LOG, dtype=str).fillna('') if BATCH_LOG.exists() else pd.DataFrame(
        columns=['batch_id', 'created_utc', 'input_file', 'n', 'stage', 'prompt_hash', 'model', 'effort', 'status', 'collected'])


# 1) 결과 받기
log = batch_log()
new_rows = []
for i, b in log[log['collected'] != 'yes'].iterrows():
    info = openai_client().batches.retrieve(b['batch_id'])
    log.loc[i, 'status'] = info.status
    print(f"{b['batch_id']}: {info.status} / 완료 {info.request_counts.completed} / 실패 {info.request_counts.failed} / 전체 {info.request_counts.total}")
    if info.status not in ('completed', 'failed', 'expired', 'cancelled'):
        continue
    raw_dir = LLM / 'raw' / b['stage'] / f'batch-{b["model"]}'
    raw_dir.mkdir(parents=True, exist_ok=True)
    for file_id in [info.output_file_id, info.error_file_id]:
        if not file_id:
            continue
        for line in openai_client().files.content(file_id).text.splitlines():
            if not line.strip():
                continue
            item = json.loads(line)
            name, ihash = item['custom_id'].split('|')
            body = (item.get('response') or {}).get('body') or {}
            now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
            row = dict(stage=b['stage'], item=name, input_hash=ihash, prompt_hash=b['prompt_hash'], runner='batch', model=b['model'], effort=b['effort'],
                       run_utc=now.isoformat(), raw_file=f"{name}_{now.strftime('%Y%m%dT%H%M%S')}.json", status='', error='')
            (raw_dir / row['raw_file']).write_text(json.dumps(item, ensure_ascii=False, indent=1), encoding='utf-8')
            u = body.get('usage') or {}
            cached = (u.get('input_tokens_details') or {}).get('cached_tokens') or 0
            row.update(input_tokens=u.get('input_tokens'), cache_read_tokens=cached, output_tokens=u.get('output_tokens'),
                       reasoning_tokens=(u.get('output_tokens_details') or {}).get('reasoning_tokens'),
                       cost_usd=round(api_cost(b['model'], u.get('input_tokens') or 0, cached, u.get('output_tokens') or 0, batch=True), 5) if u else None)
            if item.get('error') or body.get('status') != 'completed':
                row.update(status='error', error=mask(json.dumps(item.get('error') or body.get('incomplete_details') or body.get('status')))[:300])
            else:
                # 1단계 배치는 스키마로 확인, 2단계 배치(product_tree_…)는 JSON만 확인하고 형식 점검은 22절에서 한다
                errs = [e.message for e in jsonschema.Draft202012Validator(node_schema).iter_errors(api_result(body))] if b['stage'].startswith('product_node') else (api_result(body) and [])
                row.update(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])
            new_rows.append(row)
    log.loc[i, 'collected'] = 'yes'
if new_rows:
    pd.DataFrame(new_rows).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
    got = pd.DataFrame(new_rows)
    print(f"  결과 {len(got)}건 저장: 성공 {(got['status'] == 'ok').sum()}, 실패 {(got['status'] != 'ok').sum()}, 비용 ${pd.to_numeric(got['cost_usd']).sum():.2f}")
if len(log):
    log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')

# 2) 남은 묶음: 같은 지시문·입력으로 성공했거나 결과를 기다리는 묶음은 뺀다
runs = read_runs()
done = set(runs.loc[(runs['stage'] == NODE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH_API),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
waiting = set()
for b in batch_log().query("collected != 'yes'").itertuples():
    waiting |= {json.loads(l)['custom_id'] for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines() if l.strip()}
todo = {k: v for k, v in NODE_INPUTS.items() if (k, text_hash(v)) not in done and f'{k}|{text_hash(v)}' not in waiting}
spent = pd.to_numeric(runs.loc[(runs['stage'] == NODE_PROMPT) & (runs['runner'] == 'batch'), 'cost_usd'], errors='coerce').sum()
print(f'묶음 {len(NODE_INPUTS)}건 / 끝남 {sum((k, text_hash(v)) in done for k, v in NODE_INPUTS.items())} / 기다림 {len(waiting)} / 남음 {len(todo)} / 지금까지 배치 비용 ${spent:.2f}')

# 3) 제출
if RUN and BATCH_SUBMIT and todo:
    BATCH_DIR.mkdir(exist_ok=True)
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    name = f"{NODE_PROMPT}_{MODEL}_{now.strftime('%Y%m%dT%H%M%S')}.jsonl"
    lines = [json.dumps({'custom_id': f'{k}|{text_hash(v)}', 'method': 'POST', 'url': '/v1/responses', 'body': api_body(v)}, ensure_ascii=False)
             for k, v in todo.items()]
    (BATCH_DIR / name).write_text('\n'.join(lines) + '\n', encoding='utf-8')
    try:
        client = openai_client()
        with open(BATCH_DIR / name, 'rb') as f:
            uploaded = client.files.create(file=f, purpose='batch')
        batch = client.batches.create(input_file_id=uploaded.id, endpoint='/v1/responses', completion_window='24h',
                                      metadata={'stage': NODE_PROMPT, 'model': MODEL})
    except Exception as e:                        # 크레딧 부족, 한도 초과 등
        (BATCH_DIR / name).unlink(missing_ok=True)
        print('배치 제출 실패:', mask(e)[:300])
    else:
        log = pd.concat([batch_log(), pd.DataFrame([dict(batch_id=batch.id, created_utc=now.isoformat(), input_file=name, n=str(len(todo)), stage=NODE_PROMPT,
                                                         prompt_hash=NODE_HASH_API, model=MODEL, effort=EFFORT, status=batch.status, collected='')])])
        log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
        print(f'배치 제출: {batch.id} / {len(todo)}건 / 요청 파일 {name}')
elif not (RUN and BATCH_SUBMIT):
    print('RUN·BATCH_SUBMIT 중 하나가 False: 제출하지 않았다.')

묶음 141건 / 끝남 141 / 기다림 0 / 남음 0 / 지금까지 배치 비용 $2.75
RUN·BATCH_SUBMIT 중 하나가 False: 제출하지 않았다.


## 17. 1단계 결과 점검

배치 결과를 묶음마다 읽어 노드 → 그룹 표로 만든다(`stage1_groups_지시문.csv`: 노드, 그룹 번호, 종류, 제품군, 대표 이름, 담은 줄번호, 담은 언급 수).

- 형식: 결과에 없는 노드, 없는 줄번호, 그룹에 안 들어간 줄(보이는 줄만. 보이지 않는 줄은 첫 그룹으로 간다).
- 분포: 나뉜 노드, 여러 품목을 나눈 노드(같은 줄이 여러 그룹에 있음), 종류별·제품군별 언급 수, 모음 제품군으로 간 비율.

In [22]:
# ── 1단계 결과 모으기와 형식 점검 ──
runs = read_runs()
ok = runs[(runs['stage'] == NODE_PROMPT) & (runs['runner'] == 'batch') & (runs['status'] == 'ok') & (runs['prompt_hash'] == NODE_HASH_API)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
STAGE1 = {}
for r in ok.itertuples():
    body = json.loads((LLM / 'raw' / NODE_PROMPT / f'batch-{MODEL}' / r.raw_file).read_text(encoding='utf-8'))['response']['body']
    for n in api_result(body)['nodes']:
        STAGE1[n['node_id']] = n['groups']
line_n = lines_all.set_index(['node_id', 'line'])['n']
print(f'{NODE_PROMPT} 배치 결과 노드 {len(STAGE1):,}개')
visible = lines_all.groupby('node_id').head(MAX_LINES)
rows = []
for nid, gs in STAGE1.items():
    for gi, g in enumerate(gs):
        ls = [l for l in g['lines'] if (nid, l) in line_n.index]
        hidden = int(lines_all.loc[lines_all['node_id'] == nid, 'n'].iloc[MAX_LINES:].sum()) if gi == 0 else 0
        rows.append(dict(node_id=nid, group=gi, kind=g['kind'], family=g['family'], name_en=g['name_en'], name_ko=g['name_ko'],
                         lines=' '.join(g['lines']), n_mentions=int(sum(line_n[(nid, l)] for l in ls)) + hidden, note=g['note']))
G1 = pd.DataFrame(rows, columns=['node_id', 'group', 'kind', 'family', 'name_en', 'name_ko', 'lines', 'n_mentions', 'note'])
if len(G1):                                       # 결과가 없으면(배치 전) 이전 버전 파일을 건드리지 않는다
    G1.to_csv(LLM / f'stage1_groups_{NODE_PROMPT}.csv', index=False, encoding='utf-8-sig')
expected = set(nodes['id'])
bad_lines = [(n, l) for n, gs in STAGE1.items() for g in gs for l in g['lines'] if (n, l) not in line_n.index]
placed = {(n, l) for n, gs in STAGE1.items() for g in gs for l in g['lines']}
unplaced = [(r.node_id, r.line) for r in visible.itertuples() if r.node_id in STAGE1 and (r.node_id, r.line) not in placed]
multi = {n for n, gs in STAGE1.items() if len(gs) > 1 and len({l for g in gs for l in g['lines']}) < sum(len(g['lines']) for g in gs)}
print(f'결과 노드 {len(STAGE1):,}개(기대 {len(expected):,}) / 빠진 노드 {len(expected - set(STAGE1))} / 없는 노드 {len(set(STAGE1) - expected)} '
      f'/ 없는 줄번호 {len(bad_lines)} / 그룹에 안 들어간 줄 {len(unplaced)}')
split = G1.groupby('node_id').size()
print(f'그룹 {len(G1):,}개 / 나뉜 노드 {(split > 1).sum():,}개({(split > 1).mean():.1%}), 그중 여러 품목을 나눈 노드 {len(multi):,}개')
tot = max(G1['n_mentions'].sum(), 1)                 # 결과가 없을 때(배치 전) 0으로 나누지 않게
print('종류별 언급', {k: f'{v / tot:.1%}' for k, v in G1.groupby('kind')['n_mentions'].sum().sort_values(ascending=False).items()})
catch = set(FAMILIES.loc[FAMILIES['catch_all'] == 'True', 'key'])
print(f"모음 제품군으로 간 언급 {G1.loc[G1['family'].isin(catch), 'n_mentions'].sum() / tot:.1%}",
      {k: int(v) for k, v in G1[G1['family'].isin(catch)].groupby('family')['n_mentions'].sum().items()})
used_f = G1.groupby('family')['n_mentions'].sum()
print(f'제품군 {len(FAMILIES)}개 중 쓰인 것 {len(used_f)}개 / 쓰이지 않은 것', sorted(set(FAMILIES['key']) - set(used_f.index)))

product_node_v4 배치 결과 노드 12,642개


결과 노드 12,642개(기대 12,643) / 빠진 노드 1 / 없는 노드 0 / 없는 줄번호 0 / 그룹에 안 들어간 줄 0
그룹 13,688개 / 나뉜 노드 764개(6.0%), 그중 여러 품목을 나눈 노드 600개
종류별 언급 {'제품·원자재': '80.5%', '서비스': '18.1%', '기타': '1.0%', '시설·공장': '0.4%', '가격·지표': '0.1%'}
모음 제품군으로 간 언급 4.2% {'industrial_chemicals_and_additives': 347, 'industrial_equipment_and_tools': 131, 'industrial_metals_and_components': 186, 'other': 265}
제품군 273개 중 쓰인 것 273개 / 쓰이지 않은 것 []


In [23]:
# ── 확인: 언급이 많은 제품군, 나뉜 노드 예, 무작위 노드 예 ──
if len(G1):
    display(used_f.sort_values(ascending=False).head(15).rename('언급 수').to_frame().T)
    print('나뉜 노드 예(무작위 12개)')
    for nid in pd.Series(sorted(split[split > 1].index)).sample(12, random_state=3):
        print(f'■ {nid}')
        for g in STAGE1[nid]:
            print(f"   {g['kind']:6} {g['family']:40} {g['name_en']} / {g['name_ko']}" + (f"  ({g['note'][:60]})" if g['note'] else ''))
    print('\n무작위 노드 예(25개)')
    for nid in pd.Series(sorted(STAGE1)).sample(25, random_state=8):
        names = ', '.join(lines_all.loc[lines_all['node_id'] == nid, 'name'].head(3))
        print(f"■ {nid} ← {names[:60]}")
        for g in STAGE1[nid]:
            print(f"   {g['kind']:6} {g['family']:40} {g['name_en']} / {g['name_ko']}")

family,steel_products,automotive_parts,plastic_resins,nonferrous_metals,industrial_chemicals_and_additives,food_ingredients,textiles_and_leather,packaging_materials,enterprise_software_and_it_services,basic_petrochemicals,other,oncology_drugs,building_and_infrastructure_construction,cosmetics_and_personal_care,ai_models_and_platforms
언급 수,642,529,448,443,347,341,334,321,310,291,265,258,256,243,220


나뉜 노드 예(무작위 12개)
■ P:steel plate and outsourced
   제품·원자재 steel_products                           steel plates / 철판
   서비스    industrial_and_electronics_manufacturing_services outsourced manufacturing services / 외주 가공 서비스  (외주비의 구체적 내용은 미상.)
■ P:plastering waterproofing and masonry work
   서비스    construction_engineering_and_specialty_services plastering work / 미장 공사  (나열된 공종을 분리했다.)
   서비스    construction_engineering_and_specialty_services waterproofing work / 방수 공사
   서비스    construction_engineering_and_specialty_services masonry work / 조적 공사
■ P:salvaged vehicle material and part
   제품·원자재 industrial_metals_and_components         salvaged vehicle materials / 폐차 해체자원  (해체자원의 구체적 재질은 명시되지 않았다.)
   제품·원자재 automotive_parts                         salvaged vehicle parts / 폐차 해체 부품
■ P:embedded SIM
   제품·원자재 interface_and_sensor_chips               embedded SIM chips / 내장형 가입자 식별 모듈  (eSIM 칩과 이를 이용한 통신 서비스를 분리했다.)
   서비스    telecommunications_services              eSIM data services / eS

## 18. 빠진 노드 다시 보내기

**왜 필요한가**: LLM은 긴 목록을 차례로 옮겨 적다가 드물게 노드 하나를 건너뛴다. 결과 형식(스키마)은 "노드 목록"의 모양만 강제할 수 있고 "입력 노드를 빠짐없이 넣었는가"까지는 막지 못한다. v3 전체 배치에서는 12,643개 중 1개(`P:bilt AI agent`, 소프트웨어_02 묶음)가 빠졌다. 응답은 정상 종료였고, 이름순으로 비슷한 노드(bill payment, billiards equipment)가 이어진 자리였다.

**어떻게 하는가**

1. 17절 결과(`STAGE1`)에 없는 노드를 찾는다(입력 노드 − 결과 노드). 이미 다시 보내 받은 결과(`retry_…`)는 먼저 합친다.
2. **다시 보낼 노드 수와 예상 비용**(일반 요청 / 배치)을 보여 주고 **사용자에게 묻는다**: `일반`(API 일반 요청, 바로 받음), `배치`(배치로 제출, 절반 가격, 16절을 다시 실행하면 받음), `아니오`. 답한 대로만 보낸다.
   - 노트북을 `nbconvert` 등으로 실행해 입력을 받을 수 없으면 `RETRY_CHOICE`에 미리 적은 값(`일반`/`배치`)을 쓰고, 비어 있으면 보내지 않는다.
   - 배치 결과를 아직 받지 않았으면(결과 0개) 모든 노드가 빠진 것으로 보이므로 경고를 함께 보여 준다.
3. 노드 100개씩 나눠 보낸다. 항목 이름은 `retry_묶음번호_노드해시`다. 일반 요청 결과는 바로 `STAGE1`에 합치고 결과 표(`stage1_groups_지시문.csv`)를 다시 쓴다. 배치로 보낸 것은 16절에서 받으면 17절이 다른 묶음과 함께 읽는다.

매일 밤 새 제품을 처리할 때도 같은 방식으로 빠진 노드를 막는다.

In [24]:
# ── 빠진 노드 다시 보내기: 노드 수·예상 비용을 보여 주고, 답(일반/배치/아니오)에 따라 보낸다 ──
RETRY_CHOICE = ''         # 입력을 받을 수 없을 때(nbconvert 실행) 쓸 답: '' 이면 보내지 않음, '일반', '배치'


def retry_results():
    """일반 요청으로 다시 보낸 결과(retry_…)를 노드 → 그룹으로 읽는다(배치로 보낸 것은 17절이 읽는다)."""
    out = {}
    r = read_runs()
    r = r[(r['stage'] == NODE_PROMPT) & (r['runner'] == 'api') & r['item'].str.startswith('retry_') & (r['status'] == 'ok')
          & (r['prompt_hash'] == NODE_HASH_API)].sort_values('run_utc')
    for x in r.itertuples():
        for n in api_result(json.loads((LLM / 'raw' / NODE_PROMPT / f'api-{MODEL}' / x.raw_file).read_text(encoding='utf-8')))['nodes']:
            out[n['node_id']] = n['groups']
    return out


def ask(question):
    """사용자에게 묻는다. 입력을 받을 수 없으면(nbconvert) None."""
    try:
        return input(question).strip()
    except Exception:
        return None


STAGE1.update({k: v for k, v in retry_results().items() if k not in STAGE1})          # 이미 다시 받은 것부터 합친다
waiting_retry = {cid.split('|')[0] for b in batch_log().query("collected != 'yes'").itertuples()
                 for cid in (json.loads(l)['custom_id'] for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines() if l.strip())}
missing = sorted(set(nodes['id']) - set(STAGE1))
retry_chunks = {f'retry_{i // CHUNK + 1:02d}_{text_hash(" ".join(missing[i:i + CHUNK]))[:8]}': missing[i:i + CHUNK] for i in range(0, len(missing), CHUNK)}
retry_chunks = {k: v for k, v in retry_chunks.items() if k not in waiting_retry}           # 배치로 보내 결과를 기다리는 묶음은 뺀다
n_retry = sum(map(len, retry_chunks.values()))
print(f'{NODE_PROMPT}: 결과 노드 {len(STAGE1):,}개 / 빠진 노드 {len(missing):,}개 / 다시 보낼 노드 {n_retry:,}개({len(retry_chunks)}건)', missing[:10])
if retry_chunks:
    texts = {k: '\n\n'.join(node_block(n) for n in v) for k, v in retry_chunks.items()}
    e_in = sum(prompt_tok + tok_per_char * len(t) for t in texts.values())
    e_out = out_per_node * n_retry
    print(f'  예상 비용: 일반 요청 약 ${api_cost(MODEL, e_in, 0, e_out):.2f} / 배치 약 ${api_cost(MODEL, e_in, 0, e_out, batch=True):.2f}'
          f'(입력 약 {e_in:,.0f} 토큰, 출력 약 {e_out:,.0f} 토큰)')
    if not STAGE1:
        print('  주의: 배치 결과를 아직 받지 않아 모든 노드가 빠진 것으로 보인다. 보통은 16절에서 배치 결과를 먼저 받는다.')
    choice = ask(f'빠진 노드 {n_retry:,}개를 다시 보낼까요? [일반/배치/아니오]: ')
    if choice is None:
        choice = RETRY_CHOICE
        print(f'  입력을 받을 수 없어 RETRY_CHOICE를 쓴다: {choice!r}')
    if choice == '일반':
        for k, t in texts.items():
            row = run_api(NODE_PROMPT, k, t)
            pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
            print(f"  {k}: {row['status']} {row.get('duration_s')}초 비용 ${row.get('cost_usd')} {row.get('error') or ''}")
        STAGE1.update({k: v for k, v in retry_results().items() if k not in STAGE1})
        print(f"  다시 보낸 뒤 빠진 노드 {len(set(nodes['id']) - set(STAGE1))}개")
    elif choice == '배치':
        BATCH_DIR.mkdir(exist_ok=True)
        now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
        name = f"{NODE_PROMPT}_retry_{MODEL}_{now.strftime('%Y%m%dT%H%M%S')}.jsonl"
        (BATCH_DIR / name).write_text('\n'.join(json.dumps({'custom_id': f'{k}|{text_hash(t)}', 'method': 'POST', 'url': '/v1/responses',
                                                            'body': api_body(t)}, ensure_ascii=False) for k, t in texts.items()) + '\n', encoding='utf-8')
        try:
            with open(BATCH_DIR / name, 'rb') as f:
                uploaded = openai_client().files.create(file=f, purpose='batch')
            batch = openai_client().batches.create(input_file_id=uploaded.id, endpoint='/v1/responses', completion_window='24h',
                                                   metadata={'stage': NODE_PROMPT, 'model': MODEL, 'retry': 'yes'})
        except Exception as e:
            (BATCH_DIR / name).unlink(missing_ok=True)
            print('  배치 제출 실패:', mask(e)[:300])
        else:
            pd.concat([batch_log(), pd.DataFrame([dict(batch_id=batch.id, created_utc=now.isoformat(), input_file=name, n=str(len(texts)), stage=NODE_PROMPT,
                                                       prompt_hash=NODE_HASH_API, model=MODEL, effort=EFFORT, status=batch.status, collected='')])]
                      ).to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
            print(f'  배치 제출: {batch.id} / {len(texts)}건. 16절을 다시 실행하면 결과를 받는다.')
    else:
        print('  보내지 않았다.')

# 합친 결과로 노드 → 그룹 표를 다시 쓴다(17절과 같은 방식)
rows = []
for nid, gs in STAGE1.items():
    for gi, g in enumerate(gs):
        ls = [l for l in g['lines'] if (nid, l) in line_n.index]
        hidden = int(lines_all.loc[lines_all['node_id'] == nid, 'n'].iloc[MAX_LINES:].sum()) if gi == 0 else 0
        rows.append(dict(node_id=nid, group=gi, kind=g['kind'], family=g['family'], name_en=g['name_en'], name_ko=g['name_ko'],
                         lines=' '.join(g['lines']), n_mentions=int(sum(line_n[(nid, l)] for l in ls)) + hidden, note=g['note']))
G1 = pd.DataFrame(rows, columns=['node_id', 'group', 'kind', 'family', 'name_en', 'name_ko', 'lines', 'n_mentions', 'note'])
if len(G1):                                           # 결과가 없으면(배치 전) 이전 버전 파일을 건드리지 않는다
    G1.to_csv(LLM / f'stage1_groups_{NODE_PROMPT}.csv', index=False, encoding='utf-8-sig')
print(f'{NODE_PROMPT} 결과 노드 {G1.node_id.nunique():,}개 / 그룹 {len(G1):,}개 / 이 지시문 비용 합계 '
      f"${pd.to_numeric(read_runs().query('stage == @NODE_PROMPT')['cost_usd'], errors='coerce').sum():.2f}(시범·확인 포함, Codex 제외)")

product_node_v4: 결과 노드 12,643개 / 빠진 노드 0개 / 다시 보낼 노드 0개(0건) []


product_node_v4 결과 노드 12,643개 / 그룹 13,690개 / 이 지시문 비용 합계 $2.79(시범·확인 포함, Codex 제외)


## 19. 2단계 입력: 제품군 안 정리

1단계 결과(**v4**, `stage1_groups_product_node_v4.csv`)의 그룹을 제품군마다 모아 보낸다. 같은 제품군을 한꺼번에 봐야 동의어를 합치고 계층을 만들 수 있다. 지시문: `prompts/product_tree_v4.md`(최종본. 버전별 차이는 20절 표).

- 받는 것: 제품(합친 그룹, 대표 이름, 상위 제품, 세대·규격 `specs`)과 후보 하위 제품(입력에 없지만 미국 뉴스가 쓰는 구체적인 하위 제품, 검색어로만 쓴다).
- 빼는 것: 기타(`other`), 시장·경제 지표(`market_indicators`) 제품군과 종류가 가격·지표인 그룹. 지표는 계층에 넣지 않고 그 제품군에 붙은 지표로 둔다.
- 입력 한 줄: `그룹번호 | 영어 이름 | 한글 이름 | 언급 수 | 원래 표기 예(최대 3개)`. 그룹번호는 제품군 안에서 언급이 많은 순서로 g1, g2, …
- 제품군 하나가 요청 하나다(가장 큰 자동차 부품도 그룹 411개라 한 번에 들어간다).

In [25]:
# ── 2단계 입력: 제품군마다 그룹 목록, 지시문, 스키마 ──
TREE_PROMPT = 'product_tree_v4'                                 # 버전별 차이는 20절 표
STAGE1_FILE = LLM / 'stage1_groups_product_node_v4.csv'          # 1단계 결과로 쓰기로 한 v4
SKIP_FAMILIES = {'other', 'market_indicators'}
S1 = pd.read_csv(STAGE1_FILE, dtype=str).fillna('')
S1['n'] = S1['n_mentions'].astype(int)
tree_groups = S1[~S1['family'].isin(SKIP_FAMILIES) & (S1['kind'] != '가격·지표')].sort_values(['family', 'n'], ascending=[True, False]).copy()
tree_groups['gid'] = 'g' + (tree_groups.groupby('family').cumcount() + 1).astype(str)
line_name = {(r.node_id, r.line): r.name for r in lines_all.itertuples()}
tree_groups['orig'] = [', '.join(list(dict.fromkeys(line_name.get((n, l), '') for l in ls.split()))[:3]) for n, ls in zip(tree_groups['node_id'], tree_groups['lines'])]
FAM = FAMILIES.set_index('key')


def family_input(f):
    """제품군 하나의 2단계 입력 글."""
    g = tree_groups[tree_groups['family'] == f]
    head = f"제품군: {f} | {FAM.at[f, 'name_ko']} | {FAM.at[f, 'scope']}\n그룹 {len(g)}개\n"
    return head + '\n' + '\n'.join(f'{r.gid} | {r.name_en} | {r.name_ko} | {r.n} | {r.orig}' for r in g.itertuples())


TREE_INPUTS = {f: family_input(f) for f in sorted(tree_groups['family'].unique())}
TREE_MD = (ROOT / 'prompts' / f'{TREE_PROMPT}.md').read_text(encoding='utf-8')
TREE_TEXT, TREE_TEXT_API = TREE_MD + CODEX_SUFFIX, TREE_MD
TREE_HASH, TREE_HASH_API = text_hash(TREE_TEXT + MODEL + EFFORT), text_hash(TREE_TEXT_API + MODEL + EFFORT)
tree_schema = {
    'type': 'object', 'additionalProperties': False, 'required': ['products', 'candidates', 'notes'],
    'properties': {
        'products': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                                'required': ['id', 'members', 'name_en', 'name_ko', 'parent', 'parent_name', 'specs'],
                                                'properties': {'id': {'type': 'string'}, 'members': {'type': 'array', 'items': {'type': 'string'}},
                                                               'name_en': {'type': 'string'}, 'name_ko': {'type': 'string'}, 'parent': {'type': 'string'},
                                                               'parent_name': {'type': 'string'}, 'specs': {'type': 'array', 'items': {'type': 'string'}}}}},
        'candidates': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                                  'required': ['parent', 'parent_name', 'name_en', 'name_ko', 'search_terms', 'reason'],
                                                  'properties': {'parent': {'type': 'string'}, 'parent_name': {'type': 'string'}, 'name_en': {'type': 'string'}, 'name_ko': {'type': 'string'},
                                                                 'search_terms': {'type': 'array', 'items': {'type': 'string'}}, 'reason': {'type': 'string'}}}},
        'notes': {'type': 'string'}}}
TREE_SCHEMA = LLM / 'schemas' / f'{TREE_PROMPT}.schema.json'
TREE_SCHEMA.write_text(json.dumps(tree_schema, ensure_ascii=False, indent=1), encoding='utf-8')
sz = pd.Series({f: len(t) for f, t in TREE_INPUTS.items()})
print(f'제품군 {len(TREE_INPUTS)}개 / 그룹 {len(tree_groups):,}개(뺀 것: 기타·시장 지표·가격 지표 {len(S1) - len(tree_groups)}개) / '
      f'입력 글자 합 {sz.sum():,}, 가장 긴 것 {sz.idxmax()} {sz.max():,}자 / 지시문 {len(TREE_MD):,}자')
print(TREE_INPUTS['semiconductor_lithography_materials'])

제품군 271개 / 그룹 13,546개(뺀 것: 기타·시장 지표·가격 지표 144개) / 입력 글자 합 811,422, 가장 긴 것 automotive_parts 25,153자 / 지시문 2,722자
제품군: semiconductor_lithography_materials | 반도체 노광용 마스크 소재 | 포토마스크·블랭크마스크·펠리클을 담고 포토레지스트는 제외한다.
그룹 7개

g1 | photomasks | 포토마스크 | 3 | Reticle, Photomask
g2 | photomask boxes | 포토마스크 보관함 | 1 | Mask Box
g3 | pellicle frame | 펠리클 프레임 | 1 | 펠리클 프레임
g4 | photomask pellicle | 포토마스크 펠리클 | 1 | 펠리클
g5 | blank masks | 블랭크마스크 | 1 | 블랭크마스크
g6 | EUV mask blank | EUV 블랭크마스크 | 1 | EUV용 블랭크마스크
g7 | EUV pellicle | EUV 펠리클 | 1 | EUV용 펠리클


## 20. 2단계 시범과 정답 확인 규칙

시범 제품군 6개: 메모리 반도체(세대 표기 HBM4·LPDDR5X, DRAM 아래의 HBM), 노광용 마스크 소재(작은 제품군), 반도체 패키지 기판(후보 하위 제품), 배터리 양극재(후보 하위 제품), 증권 중개·매매(1단계에서 거래와 청산이 나뉜 것), 비철금속(그룹 201개, 같은 이름 aluminum 두 개, 구리 아래의 구리 합금 코드).

정답 확인(`TREE_CHECKS`): `same`(두 그룹이 같은 제품), `spec`(그룹이 든 제품의 specs에 그 표기가 있음), `under`(앞 그룹의 제품이 뒤 그룹 제품의 아래에 있거나 같음), `sibling`(두 그룹의 제품이 같은 상위 제품 아래), `candidates`(후보 하위 제품이 하나 이상). 형식 점검: 모든 그룹이 정확히 한 제품에 들어갔는지, 없는 그룹번호·없는 상위 제품·순환이 없는지, 상위 제품의 번호와 이름(`parent_name`)이 맞는지.

| 지시문 | 시범 | 결과 | 고친 것 |
|---|---|---|---|
| v1 | 제품군 6개 | 정답 확인 10/10, 형식 문제 6(비철금속 구리 > 구리 합금 > 황동 > 황동 봉이 4단계) | - |
| v2 | 제품군 16개(v1의 6개 + 큰 제품군·서비스·계층이 뚜렷한 제품군 10개) | 정답 확인 27/28(틀린 1개는 당류 > 설탕으로 맞게 둔 것을 "같은 제품"으로 본 확인 규칙 실수 → under로 고침). 합치기는 나아짐(증권 47 → 36, 부동산 임대 52 → 32). 형식 문제 20개: 자동차 부품·비철금속에서 제품군과 같은 일반 그룹("automotive parts")을 맨 위 상위로 써서 깊이 5, 자동차 부품에서 없는 상위 제품 3개 |
| v3 | 같은 16개 | 정답 확인 28/28, 형식 문제 10(깊이 5인 실제로 깊은 계층 9, 두 제품에 든 그룹 1). 그런데 비철금속에서 알루미늄 제품이 주석 아래에 붙은 **상위 번호 오기**를 발견(형식 점검으로는 안 잡힘) | 제품군과 같은 뜻의 일반 그룹은 다른 제품의 상위로 쓰지 않음, parent는 결과에 있는 id만 |
| v4 | 같은 16개 | 정답 확인 27/28(틀린 1개는 제품군 바로 아래의 형제를 인정하지 않던 확인 규칙 실수 → 고침), 형식 문제 1(상위 번호·이름 불일치 1개를 새 확인이 잡음: aluminum profiles의 상위 번호가 chromium, 이름은 aluminum extrusions → 이름으로 바로잡을 수 있음) | 상위 제품의 이름(`parent_name`)도 적게 해 번호와 맞는지 코드로 확인, 깊이 제한 없앰(꼭 필요한 단계만) | 깊이 4단계까지 허용, 막연한 일반 표현끼리는 적극적으로 합침(증권 47개 그룹이 41개 제품으로 거의 안 합쳐짐) |

> **다시 실행할 때**: 시범 사례와 정답 확인은 2026-11 데이터와 LLM 결과에서 고른 것이다. 데이터를 새로 만들면 없는 사례는 건너뛰고 알린다. 새 데이터로 지시문을 다시 다듬는다면 그 데이터에서 어려운 사례를 다시 골라 넣기를 권한다.

In [26]:
# ── 2단계 시범 제품군과 정답 확인 규칙 ──
TREE_SAMPLE = ['memory_chips', 'semiconductor_lithography_materials', 'semiconductor_package_substrates', 'battery_cathode_materials',
               'securities_brokerage_and_trading', 'nonferrous_metals',
               # v2에서 더함: 큰 제품군(자동차 부품 411, 철강 253, 항암제 211, 기업용 SW 210), 서비스(부동산 임대), 계층이 뚜렷한 것(상선, 수지, 디스플레이 패널), 통신 장비, 식품 원료
               'automotive_parts', 'steel_products', 'oncology_drugs', 'enterprise_software_and_it_services', 'real_estate_leasing_and_management',
               'commercial_ships', 'plastic_resins', 'display_panels', 'telecommunications_equipment', 'food_ingredients']
TREE_CHECKS = [
    ('memory_chips', 'same', ('memory chips', 'memory')), ('memory_chips', 'same', ('high-bandwidth memory', 'HBM4')),
    ('memory_chips', 'spec', ('high-bandwidth memory', 'HBM4')), ('memory_chips', 'under', ('high-bandwidth memory', 'DRAM')),
    ('memory_chips', 'same', ('LPDDR5X', 'LPDDR4X')),
    ('semiconductor_lithography_materials', 'under', ('EUV pellicle', 'photomask pellicle')),
    ('semiconductor_package_substrates', 'candidates', None),
    ('battery_cathode_materials', 'candidates', None),
    ('nonferrous_metals', 'same', ('aluminum', 'aluminum')), ('nonferrous_metals', 'under', ('C15000 copper alloy', 'copper alloys')),
    ('securities_brokerage_and_trading', 'same', ('investment services', 'financial investment services')),
    ('commercial_ships', 'same', ('bulk carriers', 'bulk carrier')), ('commercial_ships', 'sibling', ('LNG carriers', 'LPG carriers')),
    ('commercial_ships', 'under', ('Suezmax crude oil tankers', 'oil tankers')),
    ('plastic_resins', 'same', ('polypropylene', 'polypropylene resin')), ('plastic_resins', 'under', ('homopolymer polypropylene', 'polypropylene')),
    ('plastic_resins', 'under', ('low-density polyethylene', 'polyethylene')),
    ('display_panels', 'same', ('LCD panels', 'liquid crystal displays')), ('display_panels', 'under', ('flexible AMOLED displays', 'OLED panels')),
    ('steel_products', 'same', ('steel coils', 'steel coil')), ('steel_products', 'same', ('deformed reinforcing bars', 'deformed reinforcing steel')),
    ('real_estate_leasing_and_management', 'same', ('real estate leasing', 'property leasing')),
    ('real_estate_leasing_and_management', 'same', ('office leasing', 'office leasing')),
    ('food_ingredients', 'under', ('sugar', 'sugars')), ('food_ingredients', 'under', ('wheat flour', 'flour')),
    ('automotive_parts', 'same', ('piston rings', 'piston rings')), ('automotive_parts', 'same', ('propeller shafts', 'automotive propeller shafts')),
    ('telecommunications_equipment', 'same', ('satellite ground stations', 'satellite ground stations')),
]
gone = [f for f in TREE_SAMPLE if f not in TREE_INPUTS]
if gone:
    print(f'※ 시범 제품군 {len(gone)}개는 이번 결과에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다):', gone)
TREE_SAMPLE = [f for f in TREE_SAMPLE if f in TREE_INPUTS]
TREE_CHECKS = [c for c in TREE_CHECKS if c[0] in TREE_INPUTS]
print(f'시범 제품군 {len(TREE_SAMPLE)}개, 그룹 {tree_groups.family.isin(TREE_SAMPLE).sum()}개 / 정답 확인 {len(TREE_CHECKS)}개')

시범 제품군 16개, 그룹 2142개 / 정답 확인 28개


## 21. 2단계 시범 실행과 채점 (Codex)

시범 제품군 6개를 하나씩 Codex로 보낸다(같은 지시문·입력이면 건너뛴다). 채점은 형식 점검과 정답 확인이다. 결과 전체(제품 계층과 후보)를 보기 좋게 출력해 사람이 본다.

In [27]:
# ── 2단계 시범 실행 ──
runs = read_runs()
done = set(runs.loc[(runs['stage'] == TREE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {f: TREE_INPUTS[f] for f in TREE_SAMPLE if (f, text_hash(TREE_INPUTS[f])) not in done}
print(f'이번에 돌릴 것 {len(todo)}건')
if RUN and todo:
    run_items(TREE_PROMPT, todo, TREE_TEXT, TREE_SCHEMA, TREE_HASH)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

이번에 돌릴 것 0건
RUN = False: 실행하지 않았다.


In [28]:
# ── 2단계 채점: 형식 점검, 정답 확인, 결과 보기 ──
def tree_check_format(f, out):
    """형식 점검: [(문제, 내용)]"""
    gids = set(tree_groups.loc[tree_groups['family'] == f, 'gid'])
    seen = Counter(m for pr in out['products'] for m in pr['members'])
    ids = {pr['id'] for pr in out['products']}
    parent = {pr['id']: pr['parent'] for pr in out['products']}
    probs = [('빠진 그룹', g) for g in sorted(gids - set(seen))] + [('두 번 든 그룹', g) for g, c in seen.items() if c > 1]
    probs += [('없는 그룹번호', g) for g in set(seen) - gids] + [('없는 상위 제품', i) for i, pa in parent.items() if pa and pa not in ids]
    probs += [('후보의 없는 상위 제품', c['name_en']) for c in out['candidates'] if c['parent'] not in ids]
    for i in ids:                                     # 순환과 깊이
        path, x = [], i
        while x and x not in path and len(path) < 10:
            path.append(x); x = parent.get(x, '')
        if x:
            probs.append(('순환', ' > '.join(path)))
    name_of = {pr['id']: pr['name_en'] for pr in out['products']}
    probs += [('상위 번호·이름 불일치', f"{pr['name_en']}: {pr['parent']}={name_of.get(pr['parent'])} / {pr.get('parent_name')}")
              for pr in out['products'] if pr['parent'] and pr.get('parent_name', name_of.get(pr['parent'])) != name_of.get(pr['parent'])]
    return probs


def tree_check(f, out, what, arg):
    """정답 확인 하나. (맞음, 설명)"""
    g = tree_groups[tree_groups['family'] == f]
    prod_of = {m: pr['id'] for pr in out['products'] for m in pr['members']}
    by_id = {pr['id']: pr for pr in out['products']}
    gid = lambda name: list(g.loc[g['name_en'] == name, 'gid'])
    if what == 'candidates':
        return len(out['candidates']) > 0, f"후보 {len(out['candidates'])}개"
    a, b = gid(arg[0]), gid(arg[1]) if what != 'spec' else []
    if what == 'same':
        ps = {prod_of.get(x) for x in (a + b if arg[0] != arg[1] else a)}
        return len(ps) == 1, f'제품 {ps}'
    if what == 'spec':
        pr = by_id.get(prod_of.get(a[0])) if a else None
        return bool(pr) and any(arg[1].lower() in s.lower() for s in pr['specs']), f"specs {pr['specs'] if pr else None}"
    if what == 'sibling':
        pa, pb = by_id.get(prod_of.get(a[0])) if a else None, by_id.get(prod_of.get(b[0])) if b else None
        return bool(pa and pb) and (pa['id'] == pb['id'] or pa['parent'] == pb['parent']), f"상위 {pa and pa['parent']} / {pb and pb['parent']}"
    if what == 'under':
        pa, pb = prod_of.get(a[0]) if a else None, prod_of.get(b[0]) if b else None
        x, chain = pa, []
        while x:
            chain.append(x); x = by_id[x]['parent'] if x in by_id else ''
        return pb in chain, f'{pa} 위로 {chain}, 기대 {pb}'


runs = read_runs()
ok = runs[(runs['stage'] == TREE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH) & runs['item'].isin(TREE_SAMPLE)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
TREE_RESULT = {r.item: parse_result(json.loads((LLM / 'raw' / TREE_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8'))) for r in ok.itertuples()}
if TREE_RESULT:
    fmt = [(f, *x) for f, out in TREE_RESULT.items() for x in tree_check_format(f, out)]
    graded_tree = pd.DataFrame([dict(family=f, check=w, arg=str(a), ok=c[0], got=c[1]) for f, w, a in TREE_CHECKS if f in TREE_RESULT
                                for c in [tree_check(f, TREE_RESULT[f], w, a)]])
    print(f'형식 문제 {len(fmt)}개 / 정답 확인 {graded_tree.ok.sum()}/{len(graded_tree)}')
    for x in fmt[:20]:
        print('  ', x)
    display(graded_tree[~graded_tree.ok])
    for f, out in TREE_RESULT.items():
        g = tree_groups[tree_groups['family'] == f].set_index('gid')
        kids = {}
        for pr in out['products']:
            kids.setdefault(pr['parent'], []).append(pr)
        cands = {}
        for c in out['candidates']:
            cands.setdefault(c['parent'], []).append(c['name_en'])
        print(f"\n■ {f}: 그룹 {len(g)} → 제품 {len(out['products'])}, 후보 {len(out['candidates'])}" + (f"  ({out['notes'][:120]})" if out['notes'] else ''))

        def show(pid, depth):
            for pr in kids.get(pid, []):
                names = ', '.join(g.at[m, 'name_en'] for m in pr['members'] if m in g.index)
                extra = (f" [specs: {', '.join(pr['specs'])}]" if pr['specs'] else '') + (f" [후보: {', '.join(cands[pr['id']])}]" if pr['id'] in cands else '')
                print('   ' * (depth + 1) + f"{pr['name_en']} / {pr['name_ko']}" + (f'  ← {names[:110]}' if names else '  (중간 제품)') + extra)
                show(pr['id'], depth + 1)
        show('', 0)
else:
    print('아직 결과가 없다.')

형식 문제 1개 / 정답 확인 28/28
   ('nonferrous_metals', '상위 번호·이름 불일치', 'aluminum profiles: p95=chromium / aluminum extrusions')


,family,check,arg,ok,got



■ semiconductor_lithography_materials: 그룹 7 → 제품 5, 후보 3  (EUV용 블랭크마스크와 펠리클은 각각 노광 방식에 따른 규격으로 보아 일반 제품에 합쳤다. 보관함과 프레임은 포토마스크나 펠리클의 하위 종류가 아니므로 별도 제품으로 두었다.)
   photomask / 포토마스크  ← photomasks [후보: EUV photomask, phase-shift mask, binary mask]
   photomask box / 포토마스크 보관함  ← photomask boxes
   pellicle frame / 펠리클 프레임  ← pellicle frame
   photomask pellicle / 포토마스크 펠리클  ← photomask pellicle, EUV pellicle [specs: EUV]
   mask blank / 블랭크마스크  ← blank masks, EUV mask blank [specs: EUV]

■ memory_chips: 그룹 39 → 제품 29, 후보 5  (eMMC는 NAND 기반 저장장치이지만 검색 경로를 위해 NAND 플래시 아래에 두었다. HBM 베이스 다이는 HBM의 구성 부품이므로 별도 최상위 제품으로 두었다. eMCP와 NAND MCP는 구성에 따라 겹칠 수)
   memory chips / 메모리 반도체  ← memory chips, memory chips, memory chips, memory
   random-access memory / 랜덤 액세스 메모리  ← random access memory
      DRAM / 디램  ← DRAM, DDR5 memory [specs: DDR5] [후보: double data rate synchronous DRAM]
         high-bandwidth memory / 고대역폭 메모리  ← high-bandwidth memory, HBM4 [specs: HBM4]
         low-power DRAM / 저전력 DRA

## 22. 2단계 API 확인과 전체 실행

1. **API 확인**: 시범 제품군 2개(메모리 반도체, 비철금속)를 API 일반 요청으로 보내 Codex와 같은 품질인지 본다(정답 확인과 형식 점검).
2. **전체 실행(배치)**: 제품군 271개를 하나씩 요청으로 만들어 배치 하나로 보낸다. 결과는 16절이 받는다(노트북을 다시 실행하면 된다).
3. 둘 다 `RUN = True`일 때만 보내고, 배치는 `TREE_BATCH_SUBMIT = True`일 때만 제출한다. 실행 전에 예상 비용을 보여 준다.

결과를 적용할 때 코드가 하는 일(23절에서 만든다): 상위 제품 번호와 이름(`parent_name`)이 다르면 이름으로 바로잡고, 두 제품에 든 그룹은 하나만 남기고, 없는 상위 제품은 제품군 바로 아래로 둔다.

In [29]:
# ── 2단계 API 확인: 시범 제품군 2개를 일반 요청으로 ──
TREE_BATCH_SUBMIT = False                                # True로 바꿔야 2단계 배치를 제출한다
TREE_API_CHECK = [f for f in ['memory_chips', 'nonferrous_metals'] if f in TREE_INPUTS]   # 이번 결과에 없는 제품군은 뺀다


def tree_body(text):
    """2단계 요청 본문(Responses API, 엄격 스키마)."""
    return dict(model=MODEL, instructions=TREE_TEXT_API, input=text, reasoning={'effort': EFFORT}, store=False,
                text={'format': {'type': 'json_schema', 'name': 'product_tree', 'schema': tree_schema, 'strict': True}})


runs = read_runs()
done = set(runs.loc[(runs['stage'] == TREE_PROMPT) & (runs['runner'] == 'api') & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH_API),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = [f for f in TREE_API_CHECK if (f, text_hash(TREE_INPUTS[f])) not in done]
print(f'API 확인 {len(TREE_API_CHECK)}건 중 남은 것 {len(todo)}건')
if RUN and todo:
    for f in todo:
        row = run_api(TREE_PROMPT, f, TREE_INPUTS[f], tree_body, tree_schema, TREE_HASH_API)
        pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
        print(f"  {f}: {row['status']} {row.get('duration_s')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')} 비용 ${row.get('cost_usd')} {row.get('error') or ''}")
runs = read_runs()
r = runs[(runs['stage'] == TREE_PROMPT) & (runs['runner'] == 'api') & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH_API)].sort_values('run_utc').drop_duplicates('item', keep='last')
TREE_API_RESULT = {x.item: api_result(json.loads((LLM / 'raw' / TREE_PROMPT / f'api-{MODEL}' / x.raw_file).read_text(encoding='utf-8'))) for x in r.itertuples()}
if TREE_API_RESULT:
    fmt = [(f, *x) for f, out in TREE_API_RESULT.items() for x in tree_check_format(f, out)]
    g_api = pd.DataFrame([dict(family=f, check=w, arg=str(a), ok=c[0], got=c[1]) for f, w, a in TREE_CHECKS if f in TREE_API_RESULT
                          for c in [tree_check(f, TREE_API_RESULT[f], w, a)]])
    print(f'API 확인: 형식 문제 {len(fmt)}개 / 정답 확인 {g_api.ok.sum()}/{len(g_api)} / 비용 ${pd.to_numeric(r.cost_usd).sum():.3f}')
    for f, out in TREE_API_RESULT.items():
        print(f"  {f}: 그룹 {(tree_groups.family == f).sum()} → 제품 {len(out['products'])}, 후보 {len(out['candidates'])} "
              f"(Codex: 제품 {len(TREE_RESULT[f]['products']) if f in TREE_RESULT else '-'}, 후보 {len(TREE_RESULT[f]['candidates']) if f in TREE_RESULT else '-'})")
    for x in fmt:
        print('   ', x)

# 전체 배치 예상: API 확인의 실제 토큰으로 지시문 몫과 그룹당 출력을 어림한다
if len(r):
    chk_in, chk_out = pd.to_numeric(r.input_tokens).sum(), pd.to_numeric(r.output_tokens).sum()
    chk_chars = sum(len(TREE_INPUTS[f]) for f in r.item)
    t_prompt = chk_in * len(TREE_TEXT_API) / (len(TREE_TEXT_API) * len(r) + chk_chars)
    t_per_char = (chk_in - t_prompt * len(r)) / chk_chars
    t_out_per_group = chk_out / tree_groups.family.isin(r.item).sum()
    e_in = sum(t_prompt + t_per_char * len(t) for t in TREE_INPUTS.values())
    e_out = t_out_per_group * len(tree_groups)
    print(f'전체 배치 예상: 제품군 {len(TREE_INPUTS)}건 / 입력 약 {e_in / 1e6:.2f}백만 토큰, 출력 약 {e_out / 1e6:.2f}백만 토큰 → 약 ${api_cost(MODEL, e_in, 0, e_out, batch=True):.2f}')

API 확인 2건 중 남은 것 0건
API 확인: 형식 문제 0개 / 정답 확인 7/7 / 비용 $0.064
  memory_chips: 그룹 39 → 제품 29, 후보 8 (Codex: 제품 29, 후보 5)
  nonferrous_metals: 그룹 201 → 제품 140, 후보 12 (Codex: 제품 135, 후보 8)
전체 배치 예상: 제품군 271건 / 입력 약 0.78백만 토큰, 출력 약 0.63백만 토큰 → 약 $1.97


In [30]:
# ── 2단계 전체 배치 제출(결과는 16절이 받는다) ──
runs = read_runs()
done = set(runs.loc[(runs['stage'] == TREE_PROMPT) & (runs['runner'] == 'batch') & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH_API),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
waiting = {json.loads(l)['custom_id'] for b in batch_log().query("collected != 'yes'").itertuples()
           for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines() if l.strip()}
todo = {f: t for f, t in TREE_INPUTS.items() if (f, text_hash(t)) not in done and f'{f}|{text_hash(t)}' not in waiting}
print(f'2단계 제품군 {len(TREE_INPUTS)}건 / 끝남 {sum((f, text_hash(t)) in done for f, t in TREE_INPUTS.items())} / 기다림 {len(waiting)} / 남음 {len(todo)}')
if RUN and TREE_BATCH_SUBMIT and todo:
    BATCH_DIR.mkdir(exist_ok=True)
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    name = f"{TREE_PROMPT}_{MODEL}_{now.strftime('%Y%m%dT%H%M%S')}.jsonl"
    (BATCH_DIR / name).write_text('\n'.join(json.dumps({'custom_id': f'{f}|{text_hash(t)}', 'method': 'POST', 'url': '/v1/responses', 'body': tree_body(t)},
                                                       ensure_ascii=False) for f, t in todo.items()) + '\n', encoding='utf-8')
    try:
        with open(BATCH_DIR / name, 'rb') as fh:
            uploaded = openai_client().files.create(file=fh, purpose='batch')
        batch = openai_client().batches.create(input_file_id=uploaded.id, endpoint='/v1/responses', completion_window='24h',
                                               metadata={'stage': TREE_PROMPT, 'model': MODEL})
    except Exception as e:
        (BATCH_DIR / name).unlink(missing_ok=True)
        print('배치 제출 실패:', mask(e)[:300])
    else:
        pd.concat([batch_log(), pd.DataFrame([dict(batch_id=batch.id, created_utc=now.isoformat(), input_file=name, n=str(len(todo)), stage=TREE_PROMPT,
                                                   prompt_hash=TREE_HASH_API, model=MODEL, effort=EFFORT, status=batch.status, collected='')])]
                  ).to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
        print(f'배치 제출: {batch.id} / {len(todo)}건 / 요청 파일 {name}')
else:
    print('RUN·TREE_BATCH_SUBMIT 중 하나가 False이거나 남은 것이 없어 제출하지 않았다.')

2단계 제품군 271건 / 끝남 271 / 기다림 0 / 남음 0
RUN·TREE_BATCH_SUBMIT 중 하나가 False이거나 남은 것이 없어 제출하지 않았다.


## 23. 내보내기: 원래 언급 → 제품

1·2단계 결과를 `product_apply.ipynb`가 쓸 표로 내보낸다. LLM은 다시 부르지 않는다.

1. **2단계 결과 바로잡기**(제품군마다)
   - 상위 제품 번호와 이름(`parent_name`)이 다르면 같은 제품군에서 그 이름의 제품을 찾아 상위로 둔다. 못 찾으면 번호를 따른다.
   - 없는 상위 번호, 순환은 제품군 바로 아래로 둔다. 두 제품에 든 그룹은 처음 것만 남긴다.
   - 빠진 그룹은 그 그룹 이름으로 제품군 바로 아래 제품을 만든다.
   - 후보 하위 제품도 상위를 같은 방식으로 찾고, 상위를 못 찾거나 이미 있는 제품 이름이면 뺀다.
2. **2단계에 보내지 않은 그룹**(기타·시장 지표 제품군, 가격·지표)은 그룹마다 제품이 된다. 같은 제품군에서 영어 이름이 같으면 하나로 합친다.
3. **제품 ID**: `P:` + 소문자 영어 이름. 여러 제품군에 같은 이름이 있으면 `P:제품군/이름`. 후보 하위 제품은 `PC:`로 시작한다. 그래프를 다시 만들 때마다 데이터로 다시 정해진다(`graphDB.md` 6.4).
4. **언급 → 제품**: 언급마다 1단계 줄번호 → 그룹 → 제품을 따라간다. 보이지 않은 줄(노드당 15줄 넘는 것)은 첫 그룹으로 간다. 여러 품목을 나눈 언급은 제품이 여러 개다.

**출력** (`source/products/llm/`, 내용이 바뀔 때만 쓴다)

| 파일 | 내용 |
|---|---|
| `llm_mention_map.csv` | 언급(책, 출처, 문서, 자리, 기사 개체 번호, 원래 이름, 영어 이름) → 제품 ID(여러 개면 ` \| `로 잇는다) |
| `llm_products.csv` | 제품: ID, 제품군, 영어·한글 이름, 종류, 상위 제품 ID, 규격, 중간 제품인지, 담은 규칙 노드, 언급 수 |
| `llm_candidates.csv` | 후보 하위 제품: ID, 제품군, 상위 제품 ID, 이름, 검색어, 이유 |

In [31]:
# ── 2단계 결과 바로잡기: 상위 번호·이름 불일치, 없는 상위, 순환, 두 번 든 그룹, 빠진 그룹, 후보의 상위 ──
runs = read_runs()
ok = runs[(runs['stage'] == TREE_PROMPT) & (runs['runner'] == 'batch') & (runs['status'] == 'ok') & (runs['prompt_hash'] == TREE_HASH_API)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
TREE, stale = {}, []
for r in ok.itertuples():
    if r.item not in TREE_INPUTS:
        continue
    if r.input_hash != text_hash(TREE_INPUTS[r.item]):      # 1단계 결과가 바뀌어 입력이 달라진 제품군은 쓰지 않는다(2절의 평평한 제품이 된다)
        stale.append(r.item)
        continue
    TREE[r.item] = api_result(json.loads((LLM / 'raw' / TREE_PROMPT / f'batch-{MODEL}' / r.raw_file).read_text(encoding='utf-8'))['response']['body'])
print(f'2단계 결과 제품군 {len(TREE)}/{len(TREE_INPUTS)} / 입력이 바뀌어 쓰지 않은 것 {len(stale)}', stale[:5])

low = lambda s: re.sub(r'\s+', ' ', (s or '').strip().lower())
FIX = Counter()


def fixed_tree(f, out):
    """2단계 결과 하나를 바로잡아 (제품 목록, 후보 목록)을 돌려준다. 제품: id, name_en, name_ko, parent, specs, members."""
    gids = list(tree_groups.loc[tree_groups['family'] == f, 'gid'])
    gname = tree_groups[tree_groups['family'] == f].set_index('gid')
    prods, seen_id = [], set()
    for pr in out['products']:
        pr = dict(pr)
        if pr['id'] in seen_id:                                # 같은 번호가 두 번 나오면 뒤의 것에 새 번호
            FIX['같은 제품 번호'] += 1
            pr['id'] = f"{pr['id']}_{len(prods)}"
        seen_id.add(pr['id'])
        prods.append(pr)
    ids = {pr['id'] for pr in prods}
    name_of = {pr['id']: low(pr['name_en']) for pr in prods}
    by_name = {}
    for pr in prods:
        by_name.setdefault(low(pr['name_en']), pr['id'])

    def resolve(pid, pname):
        """상위 번호와 이름이 다르면 이름으로 찾는다. 없는 번호는 맨 위('')."""
        if pid and pname and name_of.get(pid) != low(pname) and low(pname) in by_name:
            FIX['상위를 이름으로 바로잡음'] += 1
            return by_name[low(pname)]
        if pid and pid not in ids:
            FIX['없는 상위 → 맨 위'] += 1
            return ''
        return pid

    for pr in prods:
        pr['parent'] = resolve(pr['parent'], pr['parent_name'])
        if pr['parent'] == pr['id']:
            FIX['자기 자신이 상위 → 맨 위'] += 1
            pr['parent'] = ''
    parent = {pr['id']: pr['parent'] for pr in prods}
    for pr in prods:                                           # 순환은 되돌아온 곳에서 끊는다
        path, x = [], pr['id']
        while x and x not in path:
            path.append(x)
            x = parent[x]
        if x:
            FIX['순환 → 맨 위'] += 1
            parent[path[-1]] = ''
    used_g = set()
    for pr in prods:
        pr['parent'] = parent[pr['id']]
        ms = []
        for m in pr['members']:
            if m not in gname.index:
                FIX['없는 그룹번호'] += 1
            elif m in used_g:
                FIX['두 번 든 그룹 → 처음 것만'] += 1
            else:
                used_g.add(m)
                ms.append(m)
        pr['members'] = ms
    for g in gids:                                             # 빠진 그룹은 맨 위 제품으로
        if g not in used_g:
            FIX['빠진 그룹 → 맨 위 제품'] += 1
            prods.append(dict(id=f'missing_{g}', name_en=gname.at[g, 'name_en'], name_ko=gname.at[g, 'name_ko'], parent='', specs=[], members=[g]))
    names = {low(pr['name_en']) for pr in prods}
    cands = []
    for c in out['candidates']:
        p = resolve(c['parent'], c['parent_name'])
        if not p:
            FIX['후보: 상위 없음 → 뺌'] += 1
        elif low(c['name_en']) in names:
            FIX['후보: 이미 있는 제품 → 뺌'] += 1
        else:
            cands.append(dict(c, parent=p))
            names.add(low(c['name_en']))
    return prods, cands


FIXED = {f: fixed_tree(f, out) for f, out in TREE.items()}
print('바로잡은 것', dict(FIX))
print(f"제품 {sum(len(p) for p, _ in FIXED.values()):,}개 / 후보 {sum(len(c) for _, c in FIXED.values()):,}개")

2단계 결과 제품군 271/271 / 입력이 바뀌어 쓰지 않은 것 0 []


바로잡은 것 {'빠진 그룹 → 맨 위 제품': 1}
제품 11,555개 / 후보 2,659개


In [32]:
# ── 제품 표: 2단계 제품 + 2단계에 보내지 않은 그룹. 그룹 → 제품, 전체에서 쓰는 제품 ID ──
S1['line_list'] = S1['lines'].str.split()
gid_of = tree_groups.set_index(['family', 'gid'])[['node_id', 'group']].apply(tuple, axis=1).to_dict()   # (제품군, gid) → (노드, 그룹)
prod_rows, group_prod = [], {}                         # group_prod: (노드, 그룹) → 제품 키 '제품군|지역 번호'
for f, (prods, _) in FIXED.items():
    for pr in prods:
        key = f"{f}|{pr['id']}"
        for m in pr['members']:
            group_prod[gid_of[(f, m)]] = key
        prod_rows.append(dict(key=key, family=f, name_en=pr['name_en'].strip(), name_ko=pr['name_ko'].strip(), parent_key=f"{f}|{pr['parent']}" if pr['parent'] else '',
                              specs=' | '.join(dict.fromkeys(s.strip() for s in pr['specs'] if s.strip())), intermediate=not pr['members']))
flat = S1[[(n, g) not in group_prod for n, g in zip(S1['node_id'], S1['group'])]]   # 기타·시장 지표·가격·지표, 2단계 결과가 없는 제품군
for r in flat.itertuples():
    key = f'{r.family}|flat:' + low(r.name_en or r.name_ko)       # 같은 제품군·같은 영어 이름은 한 제품(뒤의 drop_duplicates)
    prod_rows.append(dict(key=key, family=r.family, name_en=r.name_en.strip(), name_ko=r.name_ko.strip(), parent_key='', specs='', intermediate=False))
    group_prod[(r.node_id, r.group)] = key
PR = pd.DataFrame(prod_rows).drop_duplicates('key')
print(f'제품 {len(PR):,}개(2단계 {sum(len(p) for p, _ in FIXED.values()):,} + 평평한 것 {len(PR) - sum(len(p) for p, _ in FIXED.values()):,}) / 그룹 {len(S1):,}개 중 제품이 정해진 것 {len(group_prod):,}')

# 그룹에서 제품으로 올리는 값: 종류(언급 수가 가장 많은 것), 담은 규칙 노드, 언급 수
S1['pkey'] = [group_prod[(n, g)] for n, g in zip(S1['node_id'], S1['group'])]
agg = S1.groupby('pkey').agg(n_mentions=('n', 'sum'), rule_nodes=('node_id', lambda s: ' | '.join(dict.fromkeys(s))))
agg['kind'] = (S1.groupby(['pkey', 'kind'])['n'].sum().reset_index().sort_values('n', ascending=False, kind='stable')
               .drop_duplicates('pkey').set_index('pkey')['kind'])
PR = PR.join(agg, on='key')
PR['n_mentions'] = PR['n_mentions'].fillna(0).astype(int)
PR[['rule_nodes', 'kind']] = PR[['rule_nodes', 'kind']].fillna('')


def make_ids(df, prefix):
    """ID: prefix + 소문자 이름. 여러 제품군에 같은 이름이면 prefix + 제품군/이름, 그래도 겹치면 뒤에 번호."""
    slug = [low(a) or low(b) for a, b in zip(df['name_en'], df['name_ko'])]
    fams = pd.Series(df['family'].values).groupby(slug).nunique()
    ids = [prefix + (s if fams[s] == 1 else f'{f}/{s}') for s, f in zip(slug, df['family'])]
    cnt, out = Counter(), []
    for i in ids:
        cnt[i] += 1
        out.append(i if cnt[i] == 1 else f'{i}#{cnt[i]}')
    return out


PR['id'] = make_ids(PR, 'P:')
id_of = dict(zip(PR['key'], PR['id']))
PR['parent'] = [id_of.get(k, '') if k else '' for k in PR['parent_key']]
CAND = pd.DataFrame([dict(family=f, parent=id_of[f"{f}|{c['parent']}"], name_en=c['name_en'].strip(), name_ko=c['name_ko'].strip(),
                          search_terms=' | '.join(dict.fromkeys(t.strip() for t in c['search_terms'] if t.strip())), reason=c['reason'])
                     for f, (_, cs) in FIXED.items() for c in cs])
CAND.insert(0, 'id', make_ids(CAND, 'PC:'))
depth = {}
parent_of = dict(zip(PR['id'], PR['parent']))
for i in PR['id']:
    d, x = 0, i
    while parent_of.get(x):
        d, x = d + 1, parent_of[x]
    depth[i] = d
PR['depth'] = PR['id'].map(depth)
print(f"ID가 제품군을 붙인 것 {PR['id'].str.contains('/').sum():,}개, 번호를 붙인 것 {PR['id'].str.contains('#').sum()}개 / "
      f"중간 제품 {PR['intermediate'].sum()}개 / 깊이별 {PR['depth'].value_counts().sort_index().to_dict()} / 후보 {len(CAND):,}개")

제품 11,693개(2단계 11,555 + 평평한 것 138) / 그룹 13,690개 중 제품이 정해진 것 13,690
ID가 제품군을 붙인 것 98개, 번호를 붙인 것 3개 / 중간 제품 160개 / 깊이별 {0: 5656, 1: 4162, 2: 1526, 3: 317, 4: 31, 5: 1} / 후보 2,659개


In [33]:
# ── 언급 → 제품: 언급마다 1단계 줄번호 → 그룹 → 제품. 보이지 않은 줄과 그룹에 안 든 줄은 첫 그룹으로 ──
MKEY = ['book', 'source', 'doc', 'field', 'ref', 'name', 'name_en']
line_groups = {}
for r in S1.itertuples():
    for l in r.line_list:
        line_groups.setdefault((r.node_id, l), []).append(r.group)
first_group = S1.groupby('node_id')['group'].first().to_dict()
m = mentions.merge(lines_all[['node_id'] + LINE_KEYS + ['line']], on=['node_id'] + LINE_KEYS, how='left')
how_mapped = Counter()


def mention_products(nid, line):
    gs = line_groups.get((nid, line))
    if gs:
        how_mapped['줄번호'] += 1
    elif nid in first_group:
        how_mapped['보이지 않은 줄·그룹에 안 든 줄 → 첫 그룹' if int(line[1:]) > MAX_LINES else '그룹에 안 든 보이는 줄 → 첫 그룹'] += 1
        gs = [first_group[nid]]
    else:
        how_mapped['1단계 결과 없는 노드'] += 1
        return []
    return list(dict.fromkeys(id_of[group_prod[(nid, g)]] for g in gs))


m['product_ids'] = [mention_products(n, l) for n, l in zip(m['node_id'], m['line'])]
print('언급 → 제품을 정한 방법', dict(how_mapped))
MAP = (m[m['product_ids'].str.len() > 0].groupby(MKEY, sort=False)['product_ids']
       .agg(lambda s: ' | '.join(dict.fromkeys(p for ps in s for p in ps))).reset_index())
n_multi = (MAP['product_ids'].str.count(r' \| ') > 0).sum()
print(f'언급 {len(mentions):,}개 → 열쇠 {len(MAP):,}개(같은 열쇠가 여러 줄인 것은 합침) / 제품이 둘 이상인 언급 {n_multi:,}개')

used_ids = {p for ps in MAP['product_ids'] for p in ps.split(' | ')}
print(f"언급이 없는 제품: 중간 제품 {(PR['intermediate']).sum()}개, 그 밖 {(~PR['id'].isin(used_ids) & ~PR['intermediate']).sum()}개")


def save_csv(df, path):
    """내용이 바뀔 때만 쓴다(iCloud 사본 방지)."""
    text = df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8-sig') == text:
        return '그대로'
    path.write_text(text, encoding='utf-8-sig')
    return '씀'


OUT_PRODUCTS = PR[['id', 'family', 'name_en', 'name_ko', 'kind', 'parent', 'depth', 'specs', 'intermediate', 'rule_nodes', 'n_mentions']]
print('llm_mention_map.csv', save_csv(MAP, LLM / 'llm_mention_map.csv'),
      '/ llm_products.csv', save_csv(OUT_PRODUCTS, LLM / 'llm_products.csv'),
      '/ llm_candidates.csv', save_csv(CAND, LLM / 'llm_candidates.csv'))

언급 → 제품을 정한 방법

 {'줄번호': 20530, '보이지 않은 줄·그룹에 안 든 줄 → 첫 그룹': 626}
언급 21,156개 → 열쇠 20,929개(같은 열쇠가 여러 줄인 것은 합침) / 제품이 둘 이상인 언급 684개
언급이 없는 제품: 중간 제품 160개, 그 밖 0개


llm_mention_map.csv 그대로 / llm_products.csv 그대로 / llm_candidates.csv 그대로


In [34]:
# ── 확인: 섞인 노드(웨이퍼), 여러 품목 언급, 계층 예(메모리), 후보 예 ──
show = lambda ids: PR.set_index('id').loc[[i for i in ids if i in set(PR['id'])], ['family', 'name_ko', 'parent', 'specs']]
print('■ 규칙 노드 P:wafer의 언급이 간 제품')
display(m[m['node_id'] == 'P:wafer'].assign(p=lambda d: d['product_ids'].str.join(' | ')).groupby('p').size().rename('언급 수').to_frame())
print('■ 여러 품목을 나눈 언급 예')
display(MAP[MAP['product_ids'].str.contains(r' \| ')].head(5)[['source', 'doc', 'name', 'product_ids']])
print('■ 메모리 반도체 제품군 계층(맨 위 → 아래)')
fam = PR[PR['family'] == 'memory_chips'].sort_values(['depth', 'n_mentions'], ascending=[True, False])
display(fam[['id', 'name_ko', 'parent', 'depth', 'specs', 'n_mentions']].head(15))
print('■ 후보 하위 제품 예')
display(CAND[CAND['family'].isin(['memory_chips', 'semiconductor_packaging_materials'])][['id', 'parent', 'name_ko', 'search_terms']].head(10))

■ 규칙 노드 P:wafer의 언급이 간 제품


,언급 수
p,
P:semiconductor wafers,12
P:solar wafers,1
P:wafer biscuits,2


■ 여러 품목을 나눈 언급 예


,source,doc,name,product_ids
118,dart,000100,NTCB-01/02,P:ntcb-01 | P:ntcb-02
149,dart,000120,"레미콘, 이형철근",P:ready-mixed concrete | P:deformed reinforcing bar
160,dart,000150,주단조품,P:castings and forgings | P:forgings
185,dart,000150,"기타(Film, Chemical 등)",P:industrial film | P:chemicals
186,dart,000150,"철판류, 외주비 외",P:steel sheet and plate | P:contract processing


■ 메모리 반도체 제품군 계층(맨 위 → 아래)


,id,name_ko,parent,depth,specs,n_mentions
2336,P:memory chips,메모리 반도체,,0,,14
2338,P:nand flash memory,NAND 플래시 메모리,,0,TLC NAND,6
11663,P:memory chip prices,메모리 반도체 가격,,0,,2
2346,P:automotive memory,차량용 메모리 반도체,,0,,1
2353,P:embedded multi-chip package,임베디드 멀티칩 패키지,,0,,1
2354,P:memory module,메모리 모듈,,0,,1
2355,P:random-access memory,랜덤 액세스 메모리,,0,,1
2357,P:pc memory,PC용 메모리,,0,,1
2360,P:hbm base die,HBM 베이스 다이,,0,2nm,1
2361,P:cxl memory,CXL 메모리,,0,,1


■ 후보 하위 제품 예


,id,parent,name_ko,search_terms
475,PC:embedded dram,P:dram,임베디드 DRAM,embedded DRAM | eDRAM
476,PC:reduced-latency dram,P:dram,저지연 DRAM,reduced latency DRAM | RLDRAM
477,PC:3d nand flash,P:nand flash memory,3D NAND 플래시,3D NAND | vertical NAND | V-NAND
478,PC:universal flash storage,P:nand flash memory,유니버설 플래시 스토리지,Universal Flash Storage | UFS flash storage
479,PC:load-reduced dimm,P:dual in-line memory module,부하 감소형 DIMM,load-reduced DIMM | LRDIMM
480,PC:clocked unbuffered dimm,P:unbuffered dimm,클록형 언버퍼드 DIMM,clocked unbuffered DIMM | CUDIMM
481,PC:compression attached memory module,P:dram module,압착 부착형 메모리 모듈,compression attached memory module | CAMM2
1227,PC:copper bonding wire,P:semiconductor bonding wire,구리 본딩 와이어,copper bonding wire | Cu bonding wire
1228,PC:palladium-coated copper bonding wire,P:semiconductor bonding wire,팔라듐 코팅 구리 본딩 와이어,palladium-coated copper bonding wire | PCC bonding wire
1229,PC:aluminum bonding wire,P:semiconductor bonding wire,알루미늄 본딩 와이어,aluminum bonding wire | aluminium bonding wire


## 24. 3단계 입력: 브랜드 → 제품

브랜드마다 무엇의 브랜드인지 정한다(Zepbound → GLP-1 수용체 작용제). 그래프에 쓰이는 브랜드 언급이 대상이다: 공시의 브랜드 전부, 뉴스는 관계에 쓰인 것.

- **블록**: 공시는 회사 하나, 뉴스는 기사 하나. 블록마다 그 회사가 공시에 적은 제품(또는 그 기사에 나온 제품)을 LLM 제품 사전의 제품으로 보여 준다. LLM은 브랜드가 그 제품과 같은 것이면 그 번호를 고른다.
- **브랜드 줄**: `b번호 | 이름 | 영어 이름 | 무엇인지 | 소유 | 관계`. 무엇인지는 공시의 제품 칸 또는 뉴스의 범주, 소유는 공시의 자사·고객사(ODM·OEM)·불명 또는 뉴스의 브랜드 소유 회사, 관계는 뉴스에서 이 브랜드가 쓰인 관계 예.
- **LLM이 정하는 것**: 종류(제품 브랜드 / 서비스 브랜드 / 회사·그룹 이름 / 브랜드 아님), 제품군(확정 273개 중 하나, 스키마로 제한), 일반 제품 이름(구체 이름과 한 단계 넓은 이름, v2부터), 블록 제품 번호.
- **코드가 정하는 제품**(같은 제품군의 LLM 사전 제품에서, 이름은 소문자·끝 s·하이픈·공백 무시): 블록 제품 번호 → 구체 이름이 같은 제품 → 넓은 이름이 같은 제품 → 이름 안에 통째로 든 제품(두 단어 이상, 가장 긴 것) → 마지막 단어가 같은 한 단어 제품(wheat beer → beer) → 없으면 제품군에만 잇는다.
- 요청 하나에 브랜드 줄 약 80개(회사·기사 블록은 나누지 않는다). 번호(p, b)는 요청 안에서 매긴다.

In [35]:
# ── 3단계 입력: 브랜드 줄(공시는 회사, 뉴스는 기사로 묶음), 블록의 제품 목록, 지시문, 스키마 ──
BRAND_PROMPT = 'product_brand_v3'                 # 버전별 차이는 25절 표
MAX_BRAND_LINES = 80                              # 요청 하나의 브랜드 줄 수(블록은 나누지 않는다)
MAX_BLOCK_PRODUCTS = 40                           # 블록 하나에 보여 주는 제품 수
BIZ = json.loads((ROOT / 'source' / 'dart' / 'business' / 'final' / 'dart_business_v5.json').read_text(encoding='utf-8'))['companies']
LPROD = PR.set_index('id')                        # LLM 사전 제품(23절)
ids_of = lambda s: [p for p in s.split(' | ') if p]
co_products = (MAP[(MAP['source'] == 'dart') & (MAP['field'] == '제품')].groupby('doc')['product_ids']
               .agg(lambda s: list(dict.fromkeys(p for x in s for p in ids_of(x)))).to_dict())
news_products = {(d, r): ids_of(p) for d, r, p in MAP.loc[MAP['source'] == 'news', ['doc', 'ref', 'product_ids']].itertuples(index=False)}
BLOCKS = {}                                       # 블록 ID → dict(head, sector, products, lines)
for c, d in BIZ.items():
    ls, seen = [], set()
    for b in d['brands']:
        if b['name'] in seen:
            continue
        seen.add(b['name'])
        ls.append(dict(key=('dart', c, '', b['name']), name=b['name'], name_en=b.get('name_en') or '', what=b.get('product') or '',
                       owner=b.get('owner_type') or '', rel=''))
    if ls:
        sec = co_info['wi26_sector'].get(c, '')
        BLOCKS['D:' + c] = dict(head=f"## 회사 | {co_info['stock_name'].get(c, c)} | {sec}", sector=sec, products=co_products.get(c, []), lines=ls)
for aid, a in news.items():
    ents = {e['id']: e for e in a['entities']}
    ls = []
    for e in a['entities']:
        if e['type'] != '상품·브랜드' or (aid, e['id']) not in in_relation:
            continue
        rel = '; '.join(f"{ents[r['subject']]['name']} {r['role']} {ents[r['object']]['name']}" for r in a['relations']
                        if e['id'] in (r['subject'], r['object'], r['product']) and r['subject'] in ents and r['object'] in ents)
        ls.append(dict(key=('news', aid, e['id'], e['name']), name=e['name'], name_en=e['name_en'] if e['name_en'] != e['name'] else '',
                       what=e.get('category') or '', owner=e.get('parent') or '', rel=rel[:200]))
    if ls:
        prods = list(dict.fromkeys(p for e in a['entities'] if e['type'] in ('제품·기술', '원자재·지표') for p in news_products.get((aid, e['id']), [])))
        BLOCKS['A:' + aid] = dict(head=f"## 기사 | {a['title']} | {art_sector.get(aid) or '업종 없음'}", sector=art_sector.get(aid, ''), products=prods, lines=ls)


def brand_request(block_ids):
    """블록들을 요청 글 하나로 만든다. 돌려주는 것: (글, b번호 → (블록, 줄), b번호 → {p번호: 제품 ID})."""
    out, where, pref, n_p, n_b = [], {}, {}, 0, 0
    for bid in block_ids:
        bl = BLOCKS[bid]
        local, ps = {}, []
        for pid in bl['products'][:MAX_BLOCK_PRODUCTS]:
            n_p += 1
            local[f'p{n_p}'] = pid
            ps.append(f"p{n_p} {LPROD.at[pid, 'name_en']} / {LPROD.at[pid, 'name_ko']}")
        out += [bl['head'], '제품: ' + (' | '.join(ps) if ps else '(없음)')]
        for i, l in enumerate(bl['lines']):
            n_b += 1
            where[f'b{n_b}'], pref[f'b{n_b}'] = (bid, i), local
            out.append(' | '.join([f'b{n_b}', l['name'], l['name_en'], l['what'], l['owner'], l['rel']]))
        out.append('')
    return '\n'.join(out).strip(), where, pref


def chunk_blocks(block_ids, n=MAX_BRAND_LINES):
    """블록을 순서대로 담아 브랜드 줄이 n개를 넘기 전에 요청을 나눈다."""
    reqs, cur, k = [], [], 0
    for bid in block_ids:
        m = len(BLOCKS[bid]['lines'])
        if cur and k + m > n:
            reqs.append(cur)
            cur, k = [], 0
        cur.append(bid)
        k += m
    return reqs + ([cur] if cur else [])


# 전체 요청: 공시는 업종·종목코드, 뉴스는 기사 업종 순서(비슷한 브랜드끼리 한 요청에)
order_ = sorted(BLOCKS, key=lambda b: (b[0], BLOCKS[b]['sector'], b))
BRAND_REQS = {f'brand_{i + 1:03d}': bs for i, bs in enumerate(chunk_blocks(order_))}
BRAND_TEXT = ((ROOT / 'prompts' / f'{BRAND_PROMPT}.md').read_text(encoding='utf-8')
              + '\n\n## 제품군 목록\n\nkey | 한글 이름 | 종류 | 범위\n' + family_table + CODEX_SUFFIX)
BRAND_HASH = text_hash(BRAND_TEXT + MODEL + EFFORT)
BRAND_KINDS = ['제품 브랜드', '서비스 브랜드', '회사·그룹 이름', '브랜드 아님']
brand_schema = {
    'type': 'object', 'additionalProperties': False, 'required': ['brands', 'notes'],
    'properties': {
        'brands': {'type': 'array', 'items': {
            'type': 'object', 'additionalProperties': False, 'required': ['line', 'kind', 'family', 'name_en', 'name_ko', 'broad_en', 'product_ref', 'note'],
            'properties': {'line': {'type': 'string'}, 'kind': {'type': 'string', 'enum': BRAND_KINDS},
                           'family': {'type': 'string', 'enum': [''] + sorted(FAMILIES['key'])},
                           'name_en': {'type': 'string'}, 'name_ko': {'type': 'string'}, 'broad_en': {'type': 'string'},
                           'product_ref': {'type': 'string'}, 'note': {'type': 'string'}}}},
        'notes': {'type': 'string'}}}
BRAND_SCHEMA = LLM / 'schemas' / f'{BRAND_PROMPT}.schema.json'
BRAND_SCHEMA.write_text(json.dumps(brand_schema, ensure_ascii=False, indent=1), encoding='utf-8')
n_lines = pd.Series({b: len(v['lines']) for b, v in BLOCKS.items()})
print(f"브랜드 줄 {n_lines.sum():,}개(공시 {n_lines[n_lines.index.str.startswith('D:')].sum():,}, 뉴스 {n_lines[n_lines.index.str.startswith('A:')].sum():,}) / "
      f"블록 {len(BLOCKS):,}개(제품 목록이 빈 블록 {sum(not v['products'] for v in BLOCKS.values())}) / 요청 {len(BRAND_REQS)}건 / 지시문 {len(BRAND_TEXT):,}자")
print(brand_request(['D:000020'])[0][:1200])

브랜드 줄 5,838개(공시 5,015, 뉴스 823) / 블록 966개(제품 목록이 빈 블록 218) / 요청 81건 / 지시문 28,431자
## 회사 | 동화약품 | 건강관리
제품: p1 pharmaceuticals / 의약품 | p2 quasi-drugs / 의약외품 | p3 orthopedic implants / 정형외과용 임플란트 | p4 long-acting analgesics / 장기지속형 진통제 | p5 primary axillary hyperhidrosis treatments / 원발성 겨드랑이 다한증 치료제 | p6 onychomycosis treatments / 손발톱무좀 치료제 | p7 DW6013 / DW6013 | p8 DW6014 / DW6014 | p9 DW6017 / DW6017
b1 | 까스활명수큐액 | Gas Whal Myung Soo Q | 의약품 | 자사 | 
b2 | 미인활명수 | Miin Whal Myung Soo | 의약품 | 자사 | 
b3 | 까스활액 | Gas Whal | 의약품 | 자사 | 
b4 | 후시딘연고 | Fucidin ointment | 의약품 | 불명 | 
b5 | 후시딘겔 | Fucidin gel | 의약품 | 불명 | 
b6 | 후시딘밴드 | Fucidin band | 의약품·의약외품 | 불명 | 
b7 | 판콜에이 | Pancol A | 의약품 | 자사 | 
b8 | 판콜에스 | Pancol S | 의약품 | 자사 | 
b9 | 잇치 | Itchi | 의약품 | 자사 | 
b10 | 라코르 | Rakor | 순환당뇨 | 불명 | 
b11 | 맥페란 | Macperan | 소화기 의약품 | 자사 | 
b12 | ILIAD | ILIAD | 정형외과용 임플란트 | 불명 | 
b13 | ZENIUS | ZENIUS | 정형외과용 임플란트 | 불명 | 
b14 | ECCLOCK gel | ECCLOCK gel | 원발성겨드랑이다한증치료제 | 불명 | 
b15 | LUCONAC Solution | L

## 25. 3단계 시범 묶음과 정답 확인 규칙

시범은 고정 블록으로 만든다(공시 회사 16곳 + 뉴스 기사). 어려운 사례:

- 의약품 브랜드를 약효군으로(까스활명수 → 소화제, 카나브 → 고혈압 치료제, 램시마 → 면역질환 치료제), 다른 회사 의약품을 만들어 주는 회사(한국콜마의 케이캡).
- 회사·그룹 이름이 브랜드로 적힌 것(현대위아, HYUNDAI, HANKOOK)과 다른 회사 브랜드를 파는 것(미창석유공업의 BP CASTROL), 고객사 브랜드(한세실업의 GAP).
- 서비스 브랜드(위버스, 반트, YouTube, Azure, Boost Mobile), 콘텐츠(리니지2 레볼루션), 가상자산(마브렉스).
- 뉴스: 같은 제품 GLP-1(Zepbound, Ozempic, Mounjaro), 소재 브랜드(MITSUI PELLICLE → 펠리클), 브랜드가 아닌 것(North America coffee business, Advent calendars, mortgage-backed securities, eggs).

정답 확인 규칙(`BRAND_CHECKS`): `kind`(종류), `family`(제품군), `name`(일반 제품 이름에 그 말이 들어감), `prod`(코드가 정한 제품이 정답 중 하나). 함께 형식 점검(입력 줄이 모두 결과에 있는지, 없는 줄번호·블록 밖 제품 번호)과 코드가 제품을 정한 비율(블록 제품 번호 / 이름으로 찾음 / 제품군에만)을 본다.

| 지시문 | 정답 확인 | 고친 것 |
|---|---|---|
| v1 | 48/49(틀린 1개는 CREORA를 "스판덱스, 나일론"이라 synthetic fibers로 넓게 적음). 코드가 제품을 정한 것: 블록 제품 번호 92, 이름 49, 이름 안의 제품 18, 제품군만 68, 없음 7 | - |
| v2 | 49/49. 제품을 정한 것: 블록 제품 번호 106, 이름 44, 넓은 이름 45, 이름 안의 제품 4, 마지막 단어 2, 제품군만 27, 없음 6 | 한 단계 넓은 이름(`broad_en`)을 함께 받음(제품군만 남은 68줄 대부분은 사전에 넓은 제품이 있었다: topical antibiotics → antibiotics, electric sedans → electric vehicles), 여러 품목이 적혀 있으면 브랜드가 가리키는 하나, 범주가 없어도 알려진 브랜드는 아는 대로(Macan을 비워 둠). 코드: 마지막 단어가 같은 한 단어 제품도 잇는다 |
| v3 | 49/49. 296줄(무작위 블록 포함) 중 제품까지 이은 것 258(87%): 블록 제품 번호 145, 넓은 이름 63, 이름 43, 마지막 단어 4, 이름 안의 제품 3. 제품군만 30, 없음 8(여러 제품에 두루 쓰인 회사 이름, FIFA World Cup 등). 종류가 안정됨(GAP·HANKOOK·BP CASTROL → 제품 브랜드) | 회사·그룹 이름은 여러 제품에 두루 쓰이는 것만(v2가 GAP을 회사 이름으로 바꿈), 모음 제품군·other는 목록을 다 본 뒤에만(홈매트 → other). 시범에 무작위 블록 16개를 더함. 코드: 단어마다 복수형을 단수로(gene therapies ≠ gene therapy) |

> **다시 실행할 때**: 시범 사례와 정답 확인은 2026-11 데이터와 LLM 결과에서 고른 것이다. 데이터를 새로 만들면 없는 사례는 건너뛰고 알린다. 새 데이터로 지시문을 다시 다듬는다면 그 데이터에서 어려운 사례를 다시 골라 넣기를 권한다.

In [36]:
# ── 3단계 시범 블록(고정)과 정답 확인 규칙 ──
BRAND_SAMPLE_COMPANIES = ['동화약품', '보령', '한국콜마', 'HD현대', 'HD건설기계', '현대위아', '미창석유공업', '효성티앤씨', '한세실업', '하이브',
                          '신영와코루', '하이트진로', '호텔신라', '넷마블', '셀트리온', '한국타이어앤테크놀로지']
BRAND_SAMPLE_NEWS = ['Zepbound', 'Ozempic', 'Mounjaro', 'Elevidys', 'Dupixent', 'YouTube', 'Azure', 'Boost Mobile', 'MITSUI PELLICLE', 'Optimus',
                     'Boxster', 'Cybertruck', 'Tesla Model 3', 'TikTok', 'North America coffee business', 'Advent calendars',
                     'mortgage-backed securities', 'eggs', 'Pedigree', 'Doritos', 'CleanBeam']
code_of = {v: k for k, v in co_info['stock_name'].items()}
gone = [n for n in BRAND_SAMPLE_COMPANIES if n not in code_of or 'D:' + code_of[n] not in BLOCKS]
s_blocks = ['D:' + code_of[n] for n in BRAND_SAMPLE_COMPANIES if n not in gone]
for name in BRAND_SAMPLE_NEWS:                     # 그 브랜드가 나온 첫 기사(기사 ID 순서)
    b = next((b for b in sorted(BLOCKS) if b.startswith('A:') and any(l['name'] == name for l in BLOCKS[b]['lines'])), None)
    if b:
        s_blocks.append(b)
    else:
        gone.append(name)
if gone:
    print(f'※ 시범 회사·브랜드 {len(gone)}개는 이번 데이터에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다):', gone)
s_blocks = list(dict.fromkeys(s_blocks))
# v3부터: 일반 브랜드에서 얼마나 맞는지 볼 무작위 블록(공시 회사 8곳, 기사 8건, 고정). 정답 규칙 없이 사람이 본다
rand_d = pd.Series(sorted(b for b in BLOCKS if b.startswith('D:') and b not in s_blocks and len(BLOCKS[b]['lines']) <= 15)).sample(8, random_state=3).tolist()
rand_a = pd.Series(sorted(b for b in BLOCKS if b.startswith('A:') and b not in s_blocks)).sample(8, random_state=3).tolist()
BRAND_RANDOM = rand_d + rand_a
s_blocks += BRAND_RANDOM
BRAND_SAMPLE = {f'bsample_{i + 1}': bs for i, bs in enumerate(chunk_blocks(s_blocks))}
BRAND_SAMPLE_INPUTS = {k: brand_request(bs) for k, bs in BRAND_SAMPLE.items()}   # 요청 → (글, b번호 → 줄, b번호 → 제품 번호)
drug = lambda f: 'drug' in f or 'pharma' in f or 'medicine' in f or 'therap' in f
BRAND_CHECKS = [  # (회사 이름 또는 뉴스, 브랜드 이름, 확인, 정답)
    ('동화약품', '까스활명수큐액', 'family', {'gastrointestinal_and_liver_drugs'}), ('동화약품', '까스활명수큐액', 'prod', {'P:digestive medicines', 'P:herbal digestive fluid extracts'}),
    ('동화약품', '후시딘연고', 'family', {'dermatology_drugs', 'anti_infective_drugs'}),
    ('보령', '카나브정', 'family', {'cardiovascular_and_blood_disorder_drugs'}), ('보령', '듀카브정', 'family', {'cardiovascular_and_blood_disorder_drugs'}),
    ('한국콜마', '케이캡', 'family', {'gastrointestinal_and_liver_drugs'}), ('한국콜마', '컨디션', 'family', {'dietary_supplements', 'nonalcoholic_beverages'}),
    ('HD현대', 'HIMSEN', 'family', {'marine_engines', 'power_generation_equipment'}), ('HD현대', 'DEVELON', 'family', {'construction_and_mining_equipment'}),
    ('HD건설기계', 'HX220A', 'family', {'construction_and_mining_equipment'}), ('HD건설기계', 'HX220A', 'name', ['excavator']),
    ('현대위아', '현대위아', 'kind', {'회사·그룹 이름'}), ('미창석유공업', 'BP CASTROL', 'family', {'petroleum_feedstocks_and_lubricants'}),
    ('효성티앤씨', 'CREORA', 'name', ['spandex']), ('효성티앤씨', 'CREORA', 'family', {'textiles_and_leather'}),
    ('한세실업', 'GAP', 'family', {'apparel_and_accessories'}), ('하이브', '위버스', 'kind', {'서비스 브랜드'}),
    ('하이브', '위버스', 'family', {'social_media_and_community_platforms'}), ('신영와코루', '마더피아', 'family', {'apparel_and_accessories'}),
    ('하이트진로', '참이슬', 'name', ['soju']), ('하이트진로', '석수', 'family', {'nonalcoholic_beverages'}),
    ('넷마블', '리니지2 레볼루션', 'family', {'video_games'}), ('넷마블', '마브렉스(MBX)', 'family', {'digital_asset_and_blockchain_services'}),
    ('셀트리온', '램시마', 'family', {'immunology_drugs'}), ('셀트리온', '허쥬마', 'family', {'oncology_drugs'}),
    ('한국타이어앤테크놀로지', 'VENTUS', 'family', {'tires'}),
    ('뉴스', 'Zepbound', 'family', {'diabetes_and_obesity_drugs'}), ('뉴스', 'Zepbound', 'name', ['glp-1']),
    ('뉴스', 'Ozempic', 'family', {'diabetes_and_obesity_drugs'}), ('뉴스', 'Mounjaro', 'family', {'diabetes_and_obesity_drugs'}),
    ('뉴스', 'Elevidys', 'family', {'cell_and_gene_therapies'}), ('뉴스', 'YouTube', 'kind', {'서비스 브랜드'}),
    ('뉴스', 'Azure', 'kind', {'서비스 브랜드'}), ('뉴스', 'Azure', 'family', {'cloud_computing_services'}),
    ('뉴스', 'Boost Mobile', 'family', {'telecommunications_services'}), ('뉴스', 'MITSUI PELLICLE', 'name', ['pellicle']),
    ('뉴스', 'MITSUI PELLICLE', 'family', {'semiconductor_lithography_materials'}), ('뉴스', 'Optimus', 'family', {'mobile_and_service_robots'}),
    ('뉴스', 'Dupixent', 'family', {'immunology_drugs', 'dermatology_drugs', 'respiratory_and_allergy_drugs'}),
    ('뉴스', 'mortgage-backed securities', 'kind', {'브랜드 아님'}), ('뉴스', 'eggs', 'kind', {'브랜드 아님'}),
    ('뉴스', 'Cybertruck', 'family', {'electric_vehicles'}), ('뉴스', 'Tesla Model 3', 'family', {'electric_vehicles'}),
    ('뉴스', 'Boxster', 'family', {'passenger_vehicles'}), ('뉴스', 'TikTok', 'family', {'social_media_and_community_platforms'}),
    ('뉴스', 'North America coffee business', 'kind', {'브랜드 아님'}), ('뉴스', 'Advent calendars', 'kind', {'브랜드 아님'}),
    ('뉴스', 'Pedigree', 'family', {'pet_products', 'animal_feed'}), ('뉴스', 'Doritos', 'family', {'bakery_and_confectionery', 'prepared_and_packaged_foods'}),
]
present = {('뉴스' if b.startswith('A:') else co_info['stock_name'].get(b[2:], b[2:]), l['name']) for b in s_blocks for l in BLOCKS[b]['lines']}
gone = [c for c in BRAND_CHECKS if not any(n == c[1] and (w == c[0]) for w, n in present)]
if gone:
    print(f'※ 정답 확인 {len(gone)}개는 대상이 이번 시범에 없어 뺀다 (시범 사례는 2026-11 데이터에서 고른 목록이다. 다시 실행해 데이터가 달라지면 없는 사례는 건너뛴다)')
BRAND_CHECKS = [c for c in BRAND_CHECKS if c not in gone]
n_s = sum(len(BLOCKS[b]['lines']) for b in s_blocks)
print(f'시범 블록 {len(s_blocks)}개(어려운 사례 공시 {len(BRAND_SAMPLE_COMPANIES)}·기사 {len(s_blocks) - len(BRAND_SAMPLE_COMPANIES) - len(BRAND_RANDOM)}, 무작위 {len(BRAND_RANDOM)}) / 브랜드 줄 {n_s}개 / 요청 {len(BRAND_SAMPLE)}건 '
      f'{ {k: len(v[1]) for k, v in BRAND_SAMPLE_INPUTS.items()} } / 정답 확인 {len(BRAND_CHECKS)}개')

시범 블록 53개(어려운 사례 공시 16·기사 21, 무작위 16) / 브랜드 줄 296개 / 요청 4건 {'bsample_1': 71, 'bsample_2': 79, 'bsample_3': 79, 'bsample_4': 67} / 정답 확인 49개


## 26. 3단계 시범 실행과 채점 (Codex)

시범 요청을 Codex로 보낸다(같은 지시문·입력이면 건너뛴다). 채점: 형식 점검, 정답 확인, 코드가 제품을 정한 방법별 수. 결과 전체를 표로 보여 사람이 본다.

In [37]:
# ── 3단계 시범 실행 ──
runs = read_runs()
done = set(runs.loc[(runs['stage'] == BRAND_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BRAND_HASH),
                    ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: v[0] for k, v in BRAND_SAMPLE_INPUTS.items() if (k, text_hash(v[0])) not in done}
print(f'이번에 돌릴 것 {len(todo)}건')
if RUN and todo:
    run_items(BRAND_PROMPT, todo, BRAND_TEXT, BRAND_SCHEMA, BRAND_HASH)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

이번에 돌릴 것 0건
RUN = False: 실행하지 않았다.


In [38]:
# ── 3단계 채점: 형식 점검, 코드가 정한 제품, 정답 확인, 결과 보기 ──
def norm_name(s):
    """이름 비교용: 소문자, 하이픈·빗금은 공백, 단어마다 복수형을 단수로(therapies → therapy, glasses → glass, fibers → fiber)."""
    out = []
    for w in re.sub(r'[\s\-_/]+', ' ', low(s)).split():
        if len(w) > 4 and w.endswith('ies'):
            w = w[:-3] + 'y'
        elif len(w) > 4 and w.endswith(('sses', 'shes', 'ches', 'xes')):
            w = w[:-2]
        elif len(w) > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')):
            w = w[:-1]
        out.append(w)
    return ' '.join(out)
FAM_PRODUCTS = {}                                     # 제품군 → {정규화한 이름: 제품 ID}
for r in PR.itertuples():
    FAM_PRODUCTS.setdefault(r.family, {}).setdefault(norm_name(r.name_en), r.id)


def brand_product(res, pref):
    """결과 한 줄 → (제품 ID, 방법). 블록 제품 번호 → 같은 제품군에서 이름이 같은 제품 → 넓은 이름이 같은 제품
    → 이름 안에 통째로 든 제품 이름(가장 긴 것, 두 단어 이상) → 마지막 단어와 같은 한 단어 제품 → 제품군만."""
    if res['product_ref'] in pref:
        return pref[res['product_ref']], '블록 제품 번호'
    if res['product_ref']:
        return '', '블록 밖 번호'
    fam = FAM_PRODUCTS.get(res['family'], {})
    names = [norm_name(x) for x in (res.get('name_en'), res.get('broad_en')) if x]
    for how, nm in zip(['이름으로 찾음', '넓은 이름으로 찾음'], names):
        if nm in fam:
            return fam[nm], how
    inside = [k for k in fam if ' ' in k and any(re.search(rf'(^| ){re.escape(k)}($| )', nm) for nm in names)]
    if inside:
        return fam[max(inside, key=len)], '이름 안의 제품'
    head = [nm.split()[-1] for nm in names if ' ' in nm]          # 마지막 단어(wheat beer → beer)
    for h in head:
        if h in fam:
            return fam[h], '마지막 단어'
    return '', '제품군만' if res['family'] else '제품 없음'


def brand_rows(results, inputs):
    """요청별 결과를 줄마다 표로 펼친다."""
    rows = []
    for item, out in results.items():
        _, where, pref = inputs[item]
        got = {x['line']: x for x in out['brands']}
        for b, (bid, i) in where.items():
            l, x = BLOCKS[bid]['lines'][i], got.get(b)
            row = dict(item=item, line=b, block=bid, who=BLOCKS[bid]['head'].split(' | ')[1][:30] if bid.startswith('D:') else '뉴스',
                       name=l['name'], what=l['what'], found=x is not None)
            if x:
                pid, how = brand_product(x, pref[b])
                row.update(kind=x['kind'], family=x['family'], name_en=x['name_en'], name_ko=x['name_ko'], broad_en=x.get('broad_en', ''),
                           ref=x['product_ref'], product=pid, how=how, note=x['note'])
            rows.append(row)
        rows += [dict(item=item, line=b, block='', name='(입력에 없는 줄번호)', found=True) for b in set(got) - set(where)]
    return pd.DataFrame(rows).fillna('')


def brand_check(df, who, name, what, answer):
    x = df[(df['name'] == name) & ((df['who'] == '뉴스') if who == '뉴스' else (df['who'] == who))]
    if not len(x) or not x['found'].all():
        return False, '결과 없음'
    v = x.iloc[0]
    if what == 'name':
        return any(a in v['name_en'].lower() for a in answer), v['name_en']
    col = {'kind': 'kind', 'family': 'family', 'prod': 'product'}[what]
    return v[col] in answer, f"{v[col]} ({v['how']})" if what == 'prod' else v[col]


runs = read_runs()
ok = runs[(runs['stage'] == BRAND_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BRAND_HASH) & runs['item'].isin(BRAND_SAMPLE_INPUTS)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
BRAND_RESULT = {r.item: parse_result(json.loads((LLM / 'raw' / BRAND_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))
                for r in ok.itertuples() if r.input_hash == text_hash(BRAND_SAMPLE_INPUTS[r.item][0])}
if BRAND_RESULT:
    BR = brand_rows(BRAND_RESULT, BRAND_SAMPLE_INPUTS)
    graded_b = pd.DataFrame([dict(who=w, name=n, check=c, ok=g[0], got=g[1]) for w, n, c, a in BRAND_CHECKS for g in [brand_check(BR, w, n, c, a)]])
    print(f"요청 {len(BRAND_RESULT)}/{len(BRAND_SAMPLE_INPUTS)} / 브랜드 줄 {len(BR[BR['block'] != ''])}개 중 결과 없음 {(~BR['found']).sum()} / "
          f"입력에 없는 줄번호 {(BR['block'] == '').sum()} / 정답 확인 {graded_b.ok.sum()}/{len(graded_b)}")
    print('종류', BR['kind'].value_counts().to_dict())
    print('제품을 정한 방법', BR['how'].value_counts().to_dict())
    display(graded_b[~graded_b.ok])
    pd.set_option('display.max_rows', 300)
    display(BR[BR['block'] != ''][['who', 'name', 'what', 'kind', 'family', 'name_en', 'broad_en', 'product', 'how', 'note']])
else:
    print('아직 결과가 없다.')

요청 4/4 / 브랜드 줄 296개 중 결과 없음 0 / 입력에 없는 줄번호 0 / 정답 확인 49/49
종류 {'제품 브랜드': 270, '서비스 브랜드': 11, '브랜드 아님': 11, '회사·그룹 이름': 4}
제품을 정한 방법 {'블록 제품 번호': 145, '넓은 이름으로 찾음': 63, '이름으로 찾음': 43, '제품군만': 30, '제품 없음': 8, '마지막 단어': 4, '이름 안의 제품': 3}


,who,name,check,ok,got


,who,name,what,kind,family,name_en,broad_en,product,how,note
0,동화약품,까스활명수큐액,의약품,제품 브랜드,gastrointestinal_and_liver_drugs,digestive medicines,digestive medicines,P:digestive medicines,이름으로 찾음,
1,동화약품,미인활명수,의약품,제품 브랜드,gastrointestinal_and_liver_drugs,digestive medicines,digestive medicines,P:digestive medicines,이름으로 찾음,
2,동화약품,까스활액,의약품,제품 브랜드,gastrointestinal_and_liver_drugs,digestive medicines,digestive medicines,P:digestive medicines,이름으로 찾음,
3,동화약품,후시딘연고,의약품,제품 브랜드,anti_infective_drugs,topical antibiotics,antibiotics,P:antibiotics,넓은 이름으로 찾음,
4,동화약품,후시딘겔,의약품,제품 브랜드,anti_infective_drugs,topical antibiotics,antibiotics,P:antibiotics,넓은 이름으로 찾음,
5,동화약품,후시딘밴드,의약품·의약외품,제품 브랜드,medical_implants_and_consumables,adhesive wound dressings,wound dressings,P:wound dressings,넓은 이름으로 찾음,
6,동화약품,판콜에이,의약품,제품 브랜드,respiratory_and_allergy_drugs,combination cold medicines,cold medicines,P:cold medicine,넓은 이름으로 찾음,
7,동화약품,판콜에스,의약품,제품 브랜드,respiratory_and_allergy_drugs,combination cold medicines,cold medicines,P:cold medicine,넓은 이름으로 찾음,
8,동화약품,잇치,의약품,제품 브랜드,dermatology_drugs,medicated toothpaste for gum disease,gum disease treatments,,제품군만,
9,동화약품,라코르,순환당뇨,제품 브랜드,cardiovascular_and_blood_disorder_drugs,cardiovascular drugs,cardiovascular drugs,P:cardiovascular drugs,이름으로 찾음,세부 약효군을 입력에서 확인할 수 없음.


## 27. 3단계 전체 실행 (Codex 구독)

브랜드 요청 81건을 Codex 구독으로 돌린다(동시에 3건). 규모가 작아(5시간 한도의 약 80%로 어림) 배치 API 대신 구독으로 한다.

- 같은 지시문·입력으로 끝난 요청은 건너뛴다. 다시 실행하면 남은 요청부터 한다.
- **사용량 한도에 걸리면** 새 요청을 멈추고 텔레그램으로 알린다(`.env`의 `ALERT_TELEGRAM_TOKEN`, `ALERT_TELEGRAM_CHAT_ID`). Codex 계정을 바꾸거나(`codex login`) 한도가 풀린 뒤 이 셀을 다시 실행한다.
- 시작과 끝(또는 한도·오류로 남은 요청)도 텔레그램으로 알린다.
- `RUN = True`일 때만 보낸다.

In [39]:
# ── 3단계 전체 실행(Codex 구독): 남은 요청을 돌리고, 한도에 걸리면 텔레그램으로 알린다 ──
import urllib.parse
import urllib.request


def notify(text):
    """텔레그램으로 알린다(.env의 ALERT_TELEGRAM_TOKEN·CHAT_ID). 실패해도 멈추지 않고, 오류 메시지에서도 토큰을 가린다."""
    token, chat = env_value('ALERT_TELEGRAM_TOKEN'), env_value('ALERT_TELEGRAM_CHAT_ID')
    print('알림:', text)
    if not (token and chat):
        print('  텔레그램 설정이 없어 화면에만 적었다.')
        return
    try:
        data = urllib.parse.urlencode({'chat_id': chat, 'text': text}).encode()
        urllib.request.urlopen(f'https://api.telegram.org/bot{token}/sendMessage', data=data, timeout=20).read()
    except Exception as e:
        print('  텔레그램 알림 실패:', mask(str(e).replace(token, '***'))[:200])


BRAND_INPUTS = {k: brand_request(bs) for k, bs in BRAND_REQS.items()}   # 요청 → (글, b번호 → 줄, b번호 → 제품 번호)


def brand_todo():
    """아직 같은 지시문·입력으로 끝나지 않은 요청."""
    runs = read_runs()
    done = set(runs.loc[(runs['stage'] == BRAND_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BRAND_HASH),
                        ['item', 'input_hash']].itertuples(index=False, name=None))
    return {k: v[0] for k, v in BRAND_INPUTS.items() if (k, text_hash(v[0])) not in done}


todo = brand_todo()
print(f'3단계 전체 {len(BRAND_INPUTS)}건 / 끝남 {len(BRAND_INPUTS) - len(todo)} / 남음 {len(todo)}')
if RUN and todo:
    started = dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()
    notify(f'[graph_db] 브랜드 → 제품(3단계) Codex 실행 시작: {len(todo)}건 남음(전체 {len(BRAND_INPUTS)}건)')
    run_items(BRAND_PROMPT, todo, BRAND_TEXT, BRAND_SCHEMA, BRAND_HASH)
    left = brand_todo()
    runs = read_runs()
    now_runs = runs[(runs['stage'] == BRAND_PROMPT) & (runs['run_utc'] >= started)]
    n_done = len(BRAND_INPUTS) - len(left)
    if left and (now_runs['status'] == 'limit').any():
        notify(f'[graph_db] Codex 사용량 한도에 걸렸다. 브랜드 {n_done}/{len(BRAND_INPUTS)}건 끝, {len(left)}건 남음. '
               f'Codex 계정을 바꾼 뒤(codex login) product_llm 27절을 다시 실행하면 남은 것부터 한다.')
    elif left:
        notify(f'[graph_db] 브랜드 {n_done}/{len(BRAND_INPUTS)}건 끝, {len(left)}건은 한도가 아닌 이유로 실패'
               f'({now_runs["status"].value_counts().to_dict()}). 다시 실행하면 남은 것부터 한다.')
    else:
        notify(f'[graph_db] 브랜드 → 제품(3단계) Codex 실행 끝: {len(BRAND_INPUTS)}건 모두 완료.')
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

3단계 전체 81건 / 끝남 81 / 남음 0
RUN = False: 실행하지 않았다.


## 28. 3단계 결과 모으기와 점검

전체 실행 결과를 브랜드 줄마다 표로 모아 `stage3_brands_지시문.csv`에 쓴다(내용이 바뀔 때만). 줄은 원래 언급(출처, 문서, 기사 개체 번호, 이름)으로 가리켜, 적용할 때 규칙 사전의 브랜드 언급에 붙인다.

점검: 결과에 없는 줄·없는 줄번호·블록 밖 제품 번호(형식), 종류와 코드가 제품을 정한 방법의 분포, 모음 제품군으로 간 비율, 같은 브랜드 이름이 줄마다 다른 제품군으로 간 것(같은 이름이 다른 회사의 다른 브랜드일 수도 있다).

In [40]:
# ── 3단계 결과 모으기와 형식 점검 ──
runs = read_runs()
ok = runs[(runs['stage'] == BRAND_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BRAND_HASH) & runs['item'].isin(BRAND_INPUTS)]
ok = ok.sort_values('run_utc').drop_duplicates('item', keep='last')
BRAND_FULL = {r.item: parse_result(json.loads((LLM / 'raw' / BRAND_PROMPT / f'codex-{MODEL}' / r.raw_file).read_text(encoding='utf-8')))
              for r in ok.itertuples() if r.input_hash == text_hash(BRAND_INPUTS[r.item][0])}
B3 = brand_rows(BRAND_FULL, BRAND_INPUTS)
key_of = {(item, b): BLOCKS[bid]['lines'][i]['key'] for item, v in BRAND_INPUTS.items() for b, (bid, i) in v[1].items()}
B3[['source', 'doc', 'ref', 'name_key']] = [key_of.get((i, l), ('', '', '', '')) for i, l in zip(B3['item'], B3['line'])]
lines_in = B3[B3['block'] != '']
print(f"요청 {len(BRAND_FULL)}/{len(BRAND_INPUTS)} / 브랜드 줄 {len(lines_in):,}개 중 결과 없음 {(~lines_in['found']).sum()} / "
      f"입력에 없는 줄번호 {(B3['block'] == '').sum()} / 블록 밖 제품 번호 {(lines_in['how'] == '블록 밖 번호').sum()}")
OUT3 = lines_in[lines_in['found']][['source', 'doc', 'ref', 'name_key', 'kind', 'family', 'name_en', 'name_ko', 'broad_en', 'product', 'how', 'note', 'item', 'line']]
OUT3 = OUT3.rename(columns={'name_key': 'name'})
if len(OUT3):                                      # 결과가 없으면(실행 전) 파일을 건드리지 않는다
    print(f'stage3_brands_{BRAND_PROMPT}.csv', save_csv(OUT3, LLM / f'stage3_brands_{BRAND_PROMPT}.csv'))

요청 81/81 / 브랜드 줄 5,838개 중 결과 없음 0 / 입력에 없는 줄번호 0 / 블록 밖 제품 번호 0
stage3_brands_product_brand_v3.csv 그대로


In [41]:
# ── 확인: 종류, 제품을 정한 방법, 제품군, 모음 제품군, 같은 이름·다른 제품군, 무작위 예 ──
if len(OUT3):
    n = len(OUT3)
    print('종류', {k: f'{v} ({v / n:.0%})' for k, v in OUT3['kind'].value_counts().items()})
    print('제품을 정한 방법', {k: f'{v} ({v / n:.0%})' for k, v in OUT3['how'].value_counts().items()})
    print(f"제품까지 이은 줄 {(OUT3['product'] != '').mean():.1%} / 제품군만 {(OUT3['how'] == '제품군만').mean():.1%} / 제품 없음 {(OUT3['how'] == '제품 없음').mean():.1%}")
    catch = set(FAMILIES.loc[FAMILIES['catch_all'] == 'True', 'key'])
    print(f"모음 제품군으로 간 줄 {OUT3['family'].isin(catch).sum()}개", OUT3.loc[OUT3['family'].isin(catch), 'family'].value_counts().to_dict())
    print('많이 나온 제품군', OUT3['family'].value_counts().head(12).to_dict())
    fam_n = OUT3[OUT3['family'] != ''].groupby('name')['family'].nunique()
    print(f'같은 브랜드 이름이 여러 줄에 나온 것 {(OUT3.groupby("name").size() > 1).sum()}개 중 제품군이 갈린 것 {(fam_n > 1).sum()}개', list(fam_n[fam_n > 1].index[:15]))
    print('■ 제품 없음(회사·그룹 이름, 브랜드 아님 등) 예')
    display(OUT3[OUT3['how'] == '제품 없음'].sample(min(10, (OUT3['how'] == '제품 없음').sum()), random_state=1)[['source', 'name', 'kind', 'note']])
    print('■ 무작위 20줄')
    display(OUT3.sample(20, random_state=7)[['source', 'name', 'kind', 'family', 'name_en', 'broad_en', 'product', 'how']])

종류 {'제품 브랜드': '5028 (86%)', '서비스 브랜드': '588 (10%)', '회사·그룹 이름': '124 (2%)', '브랜드 아님': '98 (2%)'}
제품을 정한 방법 {'블록 제품 번호': '3334 (57%)', '제품군만': '738 (13%)', '넓은 이름으로 찾음': '734 (13%)', '이름으로 찾음': '616 (11%)', '제품 없음': '220 (4%)', '이름 안의 제품': '99 (2%)', '마지막 단어': '97 (2%)'}
제품까지 이은 줄 83.6% / 제품군만 12.6% / 제품 없음 3.8%
모음 제품군으로 간 줄 59개 {'industrial_chemicals_and_additives': 24, 'industrial_equipment_and_tools': 18, 'other': 12, 'industrial_metals_and_components': 5}
많이 나온 제품군 {'cosmetics_and_personal_care': 374, 'video_games': 300, 'cardiovascular_and_blood_disorder_drugs': 282, 'apparel_and_accessories': 239, '': 220, 'bakery_and_confectionery': 198, 'diabetes_and_obesity_drugs': 177, 'gastrointestinal_and_liver_drugs': 149, 'nonalcoholic_beverages': 126, 'neurological_and_psychiatric_drugs': 123, 'dietary_supplements': 105, 'medical_implants_and_consumables': 93}
같은 브랜드 이름이 여러 줄에 나온 것 690개 중 제품군이 갈린 것 36개 ['4DX', 'Akamine', 'BMW', 'Bio Miwon', 'DR.Presso', 'Ecomate®', 'FX-Series', 'HANKOOK',

,source,name,kind,note
3351,dart,SK하이닉스,회사·그룹 이름,DRAM과 NAND 등 여러 반도체 제품에 쓰이는 회사 이름
1883,dart,코스맥스,회사·그룹 이름,상표 사용으로 기재된 회사명이다.
3597,dart,HICO,회사·그룹 이름,여러 중공업 제품에 쓰이는 회사 이름.
5122,dart,VENOM,제품 브랜드,신발용 소재의 재질이 명시되지 않았다.
3324,dart,TEL,회사·그룹 이름,"TEL은 장비 회사 이름으로, 실리콘 부품의 제품 브랜드로 특정되지 않음."
3562,dart,LG,회사·그룹 이름,여러 가전 및 영상기기에 쓰이는 그룹 브랜드.
2957,dart,대원전선,회사·그룹 이름,여러 종류의 전선과 케이블에 쓰이는 회사 이름이다.
2262,dart,브레이닝캡슐,제품 브랜드,제품의 용도를 특정하기 어려움.
3332,dart,GEMINI,제품 브랜드,제조 장비의 구체적인 종류를 입력만으로 특정할 수 없음.
680,news,home and personal care products,브랜드 아님,생활용품과 개인용품을 함께 가리켜 단일 제품을 특정할 수 없다.


■ 무작위 20줄


,source,name,kind,family,name_en,broad_en,product,how
2410,dart,중외엔에스주사액,제품 브랜드,plasma_medicines_and_infusion_solutions,normal saline solutions,basic intravenous fluids,P:basic intravenous fluids,블록 제품 번호
2226,dart,자누메트XR서방정,제품 브랜드,diabetes_and_obesity_drugs,extended-release DPP-4 inhibitor and metformin combinations,diabetes drugs,P:diabetes drugs,넓은 이름으로 찾음
676,news,Shin Ramyun,제품 브랜드,noodles_and_pasta,instant noodles,instant noodles,P:instant noodles,이름으로 찾음
334,news,Mass Effect,제품 브랜드,video_games,action role-playing games,video games,P:video games,넓은 이름으로 찾음
729,news,Off-White,제품 브랜드,apparel_and_accessories,fashion apparel and accessories,apparel and accessories,,제품군만
3196,dart,와이즈캠프,서비스 브랜드,education_and_learning_services,smart learning services,smart learning services,P:smart learning services,블록 제품 번호
473,news,Cadillac Celestiq,제품 브랜드,electric_vehicles,electric luxury sedans,electric vehicles,P:electric vehicles,넓은 이름으로 찾음
831,dart,LG,회사·그룹 이름,,,,,제품 없음
2670,dart,Pluto HT,제품 브랜드,life_science_research_tools,automated cell sample preparation systems,laboratory sample preparation systems,P:automated cell sample preparation systems,블록 제품 번호
1706,dart,알코텔,제품 브랜드,gastrointestinal_and_liver_drugs,liver disease treatments,liver disease treatments,P:liver disease treatments,이름으로 찾음
